# CymbalGoal — Stage 4/5/6: Profile Text, Embeddings, and Staging

Generates free-text profiles for every player and club in the CymbalGoal scope, embeds them with
`gemini-embedding-001` at 3072 dimensions, QAs the result, and stages pass-2 CSVs to GCS for the
AlloyDB load.

**Runs in Colab Enterprise in the lab project.** It needs Vertex AI and GCS, both of which come
free with the Colab Enterprise runtime's service account.

**Inputs** — the eight relational tables Stage 3 staged to
`gs://class-demo/alloydb-labs/cymbalgoal/` as headerless gzipped CSV, plus
`artifacts/manifest.json` for their column order. Local Parquet copies are used instead when the
runtime still has them.

**Outputs** — `players_profiles.csv.gz`, `clubs_profiles.csv.gz`, `load_profiles.sql`, an updated
manifest, and a QA report. **This notebook does not re-export the relational tables.** They are
already staged and acceptance-checked; touching them is out of scope for pass 2.

---

### How to run it

Every stage is independently re-runnable and every long stage checkpoints. The intended path:

| Order | Section | Notes |
| :-- | :-- | :-- |
| 1 | Config → Derived facts | Cheap, deterministic, no API calls. Run freely. |
| 2 | **Sample generation (25 rows)** | ~$1. **Stop here and review with Patrick.** |
| 3 | Full generation | ~$185 at the default settings. Resumable. |
| 4 | Embeddings | ~$1. Resumable. |
| 5 | QA | Free. The export gate lives here and it is not optional. |
| 6 | Export + stage | Free. |

**Do not skip step 2.** Generating 14,235 profiles before a human has read twenty of them is the
single most expensive mistake available in this notebook.

### Model

`gemini-3.7-flash`. Gemini 2.5 Flash is scheduled for deprecation on **2026-10-16**, which is
reason enough on its own, but the economics also improve sharply: grounding drops from $35 to
$14 per 1,000 requests on the 3.x family, and the free allowance moves to 5,000/month. Token
prices go the other way ($0.30/$2.50 to $0.75/$3.75) and reasoning tokens are now unavoidable
and billed as output — `thinking_level` has no "off", only LOW/MEDIUM/HIGH. Net is still a large
saving, because grounding dominates. `THINKING_LEVEL` is LOW deliberately.

### What changed from the CymbalFlix harness

Six fixes, all called out inline where they land:

1. Checkpoints fire on a **batch counter**, not `len(results) % SAVE_INTERVAL` — the old form
   silently stops saving on any resume where the pre-loaded count isn't a multiple of the interval.
2. A **hard export gate** refuses to write a CSV if any profile still carries an error sentinel.
3. **Exponential backoff with jitter**, and 429/503 handled differently from 400/403.
4. **`temperature=0`** (the old `1.3` sat under a comment claiming grounding was off; it wasn't).
5. **Token and cost instrumentation** on every call.
6. **`EMBED_DIM = 3072`**, asserted per row before anything is written.

## 0 · Environment

In [ ]:
# Colab Enterprise usually has most of this. The install is quick and idempotent.
# gcsfs is what lets pandas read gs:// paths directly.
%pip install --quiet --upgrade google-genai pandas pyarrow numpy tqdm gcsfs

In [ ]:
from __future__ import annotations

import csv
import gzip
import hashlib
import io
import json
import math
import os
import random
import re
import subprocess
import sys
import tempfile
import threading
import time
import traceback
from collections import defaultdict
from contextlib import redirect_stdout
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Iterable

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

print(f"python  {sys.version.split()[0]}")
print(f"pandas  {pd.__version__}")
print(f"numpy   {np.__version__}")

## 1 · Config

Everything tunable lives in this cell. Nothing below it hardcodes a path, a model, or a price.

Two knobs deserve a second look before a full run:

- **`GROUND_MIN_APPEARANCES`** is the cost lever. Grounding bills per request (~$35/1,000 on
  Vertex), so it is ~96% of the total spend. `0` grounds every row. Raising it to `25` skips
  grounding for roughly 6,800 tail players — who are also the rows where Search has least to say
  and hallucinates most — and takes roughly $240 off the bill. Decide this **after** reading the
  sample, not before.
- **`EMBED_DIM = 3072` is locked.** AlloyDB's `google_ml.embedding(model_id, content)` takes two
  arguments and has no `output_dimensionality`. Lab 1 embeds the fan's query in SQL at runtime and
  gets 3072 back; a stored column of any other width fails `<=>` with a dimension mismatch, live,
  in front of the room.

In [ ]:
# ---- Provenance ------------------------------------------------------------------------------
SNAPSHOT_DATE = "2026-08-14"
SNAPSHOT_SHA = "3e6742a9c5002b202eeb8803a123bbc2d9c13ae582eb45ffb6f587843693fffa"
PROMPT_VERSION = "v1.6.0"  # bump on ANY prompt edit; lands in the manifest

# ---- Paths -----------------------------------------------------------------------------------
BASE = Path("/content/cymbalgoal")
PARQUET_DIR = BASE / "parquet"
ARTIFACT_DIR = BASE / "artifacts"
WORK_DIR = BASE / "work"      # checkpoints + logs
EXPORT_DIR = BASE / "export"  # staged CSVs

# ---- GCP -------------------------------------------------------------------------------------
PROJECT_ID = None  # None -> auto-detect from the runtime

# ⚠️ Generation and embeddings CANNOT share a location on Vertex.
#   * Gemini 3.x is served from the GLOBAL endpoint only. Asking us-central1 for
#     gemini-3.7-flash returns 404 "Publisher model not found", which reads like a permissions
#     problem and is not one.
#   * Embedding models are NOT available on the global endpoint at all — they are regional only.
# So the notebook keeps two clients. The preflight below probes each and self-corrects.
GEN_LOCATION = "global"
EMBED_LOCATION = "us-central1"
GEN_LOCATION_CANDIDATES = ["global", "us-central1", "us-east1"]
EMBED_LOCATION_CANDIDATES = ["us-central1", "us-east1", "europe-west4"]
GCS_PREFIX = "gs://class-demo/alloydb-labs/cymbalgoal"

# ---- Models ----------------------------------------------------------------------------------
GEN_MODEL = "gemini-3.7-flash"
EMBED_MODEL = "gemini-embedding-001"
EMBED_DIM = 3072            # LOCKED. See note above.
EMBED_TASK_TYPE = "RETRIEVAL_DOCUMENT"
EMBED_SIG_DIGITS = 6        # pgvector literal precision; 2.15x smaller than full float repr
TEMPERATURE = 0.0
TARGET_WORDS = 250

# Gemini 3.x replaced the integer `thinking_budget` with a `thinking_level` enum: LOW, MEDIUM,
# HIGH, defaulting to MEDIUM. There is no "off" any more — LOW is the floor — and reasoning
# tokens bill at the OUTPUT rate. For 250 words of descriptive prose over a pre-supplied fact
# block, LOW is the right setting; MEDIUM buys nothing here and every thinking token is billed.
THINKING_LEVEL = "LOW"

# Must cover reasoning AND the response. The old 900 was sized for a non-thinking model and
# would now truncate mid-profile, which surfaces as an empty or half-finished response rather
# than an error. Unused headroom is free.
MAX_OUTPUT_TOKENS = 16_384

# ---- Editorial ------------------------------------------------------------------------------
# The v1.0 sample surfaced off-pitch material about real, living people — social-media sentiment
# about a 23-year-old, a coach's public criticism of a player, character judgements. All probably
# sourced; all of it liable to appear on a projector in front of a thousand developers. True set
# here keeps reputation and temperament WHERE THEY CONCERN FOOTBALL and drops the rest.
# Flip to False to restore v1.0 behaviour.
EXCLUDE_OFF_PITCH_CONTROVERSY = True

# ---- Grounding -------------------------------------------------------------------------------
USE_GROUNDING = True
GROUND_MIN_APPEARANCES = 0  # 0 = ground everyone. THE cost lever. See note above.

# ---- Concurrency and resilience --------------------------------------------------------------
BATCH_SIZE = 100
MAX_WORKERS = 50
SAVE_EVERY_N_BATCHES = 5    # FIX 1: batch counter, never len(results) % N
# Retries are split by failure type, because 429 and 503 are not the same problem.
#   * 503 / connection reset — transient. Retry fast; it usually works second time.
#   * 429 — a QUOTA WALL. Vertex generative quotas are per-MINUTE, so retrying inside that window
#     cannot succeed. Worse, at MAX_WORKERS=50 a tight retry loop means 50 x N extra requests
#     against the very window that is already saturated. A 429 must be OUTWAITED, and if they
#     persist the answer is fewer workers, not more attempts.
MAX_RETRIES = 8
RETRY_BUDGET_S = 120.0        # per row, across all attempts — a wall clock, not just a count
BACKOFF_BASE = 2.0
BACKOFF_CAP_S = 60.0
RATE_LIMIT_MIN_WAIT_S = 20.0  # 429 floor. Shorter waits just re-hit the same quota minute.

# Adaptive concurrency. If more than this share of calls in a batch come back 429, the pool
# shrinks for the next batch; sustained clean batches let it grow back toward MAX_WORKERS.
THROTTLE_TRIGGER_RATE = 0.15
MIN_WORKERS = 4

# ---- QA thresholds ---------------------------------------------------------------------------
MIN_ACCEPTABLE_WORDS = 60       # below this a profile is too thin to embed usefully
NEAR_DUP_COSINE = 0.97          # above this two profiles are effectively the same text
REGULAR_APPEARANCE_FLOOR = 25   # below this a player is a "non-regular" and gets extra QA scrutiny

# ---- Pricing, USD. Verified against Vertex pricing, August 2026. -----------------------------
# ⚠️ $0.75/$3.75 is INTRODUCTORY and runs through 2026-12-31. From 2027-01-01 it doubles to
#    $1.50/$7.50. Regenerating this corpus next year costs twice what it costs today.
# Reasoning tokens are billed at the OUTPUT rate.
PRICE_IN_PER_M = 0.75
PRICE_OUT_PER_M = 3.75
# Gemini 3.x grounding is $14/1,000 with 5,000 free per MONTH, against 2.5's $35/1,000 with a
# daily free allowance. The free tier is account-wide and shared with anything else in the
# project, so treat the saving it implies as a ceiling, not a promise.
PRICE_GROUNDING_PER_1K_REQ = 14.00
GROUNDING_FREE_PER_MONTH = 5_000
PRICE_EMBED_PER_M = 0.15

# ---- The error sentinel ----------------------------------------------------------------------
# Failures are written into the text column so the resume path can find and retry them. FIX 2
# gates on this string at export so it can never reach AlloyDB.
ERROR_SENTINEL = "[[PROFILE_GENERATION_FAILED"

for d in (WORK_DIR, EXPORT_DIR, ARTIFACT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---- Checkpoints that outlive the runtime ----------------------------------------------------
# Colab's /content is ephemeral. A runtime failure eight hours into an embedding run took the
# work with it, and the only reason ~$156 of generated profiles survived was a manual copy to
# GCS. That should not have been manual. Checkpoints now mirror to a bucket automatically:
# restored on startup if missing locally, pushed after each stage and periodically during the
# long ones.
CKPT_GCS = f"{GCS_PREFIX}/checkpoints"
GCS_PUSH_EVERY_S = 600.0      # during long runs. The players parquet is ~380 MB; do not thrash it.

# A full regeneration costs roughly $156 and takes hours. If the checkpoints fail to restore, a
# resume looks exactly like a cold start — same code path, same flags, silently 156 dollars.
# This flag is the difference between "resume" and "spend"; it must be set deliberately.
ALLOW_FULL_REGENERATION = False


def _gcs(*args: str, quiet: bool = True) -> subprocess.CompletedProcess:
    return subprocess.run(["gcloud", "storage", *args],
                          capture_output=True, text=True,
                          check=False)


def push_checkpoints(verbose: bool = True) -> int:
    """Mirror local checkpoints to GCS. Never raises — a failed backup must not kill a live run."""
    sent = 0
    for f in sorted(WORK_DIR.glob("gen_*.csv")) + sorted(WORK_DIR.glob("emb_*.parquet")):
        try:
            r = _gcs("cp", str(f), f"{CKPT_GCS}/{f.name}")
            if r.returncode == 0:
                sent += 1
            elif verbose:
                print(f"  push failed for {f.name}: {r.stderr.strip()[:120]}")
        except Exception as exc:  # noqa: BLE001
            if verbose:
                print(f"  push failed for {f.name}: {type(exc).__name__}")
    if verbose:
        print(f"  checkpoints pushed to {CKPT_GCS}: {sent} file(s)")
    return sent


def restore_checkpoints(verbose: bool = True) -> list[str]:
    """
    Pull any checkpoint that exists in GCS but not locally. Safe to run every time: it never
    overwrites a local file, so a fresher local checkpoint always wins.
    """
    try:
        listing = _gcs("ls", f"{CKPT_GCS}/")
    except Exception as exc:  # noqa: BLE001
        if verbose:
            print(f"checkpoint restore skipped ({type(exc).__name__}); gcloud unavailable")
        return []
    if listing.returncode != 0:
        if verbose:
            print("no checkpoint backups in GCS yet (this is normal on a first run)")
        return []

    restored = []
    for uri in [l.strip() for l in listing.stdout.splitlines() if l.strip().endswith((".csv", ".parquet"))]:
        name = uri.rsplit("/", 1)[-1]
        local = WORK_DIR / name
        if local.exists():
            continue
        r = _gcs("cp", uri, str(local))
        if r.returncode == 0:
            restored.append(name)
            if verbose:
                print(f"  restored {name}  ({local.stat().st_size/1e6:.1f} MB)")
    if verbose and not restored:
        # Three different facts; one line for all of them was useless when it mattered.
        local = sorted(WORK_DIR.glob("gen_*.csv")) + sorted(WORK_DIR.glob("emb_*.parquet"))
        if local:
            print(f"checkpoints: {len(local)} local copy(ies) already present, nothing to fetch")
            for f in local:
                print(f"    {f.name}  ({f.stat().st_size/1e6:.1f} MB)")
        else:
            print("checkpoints: NOTHING LOCAL and nothing in the bucket to restore.")
            print(f"    If you expected a resume, check {CKPT_GCS}/ before running generation.")
    return restored


restore_checkpoints()

RUN_STARTED_UTC = datetime.now(timezone.utc).isoformat(timespec="seconds")
print(f"prompt version {PROMPT_VERSION}   gen {GEN_MODEL}   embed {EMBED_MODEL}@{EMBED_DIM}")
print(f"grounding {'ON' if USE_GROUNDING else 'OFF'}, min appearances {GROUND_MIN_APPEARANCES}")
print(f"thinking level {THINKING_LEVEL}   max output {MAX_OUTPUT_TOKENS:,} tokens")
print(f"off-pitch controversy excluded: {EXCLUDE_OFF_PITCH_CONTROVERSY}")

## 1.5 · `status()` — where the pipeline actually is

Deliberately placed here, immediately after the config and **before** anything that needs Vertex
or the data. It reads only the filesystem, so after a Colab runtime drop — which will happen on a
multi-hour run — you can recover the whole picture by running three cells: imports, config, this.
No API calls, no reload, no cost.

Everything it reports comes from the checkpoints and the export directory, which are the real
source of truth. In-memory dataframes are not; they die with the runtime.

In [ ]:
EXPECTED_PROFILE_ROWS = {"players": 13_439, "clubs": 796}


def _ckpt(kind: str) -> Path:
    return WORK_DIR / f"gen_{kind}_{PROMPT_VERSION}.csv"


def _emb(kind: str) -> Path:
    return WORK_DIR / f"emb_{kind}_{PROMPT_VERSION}.parquet"


def status(verbose: bool = True) -> dict:
    """Stage-by-stage state of the pipeline, plus the single next action."""
    st: dict = {"prompt_version": PROMPT_VERSION, "stages": {}}
    lines = [f"CymbalGoal Stage 4/5/6 — prompt {PROMPT_VERSION}, model {GEN_MODEL}", ""]

    spend = 0.0
    blocking = None

    lines.append("GENERATION")
    for kind, expected in EXPECTED_PROFILE_ROWS.items():
        path = _ckpt(kind)
        if not path.exists():
            lines.append(f"  {kind:<9} not started            0 / {expected:,}")
            st["stages"][f"gen_{kind}"] = {"done": 0, "expected": expected}
            blocking = blocking or f"run generation for {kind}"
            continue
        df = pd.read_csv(path)
        df["text"] = df["text"].fillna("")
        failed = int(df["text"].str.startswith(ERROR_SENTINEL).sum())
        ok = len(df) - failed
        tok_in = int(df.get("in_tokens", pd.Series(dtype=int)).sum())
        tok_out = int(df.get("out_tokens", pd.Series(dtype=int)).sum())
        gr = df.get("grounding_metadata")
        gr_rate = float(pd.Series(gr).astype(str).str.lower().eq("true").mean()) if gr is not None else float("nan")
        cost = (tok_in / 1e6) * PRICE_IN_PER_M + (tok_out / 1e6) * PRICE_OUT_PER_M
        billable = max(0, ok - GROUNDING_FREE_PER_MONTH)
        cost += billable / 1000 * PRICE_GROUNDING_PER_1K_REQ
        spend += cost
        pct = ok / expected * 100 if expected else 0
        lines.append(f"  {kind:<9} {ok:>7,} / {expected:,}  ({pct:5.1f}%)   "
                     f"{failed:,} awaiting retry   searched {gr_rate:.0%}   ~${cost:,.2f}")
        st["stages"][f"gen_{kind}"] = {"done": ok, "failed": failed, "expected": expected,
                                       "cost_usd": cost, "grounded_rate": gr_rate}
        if failed:
            blocking = blocking or f"re-run generation to retry {failed:,} failed {kind} rows"
        elif ok < expected:
            blocking = blocking or f"continue generation for {kind}"

    lines.append("")
    lines.append("EMBEDDINGS")
    for kind, expected in EXPECTED_PROFILE_ROWS.items():
        path = _emb(kind)
        if not path.exists():
            lines.append(f"  {kind:<9} not started            0 / {expected:,}")
            st["stages"][f"emb_{kind}"] = {"done": 0, "expected": expected}
            continue
        e = pd.read_parquet(path)
        good = int(((e["dims"] == EMBED_DIM) & (e["embedding"].str.len() > 0)).sum())
        tok = int(e.get("tokens", pd.Series(dtype=int)).sum())
        cost = tok / 1e6 * PRICE_EMBED_PER_M
        spend += cost
        lines.append(f"  {kind:<9} {good:>7,} / {expected:,}  ({good/expected*100:5.1f}%)"
                     f"   ~${cost:,.2f}")
        st["stages"][f"emb_{kind}"] = {"done": good, "expected": expected, "cost_usd": cost}

    lines.append("")
    lines.append("EXPORT")
    for fname, key in (("players_profiles.csv.gz", "players"), ("clubs_profiles.csv.gz", "clubs")):
        f = EXPORT_DIR / fname
        if f.exists():
            lines.append(f"  {fname:<26} {f.stat().st_size/1e6:>7.1f} MB written")
        else:
            lines.append(f"  {fname:<26}         not written")
    sql = ARTIFACT_DIR / "load_profiles.sql"
    lines.append(f"  {'load_profiles.sql':<26} {'written' if sql.exists() else 'not written':>15}")

    lines.append("")
    lines.append(f"ESTIMATED SPEND SO FAR   ~${spend:,.2f}")
    st["spend_usd"] = spend

    if blocking is None:
        gen_done = all(st["stages"].get(f"gen_{k}", {}).get("done", 0) >= v
                       for k, v in EXPECTED_PROFILE_ROWS.items())
        emb_done = all(st["stages"].get(f"emb_{k}", {}).get("done", 0) >= v
                       for k, v in EXPECTED_PROFILE_ROWS.items())
        if not gen_done:
            blocking = "continue generation"
        elif not emb_done:
            blocking = "run the fresh review, then embeddings"
        elif not (EXPORT_DIR / "players_profiles.csv.gz").exists():
            blocking = "run QA and export"
        else:
            blocking = "upload to GCS and update the manifest"
    lines.append(f"NEXT                     {blocking}")
    st["next"] = blocking

    if verbose:
        print("\n".join(lines))
    return st


status()

## 2 · Vertex client

One `genai.Client` **per thread**, via `threading.local()`. A single client shared across 50
workers is the classic way to turn a parallel generator into a serialised one with intermittent
transport errors.

In [ ]:
from google import genai
from google.genai import types as gtypes
from google.genai import errors as gerrors

if PROJECT_ID is None:
    PROJECT_ID = (
        os.environ.get("GOOGLE_CLOUD_PROJECT")
        or subprocess.run(
            ["gcloud", "config", "get-value", "project"],
            capture_output=True, text=True, check=False,
        ).stdout.strip()
    )
if not PROJECT_ID or PROJECT_ID == "(unset)":
    raise RuntimeError("Could not determine PROJECT_ID. Set it explicitly in the config cell.")

_tls = threading.local()


def get_client(location: str | None = None) -> "genai.Client":
    """
    Thread-local client, cached per location. Never share one across the pool, and never assume
    one location serves both models — see the config note.
    """
    loc = location or GEN_LOCATION
    cache = getattr(_tls, "clients", None)
    if cache is None:
        cache = _tls.clients = {}
    client = cache.get(loc)
    if client is None:
        client = cache[loc] = genai.Client(vertexai=True, project=PROJECT_ID, location=loc)
    return client


# Which optional fields this google-genai build actually accepts. Lets the notebook use newer
# config knobs without hard-failing on an older SDK in a lab runtime we do not control.
_GCC_FIELDS = set(getattr(gtypes.GenerateContentConfig, "model_fields", None) or {})


def afc_off() -> dict[str, Any]:
    """
    Silences the 'Direct use of automatic function calling is not recommended' warning. Correct
    as well as quiet: `google_search` is a server-side tool, so there is no client-side function
    for the SDK to call and the AFC loop has nothing to do.
    """
    if "automatic_function_calling" not in _GCC_FIELDS:
        return {}
    try:
        return {"automatic_function_calling":
                gtypes.AutomaticFunctionCallingConfig(disable=True)}
    except Exception:
        return {}


def _probe_generation(model: str, location: str) -> tuple[bool, str]:
    try:
        r = get_client(location).models.generate_content(
            model=model,
            contents="Reply with the single word: ready",
            config=gtypes.GenerateContentConfig(
                temperature=0.0, max_output_tokens=2048, **afc_off()),
        )
        return True, (r.text or "").strip()[:40] or "(empty)"
    except Exception as exc:  # noqa: BLE001
        return False, f"{type(exc).__name__}: {str(exc)[:150]}"


def _probe_embedding(model: str, location: str) -> tuple[bool, str]:
    try:
        r = get_client(location).models.embed_content(
            model=model, contents="ready",
            config=gtypes.EmbedContentConfig(output_dimensionality=EMBED_DIM),
        )
        n = len(r.embeddings[0].values)
        return (n == EMBED_DIM), f"{n} dims"
    except Exception as exc:  # noqa: BLE001
        return False, f"{type(exc).__name__}: {str(exc)[:150]}"


def preflight() -> None:
    """
    Resolve a working location for each model before anything is paid for. A 404 here is almost
    always the wrong endpoint rather than missing access, so say so plainly instead of letting a
    raw traceback imply the project is misconfigured.
    """
    global GEN_LOCATION, EMBED_LOCATION
    print(f"project {PROJECT_ID}\n")

    tried: list[str] = []
    for loc in [GEN_LOCATION] + [x for x in GEN_LOCATION_CANDIDATES if x != GEN_LOCATION]:
        ok, msg = _probe_generation(GEN_MODEL, loc)
        print(f"  {'✅' if ok else '  '} generation  {GEN_MODEL:<20} @ {loc:<14} {msg}")
        if ok:
            GEN_LOCATION = loc
            break
        tried.append(loc)
    else:
        raise RuntimeError(
            f"{GEN_MODEL} is not reachable from any of {tried}.\n"
            "  • Gemini 3.x models are served from the GLOBAL endpoint only — a 404 saying the\n"
            "    publisher model 'was not found' in a region means the region is wrong, not that\n"
            "    the project lacks access. GEN_LOCATION should be 'global'.\n"
            "  • If 'global' also 404s, the model is genuinely not enabled for this project.\n"
            "    Check `gcloud ai models list` and Model Garden, or set GEN_MODEL back to a model\n"
            "    the project can see."
        )

    tried = []
    for loc in [EMBED_LOCATION] + [x for x in EMBED_LOCATION_CANDIDATES if x != EMBED_LOCATION]:
        ok, msg = _probe_embedding(EMBED_MODEL, loc)
        print(f"  {'✅' if ok else '  '} embedding   {EMBED_MODEL:<20} @ {loc:<14} {msg}")
        if ok:
            EMBED_LOCATION = loc
            break
        tried.append(loc)
    else:
        raise RuntimeError(
            f"{EMBED_MODEL} is not reachable at {EMBED_DIM} dims from any of {tried}.\n"
            "  • Embedding models are NOT available on the global endpoint. EMBED_LOCATION must\n"
            "    be a region, which is why it is configured separately from GEN_LOCATION.\n"
            "  • If the width is wrong rather than the location, stop: 3072 is fixed by the\n"
            "    AlloyDB schema and is not a tuning knob."
        )

    print(f"\ngeneration -> {GEN_LOCATION}   embeddings -> {EMBED_LOCATION}")
    if GEN_LOCATION != EMBED_LOCATION:
        print("(Different endpoints, as expected. Two clients, cached per location per thread.)")


preflight()

## 3 · Load the Stage 3 tables

Stage 3 staged all eight relational tables to GCS as gzipped CSV, and left Parquet copies in the
local runtime. Either is fine — they are the same data — so the loader prefers local Parquet when
it is there (faster, no egress) and falls back to GCS otherwise. Colab Enterprise runtimes are
ephemeral, so on a fresh runtime the GCS path is the one that runs.

### Three things about the staged CSVs that will bite if ignored

**They have no header row.** That is an AlloyDB CSV import requirement, not an oversight. Column
order comes from `manifest.json` → `staged_files[<table>].column_order`, which is written by the
same code that wrote the files, so it cannot drift. Never hand-type the column list.

**Their dtypes are not self-describing.** A headerless CSV read gives pandas nothing to go on, so
it guesses — and it guesses wrong in two ways that matter here. `game_event_id` is a 32-char hex
digest, and any digest that happens to be all digits gets read as a float. Dates arrive as
strings. Both are forced explicitly below.

**⚠️ Nullable integer columns become floats.** The moment an integer column takes a NULL, pandas
widens `int64` to `float64` and `current_club_id` becomes `69261.0`. This is the trap that hit 22
columns across 7 tables in Stage 3. Three concrete consequences, verified rather than assumed:

1. **It breaks the load.** `69261.0` written to CSV is rejected outright by PostgreSQL `INTEGER`.
   This is the one that bit Stage 3, and it surfaces at provision time, not here.
2. **It leaks into the prompt.** An f-string renders the id as `69261.0`, so a fact block can end
   up reading "club 69261.0" — which the model will then dutifully work into a sentence.
3. **`.astype(int)` raises** on any column that still holds a NaN, which is how this usually
   announces itself mid-run.

What it does *not* do, despite being widely repeated: break `dict.get()`. Python hashes `1.0` and
`1` identically, so `club_names.get(69261.0)` returns the club just fine, and a float-vs-`Int64`
merge coerces rather than silently dropping rows. Both were tested. Worth stating plainly, because
hunting a lookup bug that cannot exist wastes an afternoon.

Everything id-shaped is cast to pandas nullable `Int64` at load, the lookups below use `int()`
keys anyway, and the exporter re-checks the written bytes.

In [ ]:
EXPECTED_ROWS = {
    "competitions": 65,
    "clubs": 796,
    "players": 13_439,
    "games": 29_740,
    "appearances": 832_193,
    "game_events": 417_617,
    "player_valuations": 297_822,
    "transfers": 65_494,
}

# Anything id- or count-shaped that can take a NULL. Cast to nullable Int64, never float64.
INT_COLS = {
    "club_id", "player_id", "game_id", "transfer_id", "current_club_id", "player_club_id",
    "home_club_id", "away_club_id", "from_club_id", "to_club_id", "player_in_id",
    "player_assist_id", "season", "last_season", "minute", "attendance", "height_in_cm",
    "squad_size", "foreigners_number", "national_team_players", "stadium_seats", "total_clubs",
    "international_caps", "international_goals", "market_value_in_eur",
    "highest_market_value_in_eur", "net_transfer_record_eur", "transfer_fee",
    "home_club_goals", "away_club_goals", "home_club_position", "away_club_position",
}
DATE_COLS = {
    "date_of_birth", "contract_expiration_date", "game_date", "appearance_date",
    "event_date", "valuation_date", "transfer_date",
}
# TEXT in the schema, and dangerous to let pandas infer. game_event_id is a hex digest whose
# all-digit values would otherwise be read as floats.
TEXT_COLS = {
    "appearance_id", "game_event_id", "competition_id", "domestic_competition_id", "club_code",
    "transfer_season", "round_label", "aggregate", "from_club_name", "to_club_name",
}


def _cat_gcs(uri: str) -> str | None:
    try:
        out = subprocess.run(["gcloud", "storage", "cat", uri], capture_output=True, text=True)
    except Exception:  # noqa: BLE001
        return None
    return out.stdout if out.returncode == 0 else None


def columns_from_ddl(sql: str) -> dict[str, list[str]]:
    """
    Recover per-table column order from CREATE TABLE statements.

    P-40 established that the DDL is the authoritative source of column order and the manifest
    merely records it. So when the manifest is gone, the DDL is not a degraded substitute — it is
    the original. Table-level constraints are skipped; only column definitions count.
    """
    out: dict[str, list[str]] = {}
    for m in re.finditer(r"CREATE\s+TABLE\s+(?:IF\s+NOT\s+EXISTS\s+)?[\"`]?(\w+)[\"`]?\s*\((.*?)\n\s*\)\s*;",
                         sql, re.S | re.I):
        name, body = m.group(1), m.group(2)
        cols: list[str] = []
        depth = 0
        for raw in body.split("\n"):
            line = raw.strip()
            if not line or line.startswith("--"):
                continue
            if depth == 0 and not re.match(
                    r"(?i)(PRIMARY\s+KEY|FOREIGN\s+KEY|CONSTRAINT|UNIQUE|CHECK|EXCLUDE)\b", line):
                tok = line.split()[0].strip('"`,')
                if tok:
                    cols.append(tok)
            depth += line.count("(") - line.count(")")
        if cols:
            out[name] = cols
    return out


def _manifest() -> dict:
    p = ARTIFACT_DIR / "manifest.json"
    if p.exists():
        return json.loads(p.read_text())

    # Fresh runtime: try the bucket, in the places it might have been written.
    for uri in (f"{GCS_PREFIX}/manifest.json",
                f"{GCS_PREFIX}/artifacts/manifest.json",
                f"{CKPT_GCS}/manifest.json"):
        body = _cat_gcs(uri)
        if body:
            print(f"manifest restored from {uri}")
            p.write_text(body)
            return json.loads(body)

    # No manifest anywhere. Rebuild what this notebook actually needs — the column order — from
    # schema.sql, which IS staged. The manifest was written locally by Stage 3 and never uploaded,
    # so a dead runtime takes it with them; that is a gap in Stage 3, not a reason to stop here.
    sql = _cat_gcs(f"{GCS_PREFIX}/schema.sql")
    if sql:
        cols = columns_from_ddl(sql)
        if cols:
            print("No manifest found. Rebuilt column order from the staged schema.sql:")
            for t, c in cols.items():
                print(f"  {t:<20} {len(c):>2} columns")
            derived = {
                "snapshot": {"date": "unknown", "sha256": "unknown",
                             "note": "manifest was absent; snapshot fields could not be recovered"},
                "staged_files": {t: {"file": f"{t}.csv.gz", "column_order": c, "header": False}
                                 for t, c in cols.items()},
                "derived_from": f"{GCS_PREFIX}/schema.sql",
            }
            p.write_text(json.dumps(derived, indent=2))
            print(f"  written to {p} and mirrored to the bucket so this cannot recur")
            try:
                subprocess.run(["gcloud", "storage", "cp", str(p), f"{GCS_PREFIX}/manifest.json"],
                               capture_output=True, check=False)
            except Exception:  # noqa: BLE001
                pass
            return derived

    raise FileNotFoundError(
        f"No manifest at {p}, none in the bucket, and no schema.sql to rebuild it from.\n"
        f"Checked: {GCS_PREFIX}/manifest.json, {GCS_PREFIX}/artifacts/manifest.json,\n"
        f"         {CKPT_GCS}/manifest.json, {GCS_PREFIX}/schema.sql\n"
        "The staged CSVs are headerless, so column order cannot be guessed. List the bucket:\n"
        f"  !gcloud storage ls -r {GCS_PREFIX}/"
    )


MANIFEST = _manifest()
STAGED = MANIFEST.get("staged_files", {})


def normalize(df: pd.DataFrame) -> pd.DataFrame:
    """Force the dtypes the schema says, not the ones pandas guessed."""
    for c in df.columns:
        if c in INT_COLS:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")
        elif c in DATE_COLS:
            df[c] = pd.to_datetime(df[c], errors="coerce")
        elif c in TEXT_COLS:
            df[c] = df[c].astype("string")
    return df


def read_table(name: str) -> tuple[pd.DataFrame, str]:
    local = PARQUET_DIR / f"{name}.parquet"
    if local.exists():
        return normalize(pd.read_parquet(local)), "local parquet"

    entry = STAGED.get(name) or {}
    cols = entry.get("column_order")
    if not cols:
        raise KeyError(
            f"manifest staged_files['{name}'].column_order is missing. Refusing to guess the "
            "column order of a headerless CSV."
        )
    uri = f"{GCS_PREFIX}/{name}.csv.gz"
    try:
        return normalize(pd.read_csv(uri, header=None, names=cols, compression="gzip")), "gcs (gcsfs)"
    except Exception:
        # gcsfs missing or unhappy — stream it through the CLI instead. Never gsutil.
        proc = subprocess.run(["gcloud", "storage", "cat", uri], capture_output=True, check=True)
        buf = gzip.decompress(proc.stdout)
        df = pd.read_csv(io.BytesIO(buf), header=None, names=cols)
        return normalize(df), "gcs (gcloud storage cat)"


T: dict[str, pd.DataFrame] = {}
for name in EXPECTED_ROWS:
    T[name], src = read_table(name)
    got, want = len(T[name]), EXPECTED_ROWS[name]
    mark = "✅" if got == want else "🔴"
    print(f"  {mark} {name:<20} {got:>9,} rows x {T[name].shape[1]:>2} cols   [{src}]")

_mismatch = {n: (len(T[n]), EXPECTED_ROWS[n]) for n in EXPECTED_ROWS if len(T[n]) != EXPECTED_ROWS[n]}
if _mismatch:
    print("\n⚠️  ROW COUNT MISMATCH vs the Stage 3 manifest:")
    for n, (got, want) in _mismatch.items():
        print(f"     {n:<20} got {got:>9,}   expected {want:>9,}")
    print("     If Stage 3 was legitimately re-run, update EXPECTED_ROWS. Otherwise stop and")
    print("     find out why the snapshot moved before spending money on generation.")
else:
    print("\nAll eight tables loaded, row counts match the Stage 3 manifest.")

# Prove the dtype normalization took. A float id here means silent wrong text later.
_float_ids = [(n, c) for n, df in T.items() for c in df.columns
              if c in INT_COLS and str(df[c].dtype).startswith("float")]
if _float_ids:
    raise RuntimeError(f"Integer columns still float64 after normalize(): {_float_ids}")
print(f"id dtypes clean — players.current_club_id is {T['players']['current_club_id'].dtype}, "
      f"{int(T['players']['current_club_id'].isna().sum()):,} NULL")

## 4 · Stage 4a — Derived facts

**Decision D-11: inject the hard numbers, then ground.** Everything in this section is computed
from the relational tables, is exact, and goes into the prompt as authoritative. The model's job
is to confirm identity against those numbers and then blend them with what Google Search knows
about style, reputation, and career narrative.

Two things this buys us:

1. **The numbers can't be hallucinated**, because the model is never asked to recall them. That
   collapses the QA burden — the spot-check only has to police claims the model *added*.
2. **Identity disambiguation.** 155 players in scope share a name with someone else (`Paulinho`
   ×6, `Danilo` ×5, `Fernando` ×4). Handing the model a birth date, a birth city, and a specific
   club history is what stops Search from confidently retrieving the wrong Paulinho.

One scoping caveat that gets threaded into the prompt: these stats cover **only** the Big 5
leagues plus the Champions and Europa Leagues, 2012–2025. For a player whose career ran wider or
longer, they are not career totals, and the prompt says so explicitly so the model doesn't present
them as such.

In [ ]:
apps = T["appearances"]
games = T["games"][["game_id", "season", "competition_id"]].rename(
    columns={"competition_id": "game_competition_id"}
)
# int() keys, deliberately. With nullable Int64 these dicts would otherwise key on numpy scalars,
# and any lookup that arrives as a plain float would miss silently. See the loader note.
# ⚠️ Transfermarkt's competitions.name is a URL SLUG, not a display name — "laliga",
# "uefa-champions-league". The v1.1 sample proved this matters: Real Madrid's profile came back
# reading "competing in laliga … across 532 matches and in the uefa-champions-league over 170".
# Most rows the model tidied up on its own, which is exactly why it slipped through — an
# inconsistent defect looks like a one-off. Fixed deterministically here rather than asked for in
# the prompt, because a lookup table cannot forget.
COMPETITION_DISPLAY = {
    "GB1": "the Premier League", "ES1": "LaLiga", "IT1": "Serie A",
    "L1": "the Bundesliga", "FR1": "Ligue 1",
    "CL": "the UEFA Champions League", "EL": "the UEFA Europa League",
}
_ACRONYMS = {"uefa": "UEFA", "fa": "FA", "dfb": "DFB", "efl": "EFL", "mls": "MLS",
             "usl": "USL", "afc": "AFC", "caf": "CAF", "u21": "U21", "u19": "U19"}
_RAW_COMPS = T["competitions"].set_index("competition_id")["competition_name"].to_dict()
_COMP_COUNTRY = T["competitions"].set_index("competition_id")["country_name"].to_dict()


def _deslug(raw: str) -> str:
    if not isinstance(raw, str) or not raw.strip():
        return ""
    # Already a display name if it has spaces and any capitalisation.
    if " " in raw and raw != raw.lower():
        return raw
    parts = raw.replace("_", "-").split("-")
    return " ".join(_ACRONYMS.get(x.lower(), x.capitalize()) for x in parts if x)


def _comp(cid, with_country: bool = False) -> str:
    """
    Human-readable competition name. Never let a URL slug reach the prompt.

    with_country matters for competitions OUTSIDE our seven. The v1.2 sample had Sturm Graz —
    an Austrian club — described as "competing in the Bundesliga", which reads as the German
    league that IS in scope. The country disambiguates it, and the source has the column.
    """
    key = str(cid)
    if key in COMPETITION_DISPLAY:
        return COMPETITION_DISPLAY[key]
    name = _deslug(_RAW_COMPS.get(key, "")) or key
    if with_country:
        country = _COMP_COUNTRY.get(key)
        if isinstance(country, str) and country.strip() and country.strip().lower() not in name.lower():
            name = f"{name} ({country.strip()})"
    return name


comps = {str(k): _comp(k) for k in _RAW_COMPS}
_slugged = {k: v for k, v in _RAW_COMPS.items() if isinstance(v, str) and v == v.lower() and v}
print(f"competition names: {len(_slugged)} of {len(_RAW_COMPS)} arrived as URL slugs, de-slugged")
for k in list(COMPETITION_DISPLAY):
    if k in _RAW_COMPS:
        print(f"  {k:<4} {str(_RAW_COMPS[k]):<24} -> {comps[k]}")
club_names = {int(k): v for k, v in
              T["clubs"].set_index("club_id")["club_name"].to_dict().items()}


def _cn(cid) -> str | None:
    """Club name from anything id-shaped — int, numpy scalar, float, or NA."""
    if cid is None or pd.isna(cid):
        return None
    return club_names.get(int(cid))

apps_s = apps.merge(games[["game_id", "season"]], on="game_id", how="left")

# ---- Career totals ---------------------------------------------------------------------------
career = (
    apps_s.groupby("player_id")
    .agg(
        appearances=("appearance_id", "count"),
        minutes=("minutes_played", "sum"),
        goals=("goals", "sum"),
        assists=("assists", "sum"),
        yellow_cards=("yellow_cards", "sum"),
        red_cards=("red_cards", "sum"),
        first_season=("season", "min"),
        last_season_played=("season", "max"),
        first_app=("appearance_date", "min"),
        last_app=("appearance_date", "max"),
    )
)
career["goals_per_90"] = np.where(
    career["minutes"] > 0, career["goals"] * 90.0 / career["minutes"], 0.0
)
career["assists_per_90"] = np.where(
    career["minutes"] > 0, career["assists"] * 90.0 / career["minutes"], 0.0
)

# ---- Split by competition --------------------------------------------------------------------
by_comp = (
    apps_s.groupby(["player_id", "competition_id"])
    .agg(a=("appearance_id", "count"), g=("goals", "sum"), s=("assists", "sum"))
    .reset_index()
    .sort_values(["player_id", "a"], ascending=[True, False])
)
comp_lines: dict[int, list[str]] = defaultdict(list)
for pid, cid, a, g, s in by_comp.itertuples(index=False):
    if len(comp_lines[pid]) < 5:
        comp_lines[pid].append(f"{_comp(cid)}: {a} apps, {g} goals, {s} assists")

# ---- Best single season ----------------------------------------------------------------------
by_season = (
    apps_s.groupby(["player_id", "season"])
    .agg(a=("appearance_id", "count"), g=("goals", "sum"), s=("assists", "sum"))
    .reset_index()
)
by_season["ga"] = by_season["g"] + by_season["s"]
best_season = (
    by_season.sort_values(["player_id", "ga", "g"], ascending=[True, False, False])
    .groupby("player_id")
    .head(1)
    .set_index("player_id")
)

# ---- Club history in scope, ordered by first appearance --------------------------------------
club_spell = (
    apps_s.dropna(subset=["player_club_id"])
    .groupby(["player_id", "player_club_id"])
    .agg(frm=("season", "min"), to=("season", "max"), a=("appearance_id", "count"))
    .reset_index()
    .sort_values(["player_id", "frm"])
)
club_history: dict[int, list[str]] = defaultdict(list)
for pid, cid, frm, to, a in club_spell.itertuples(index=False):
    nm = _cn(cid) or f"club {int(cid)}"
    span = f"{int(frm)}" if frm == to else f"{int(frm)}–{int(to)}"
    club_history[pid].append(f"{nm} ({span}, {a} apps)")

# ---- Late goals: a small, real style signal ---------------------------------------------------
ge = T["game_events"]
goal_ev = ge[(ge["event_type"] == "Goals") & ge["player_id"].notna()]
# .fillna(False) matters: with nullable Int64 minutes, the comparison yields a nullable boolean
# and .astype(int) raises on pd.NA rather than treating an unknown minute as "not late".
late = (
    goal_ev.assign(late=(goal_ev["minute"] >= 75).fillna(False).astype(int))
    .groupby("player_id")
    .agg(evt_goals=("game_event_id", "count"), late_goals=("late", "sum"))
)

# ---- Transfers: the headline moves ------------------------------------------------------------
# from_club_name / to_club_name are populated even when the id is NULL (~49% / ~40% of rows),
# which is exactly why Stage 3 kept them. They are what lets a profile say "joined from Santos"
# instead of "joined from somewhere untracked" — good, specific material for the model to ground
# on. Two special values need care: 'Without Club' means a free agent, and youth or reserve sides
# (RM Castilla, Ajax U21, Barça Youth) are common and should read as a promotion, not a transfer.
NON_CLUB = {"without club", "unknown", "retired", "career break", "ban"}
YOUTH_RE = re.compile(r"(?i)\b(u\s?1[5-9]|u\s?2[0-3]|youth|castilla|academy|reserve|ii|b team)\b")


def _move_phrase(frm: str | None, to: str | None, year: int, fee) -> str:
    frm = (frm or "Unknown").strip()
    to = (to or "Unknown").strip()
    if frm.lower() in NON_CLUB:
        lead = f"signed by {to} as a free agent in {year}"
    elif YOUTH_RE.search(frm):
        lead = f"promoted from {frm} to {to} in {year}"
    else:
        lead = f"{frm} to {to} in {year}"
    if fee is not None and not pd.isna(fee) and fee > 0:
        lead += f" for €{int(fee):,}"
    elif fee is not None and not pd.isna(fee) and fee == 0:
        lead += " on a free transfer"
    return lead


tr = T["transfers"].sort_values("transfer_fee", ascending=False, na_position="last")
big_moves: dict[int, list[str]] = defaultdict(list)
for r in tr.itertuples(index=False):
    pid = int(r.player_id)
    if len(big_moves[pid]) >= 3 or pd.isna(r.transfer_fee) or r.transfer_fee <= 0:
        continue
    big_moves[pid].append(
        _move_phrase(r.from_club_name, r.to_club_name,
                     pd.to_datetime(r.transfer_date).year, r.transfer_fee)
    )

# The first and most recent move, regardless of fee — these carry the career arc even when the
# fee is unknown, and for a free agent or academy graduate they are the only transfer story there is.
_tr_time = T["transfers"].sort_values("transfer_date")
career_moves: dict[int, list[str]] = defaultdict(list)
for r in _tr_time.itertuples(index=False):
    pid = int(r.player_id)
    career_moves[pid].append(
        _move_phrase(r.from_club_name, r.to_club_name,
                     pd.to_datetime(r.transfer_date).year, r.transfer_fee)
    )

free_moves = (
    tr[(tr["transfer_fee"] == 0) | tr["transfer_fee"].isna()]
    .groupby("player_id").size().to_dict()
)

# ---- Market value peak ------------------------------------------------------------------------
pv = T["player_valuations"].dropna(subset=["market_value_in_eur"])
peak_idx = pv.groupby("player_id")["market_value_in_eur"].idxmax()
peak_val = pv.loc[peak_idx].set_index("player_id")[["market_value_in_eur", "valuation_date"]]

# How badly do the two tables disagree? Worth knowing, not just working around.
_ev = late["evt_goals"].reindex(career.index).fillna(0).astype(int)
_ap = career["goals"].astype(int)
_gap = (_ev - _ap)[(_ap >= 10)]
print(f"goal-count agreement (appearances vs game_events, players with 10+ goals): "
      f"{int((_gap == 0).sum()):,} exact, {int((_gap != 0).sum()):,} differ "
      f"(max gap {int(_gap.abs().max()) if len(_gap) else 0})")
print("  appearances is authoritative; the late-goal line is stated against it.")
print(f"derived facts computed for {len(career):,} players with at least one appearance")
print(f"players with no appearance rows at all: {len(T['players']) - len(career):,}")

### 4a.2 — Assemble the per-player fact block

One formatted text block per player, ready to drop into the prompt. Deliberately written as
labelled prose rather than JSON: the generation model handles labelled prose better, and it keeps
the "these are authoritative" framing legible.

In [ ]:
players = T["players"].copy()


def _eur(v) -> str:
    if pd.isna(v) or v is None:
        return "not recorded"
    v = int(v)
    if v >= 1_000_000:
        return f"€{v/1_000_000:.1f}m"
    if v >= 1_000:
        return f"€{v/1_000:.0f}k"
    return f"€{v}"


def _d(v) -> str:
    if pd.isna(v) or v is None:
        return "unknown"
    return pd.to_datetime(v).strftime("%d %B %Y")


def player_identity_block(p) -> str:
    """The authoritative 'who is this' block. Exists to stop Search retrieving a namesake."""
    born = _d(p.date_of_birth)
    place = ", ".join([x for x in (p.city_of_birth, p.country_of_birth) if isinstance(x, str) and x])
    club = _cn(p.current_club_id)
    hist = club_history.get(p.player_id, [])
    lines = [
        f"Full name: {p.player_name}",
        f"Born: {born}" + (f" in {place}" if place else ""),
        f"Citizenship: {p.country_of_citizenship if isinstance(p.country_of_citizenship, str) else 'not recorded'}",
        f"Position: {p.main_position or 'not recorded'}"
        + (f" — {p.detailed_position}" if isinstance(p.detailed_position, str) and p.detailed_position else ""),
        f"Preferred foot: {p.foot if isinstance(p.foot, str) and p.foot else 'not recorded'}",
        f"Height: {int(p.height_in_cm)} cm" if pd.notna(p.height_in_cm) and p.height_in_cm > 0 else "Height: not recorded",
        f"Current club: {club}" if club else "Current club: a club outside the Big 5 leagues",
        f"Clubs he played for in these competitions: {'; '.join(hist) if hist else 'none recorded'}",
        f"Transfermarkt profile: {p.url}" if isinstance(p.url, str) and p.url else "",
    ]
    return "\n".join(x for x in lines if x)


def player_stats_block(p) -> str:
    """Authoritative numbers. The model is told never to contradict or extend these."""
    pid = p.player_id
    if pid not in career.index:
        return (
            "He did not play in the Big 5 leagues or the UEFA Champions or Europa League between "
            "2012 and 2025. Write about his career as Search describes it, and do not remark on "
            "the absence of these figures."
        )
    c = career.loc[pid]
    out = [
        f"Appearances: {int(c.appearances)}",
        f"Minutes played: {int(c.minutes):,}",
        f"Goals: {int(c.goals)}   Assists: {int(c.assists)}",
        f"Goals per 90 minutes: {c.goals_per_90:.2f}   Assists per 90: {c.assists_per_90:.2f}",
        f"Yellow cards: {int(c.yellow_cards)}   Red cards: {int(c.red_cards)}",
        f"Active in these seasons: {int(c.first_season)}–{int(c.last_season_played)}"
        f" (first appearance {_d(c.first_app)}, most recent {_d(c.last_app)})",
    ]
    if pid in best_season.index:
        b = best_season.loc[pid]
        out.append(
            f"Best season by goal involvement: {int(b.season)}/{str(int(b.season)+1)[-2:]} "
            f"— {int(b.a)} apps, {int(b.g)} goals, {int(b.s)} assists"
        )
    if comp_lines.get(pid):
        out.append("By competition: " + "; ".join(comp_lines[pid]))
    # ⚠️ game_events and appearances disagree about goal counts for some players — the v1.3
    # sample surfaced Danilo, whose event data holds 13 goals against 11 in appearances. The
    # model resolved the clash by inventing a category ("13 career European goal contributions").
    # appearances is the authoritative total, so the denominator comes from there and the late
    # count is clamped to it. Never hand the model two different goal totals.
    if pid in late.index and int(c.goals) >= 10:
        lg = min(int(late.loc[pid, "late_goals"]), int(c.goals))
        out.append(f"Goals scored in the 75th minute or later: {lg} of his {int(c.goals)} goals")
    if pd.notna(p.international_caps):
        ig = int(p.international_goals) if pd.notna(p.international_goals) else 0
        out.append(f"Senior international record: {int(p.international_caps)} caps, {ig} goals "
                   "(senior national team, NOT youth level)")
    else:
        out.append("Senior international record: unknown — say nothing about caps or "
                   "international goals unless Search establishes them")
    out.append(f"Market value at snapshot: {_eur(p.market_value_in_eur)}")
    if pid in peak_val.index:
        pk = peak_val.loc[pid]
        out.append(
            f"Peak recorded market value: {_eur(pk.market_value_in_eur)} "
            f"(on {_d(pk.valuation_date)})"
        )
    if big_moves.get(pid):
        out.append("Largest recorded transfers: " + "; ".join(big_moves[pid]))
    cm = career_moves.get(pid, [])
    if cm:
        arc = cm if len(cm) <= 2 else [cm[0], cm[-1]]
        out.append(f"Transfer history ({len(cm)} recorded moves) — first and most recent: "
                   + "; ".join(arc))
    if free_moves.get(pid):
        out.append(f"Free or undisclosed-fee moves on record: {int(free_moves[pid])}")
    if isinstance(p.agent_name, str) and p.agent_name:
        out.append(f"Agent: {p.agent_name}")
    if pd.notna(p.contract_expiration_date):
        out.append(f"Contract runs to: {_d(p.contract_expiration_date)}")
    return "\n".join(out)


players["_appearances"] = players["player_id"].map(career["appearances"]).fillna(0).astype(int)
players["_identity"] = [player_identity_block(p) for p in players.itertuples(index=False)]
players["_stats"] = [player_stats_block(p) for p in players.itertuples(index=False)]

# Name collisions drive the "lead with name plus club and era" rule in the prompt.
_dupe_names = set(players.loc[players.duplicated("player_name", keep=False), "player_name"])
players["_name_is_shared"] = players["player_name"].isin(_dupe_names)
print(f"{players['_name_is_shared'].sum():,} players share a name with at least one other player")
print(f"{len(_dupe_names):,} distinct shared names")
print("\n--- sample fact block ---\n")
_ex = players[players["player_name"] == "Lionel Messi"]
if len(_ex):
    print(_ex.iloc[0]["_identity"], "\n", sep="")
    print(_ex.iloc[0]["_stats"])

### 4a.3 — Club facts

796 rows and a genuinely different entity. Clubs get their own fact block and their own prompt:
what makes a club interesting is history, honours, the ground, rivalry, and playing identity, none
of which a player prompt asks for.

In [ ]:
clubs = T["clubs"].copy()
g_all = T["games"]

_home = g_all.rename(columns={"home_club_id": "cid", "home_club_goals": "gf", "away_club_goals": "ga"})
_away = g_all.rename(columns={"away_club_id": "cid", "away_club_goals": "gf", "home_club_goals": "ga"})
club_games = pd.concat(
    [_home[["cid", "season", "competition_id", "gf", "ga", "attendance"]],
     _away[["cid", "season", "competition_id", "gf", "ga", "attendance"]]],
    ignore_index=True,
).dropna(subset=["cid"])
club_games = club_games.dropna(subset=["gf", "ga"])
club_games["won"] = (club_games["gf"] > club_games["ga"]).fillna(False).astype(int)
club_games["drew"] = (club_games["gf"] == club_games["ga"]).fillna(False).astype(int)
club_games["lost"] = (club_games["gf"] < club_games["ga"]).fillna(False).astype(int)

club_rec = club_games.groupby("cid").agg(
    matches=("gf", "count"), won=("won", "sum"), drew=("drew", "sum"), lost=("lost", "sum"),
    gf=("gf", "sum"), ga=("ga", "sum"),
    first_season=("season", "min"), last_season=("season", "max"),
)
club_att = (
    g_all.dropna(subset=["attendance", "home_club_id"])
    .groupby("home_club_id")["attendance"].mean()
)
club_comps: dict[int, list[str]] = defaultdict(list)
for cid, cmp_id, n in (
    club_games.groupby(["cid", "competition_id"]).size().reset_index(name="n")
    .sort_values(["cid", "n"], ascending=[True, False]).itertuples(index=False)
):
    if len(club_comps[int(cid)]) < 5:
        club_comps[int(cid)].append(f"{_comp(cmp_id)} ({n} matches)")

pname = {int(k): v for k, v in
         T["players"].set_index("player_id")["player_name"].to_dict().items()}
top_scorers: dict[int, list[str]] = defaultdict(list)
_ts = (
    apps_s[apps_s["goals"] > 0].groupby(["player_club_id", "player_id"])["goals"].sum()
    .reset_index().sort_values(["player_club_id", "goals"], ascending=[True, False])
)
for cid, pid, g in _ts.itertuples(index=False):
    if len(top_scorers[int(cid)]) < 5:
        top_scorers[int(cid)].append(f"{pname.get(int(pid), pid)} ({int(g)})")

signings: dict[int, list[str]] = defaultdict(list)
for r in tr.itertuples(index=False):
    if pd.isna(r.to_club_id) or pd.isna(r.transfer_fee) or r.transfer_fee <= 0:
        continue
    cid = int(r.to_club_id)
    if len(signings[cid]) < 3:
        signings[cid].append(
            f"{pname.get(int(r.player_id), r.player_id)} from {r.from_club_name or 'Unknown'}, "
            f"{pd.to_datetime(r.transfer_date).year}, €{int(r.transfer_fee):,}"
        )


def club_fact_block(c) -> str:
    cid = int(c.club_id)
    lines = [
        f"Club: {c.club_name}",
        f"Primary domestic competition: {_comp(c.domestic_competition_id, with_country=True)}",
        f"Stadium: {c.stadium_name}" + (f", capacity {int(c.stadium_seats):,}" if pd.notna(c.stadium_seats) and c.stadium_seats else ""),
        f"Head coach at snapshot: {c.coach_name}" if isinstance(c.coach_name, str) and c.coach_name else "",
        f"Squad at snapshot: {int(c.squad_size)} players" if pd.notna(c.squad_size) else "",
        f"Average squad age: {c.average_age}" if pd.notna(c.average_age) else "",
        f"Foreign players: {int(c.foreigners_number)} ({c.foreigners_percentage}%)" if pd.notna(c.foreigners_number) else "",
        f"Current internationals in the squad: {int(c.national_team_players)}" if pd.notna(c.national_team_players) else "",
        f"Net transfer record: {_eur(abs(c.net_transfer_record_eur))} "
        f"{'net spend' if (pd.notna(c.net_transfer_record_eur) and c.net_transfer_record_eur < 0) else 'net income'}"
        if pd.notna(c.net_transfer_record_eur) else "",
        f"Transfermarkt profile: {c.url}" if isinstance(c.url, str) and c.url else "",
    ]
    if cid in club_rec.index:
        r = club_rec.loc[cid]
        lines += [
            f"Record in the Big 5 leagues and the UEFA Champions and Europa League, "
            f"{int(r.first_season)}–{int(r.last_season)}: "
            f"{int(r.matches)} matches, {int(r.won)} won, {int(r.drew)} drawn, {int(r.lost)} lost",
            f"Goals scored {int(r.gf):,}, conceded {int(r.ga):,}",
        ]
    else:
        lines.append(
            "This club did not play in the Big 5 leagues or the UEFA Champions or Europa League "
            "between 2012 and 2025. Write about the club on its own terms — its own league, its "
            "history, its ground, its identity — and do not remark on the absence of those "
            "fixtures or on how it relates to other leagues."
        )
    if cid in club_att.index and pd.notna(club_att.loc[cid]):
        lines.append(f"Average recorded home attendance: {int(club_att.loc[cid]):,}")
    if club_comps.get(cid):
        lines.append("Competitions featured in: " + "; ".join(club_comps[cid]))
    if top_scorers.get(cid):
        lines.append("Leading scorers for the club in these competitions: " + "; ".join(top_scorers[cid]))
    if signings.get(cid):
        lines.append("Largest recorded incoming transfers: " + "; ".join(signings[cid]))
    return "\n".join(x for x in lines if x)


clubs["_facts"] = [club_fact_block(c) for c in clubs.itertuples(index=False)]
clubs["_matches"] = clubs["club_id"].map(club_rec["matches"]).fillna(0).astype(int)
print(f"club fact blocks built for {len(clubs):,} clubs")
print(f"clubs with no matches in scope: {(clubs['_matches'] == 0).sum():,}")
print("\n--- sample club fact block ---\n")
print(clubs.sort_values("_matches", ascending=False).iloc[0]["_facts"])

## 5 · Stage 4b — The prompts

**Bump `PROMPT_VERSION` on every edit here.** It lands in the manifest, and it is the only thing
that makes a future regeneration explicable.

Two prompts, one per entity type. Both are built around the same three obligations, which come
straight from what Lab 1 needs the text to do:

**1 · Lead with the name, the club, and the era.** The first sentence carries the subject's name
verbatim. This is what the BM25 index bites on in Lab 1 Task 3, and it is what lets vector search
separate the six players called Paulinho into six comprehensible results instead of one
undifferentiated blob.

**2 · Describe style in the language a fan would use, not in statistics.** This is the whole ball
game for the semantic query. `"diminutive Argentine playmaker with a magical left foot"` matches
nothing in a restated stat line. It matches a profile that says *small*, *Argentine*, *creator*,
*left-footed*. The prompt asks for stature and build in words, nationality as an adjective, the
role actually played rather than the listed position, and what the preferred foot means for how
the player plays. Everything else in the prompt is negotiable; this paragraph is not.

**3 · Never invent a number.** The fact block is authoritative and the model is told so twice. It
is also told, explicitly, that the stats cover only the Big 5 leagues plus the Champions and
Europa Leagues from 2012 to 2025 — otherwise a model that knows the player's real career will
present a partial total as a career total, which is the most common way this class of prompt
produces something technically hallucinated from correct inputs.

In [ ]:
PLAYER_PROMPT = """You are writing a player profile for CymbalGoal, a global football fan and \
analytics platform. Fans read these profiles, and they search them.

## Who you are writing about — authoritative
{identity}

## Verified statistics — authoritative
These figures cover the Big 5 European leagues (Premier League, LaLiga, Serie A, Bundesliga, \
Ligue 1) and the UEFA Champions League and Europa League, seasons 2012 through 2025. If this \
player's career ran wider or longer, they are not career totals.

HOW TO USE THEM: attach every figure to the competition and the years it belongs to, the way a \
match report would — "sixty-one goals across four Serie A seasons", "in his Champions League \
campaigns between 2014 and 2019". That is how you stay accurate without ever describing where \
the numbers came from.

{stats}

## Your task

First, use Google Search to work out who this player is. Match on the birth date, the birth city, \
and the club history above — do not settle for a name match. {collision_note}

Then write a profile of about {target_words} words in flowing prose.

Requirements:

- The FIRST SENTENCE must contain the player's full name exactly as written above, the club they \
are most associated with, and the period they played. A reader who searches the name must be able \
to tell instantly which player this is.
- Describe how this player actually plays, in the ordinary descriptive language a supporter would \
use. Cover, where you can support it:
  - their physical character — stature, build, pace, strength in the air, balance — described in \
words, not only in centimetres
  - their nationality as an adjective, naturally placed
  - the role they genuinely occupy on the pitch — playmaker, poacher, destroyer, ball-playing \
centre-back, inverted winger, sweeper-keeper, target man — rather than only the position label above
  - what their preferred foot means for the way they play
  - their temperament and reputation, where these are documented
- Weave the verified numbers into the prose where they carry the story. Do not list them all, and \
do not write a statistical summary.
- Add what Search tells you that the statistics cannot: honours, international tournaments, \
notable moments, how they are regarded.

Hard rules:

- Never contradict the verified statistics, and never state a figure that contradicts them.
- Any number you introduce that is not in the verified block must come from Search and must \
concern something the block does not cover.
- If Search turns up little or nothing about this player, write a shorter, honest profile from the \
verified facts alone. A thin true profile is fine. An invented one is not.
- Plain prose only. No bullet points, no headings, no markdown, and no emphasis markers of any \
kind — not **bold**, not *italics*, not _underscores_.
- **You are writing for a football supporter, not for a data engineer. Never refer to the data \
itself.** Do not write "this dataset", "the provided dataset", "the verified statistics", "the \
specified leagues", "covered competitions", "the snapshot", "in this profile", "the scope of \
these statistics", or any variant. When a figure needs a boundary, give the boundary as a \
competition and a span of years — never as a description of what the data does or does not \
include. A reader must never be able to tell that a fact block existed.
- Do not mention Google Search, this prompt, or that you are an AI.
- Do not hedge with "appears to be" or "reportedly" — either you can support it or you leave it out.
- Write every figure as a NUMERAL — "26 goals", "79 yellow cards", "the 75th minute" — never as \
words ("twenty-six goals"). Keep the thousands separators on large figures: "€5,000,000", not \
"€5000000". This applies to STATISTICS. Ordinary prose counts stay words where a numeral would \
read oddly — "the first five European Cups", never "the 1st 5 European Cups". Readers search these profiles for numbers, and a spelled-out figure \
is invisible to a keyword search.
- Write ONE profile. Do not draft, restart, revise, or emit an alternative version, and never \
wrap any part of the text in brackets.
- Use no square brackets and no citation markers of any kind — not "[1]", not "[INDEX_0]", not \
"[source]". Parentheses for an aside are fine. Your output is prose for a supporter to read, not \
a document with references.
- The blocks above are notes for you, not text to quote. NEVER copy a label out of them — not \
"Full name:", not "Born:", not "Current club:", not "Senior international record:", not any \
other. Your first word is the first word of the profile.
- Vary how you begin. "X is most associated with Y" is one option among many, not a template — \
open with the player's defining quality, era, position, or a signature moment where that reads \
better.
{editorial}
Write only the profile text. Nothing before it, nothing after it."""

EDITORIAL_RULE = """- Reputation and temperament belong here where they concern football — \
competitiveness, leadership, big-game temperament, work rate, discipline on the pitch. Do NOT \
include off-pitch controversy, social-media sentiment, criticism of the player by named \
individuals, legal or personal matters, or judgements about their character as a person. If the \
only notable thing Search returns about someone is a controversy, leave it out and write a \
shorter profile.
"""

COLLISION_NOTE = (
    "⚠️ This player SHARES A NAME with at least one other footballer. Getting the "
    "wrong one is the most likely failure here. Verify against the birth date and club history "
    "before you write a single word, and make the opening sentence unmistakable about which "
    "player this is."
)

CLUB_PROMPT = """You are writing a club profile for CymbalGoal, a global football fan and \
analytics platform. Fans read these profiles, and they search them.

## The club — authoritative
Any match record below covers the Big 5 European leagues and the UEFA Champions and Europa \
League, seasons 2012 through 2025 — not the club's full history. Attach those figures to the \
competitions and years they belong to, the way a season review would, and never describe where \
they came from.

{facts}

## Your task

First, use Google Search to identify this club and gather its history.

Then write a profile of about {target_words} words in flowing prose.

Requirements:

- The FIRST SENTENCE must contain the club's name exactly as written above, the city and country \
it plays in, and the league it belongs to.
- Cover, where you can support it:
  - founding and the shape of the club's history — the eras that define it
  - major honours, domestic and European
  - the stadium: its name, its character, what it is like on a matchday
  - the rivalries that matter, and who they are against
  - the club's playing identity and traditions — how they are understood to play, what the \
supporters expect, the colours and nicknames
  - where the club stands now, using the squad and transfer figures above
- Weave the verified numbers in where they carry the story rather than listing them.

Hard rules:

- Never contradict the verified figures, and never state a figure that contradicts them.
- Any number you introduce that is not in the block above must come from Search.
- If Search turns up little about this club — a small side that appears here only as a transfer \
counterparty — write a shorter, honest profile from the verified facts. A thin true profile is \
fine. An invented one is not.
- Plain prose only. No bullet points, no headings, no markdown, and no emphasis markers of any \
kind — not **bold**, not *italics*, not _underscores_. Nicknames go in plain text or quotation \
marks.
- Write every statistic as a NUMERAL, never as words, and keep thousands separators on large \
figures: "€9,000,000", not "€9000000". Ordinary prose counts stay words — "the first five \
European Cups", never "the 1st 5 European Cups".
- Write ONE profile. Do not draft, restart, or emit an alternative version, and never wrap any \
part of the text in brackets.
- Use no square brackets and no citation markers of any kind — not "[1]", not "[INDEX_0]". \
Parentheses for an aside are fine.
- The block above is notes for you, not text to quote. NEVER copy a label out of it — not \
"Club:", not "Stadium:", not "Head coach at snapshot:", not any other.
- **You are writing for a football supporter, not for a data engineer. Never refer to the data \
itself.** Do not write "this dataset", "covered competitions", "the snapshot", or any variant, \
and never remark on which competitions are or are not represented. A reader must never be able to \
tell that a fact block existed.
- Do not mention Google Search, this prompt, or that you are an AI.
{editorial}
Write only the profile text. Nothing before it, nothing after it."""


_EDITORIAL = EDITORIAL_RULE if EXCLUDE_OFF_PITCH_CONTROVERSY else ""


def build_player_prompt(row) -> str:
    return PLAYER_PROMPT.format(
        identity=row["_identity"],
        stats=row["_stats"],
        target_words=TARGET_WORDS,
        collision_note=COLLISION_NOTE if row["_name_is_shared"] else "",
        editorial=_EDITORIAL,
    )


def build_club_prompt(row) -> str:
    return CLUB_PROMPT.format(facts=row["_facts"], target_words=TARGET_WORDS,
                              editorial=_EDITORIAL)


print(f"player prompt ≈ {len(build_player_prompt(players.iloc[0])) // 4:,} tokens")
print(f"club prompt   ≈ {len(build_club_prompt(clubs.iloc[0])) // 4:,} tokens")

## 6 · Stage 4c — The generation engine

The CymbalFlix harness's shape, with the six fixes.

In [ ]:
RETRYABLE_CODES = {408, 409, 425, 429, 500, 502, 503, 504}


def thinking_config() -> "gtypes.ThinkingConfig":
    """
    Gemini 3.x swapped the integer `thinking_budget` for a `thinking_level` enum and removed the
    ability to turn thinking off. If the installed google-genai predates that change it will
    reject the field, so fall back rather than failing 14,235 times.
    """
    try:
        return gtypes.ThinkingConfig(thinking_level=THINKING_LEVEL)
    except Exception as exc:  # pydantic ValidationError on older SDKs
        print(f"⚠️  thinking_level rejected by this google-genai build ({type(exc).__name__}). "
              "Upgrade the SDK. Falling back to thinking_budget=0.")
        return gtypes.ThinkingConfig(thinking_budget=0)


class _Fatal(Exception):
    """Retrying will not help. Fail this row now instead of paying for it five times."""


@dataclass
class GenTask:
    kind: str      # "player" | "club"
    key: int       # player_id | club_id
    prompt: str
    ground: bool


class Meter:
    """Thread-safe token, cost, and outcome accounting."""

    def __init__(self) -> None:
        self._lock = threading.Lock()
        self.in_tokens = 0
        self.out_tokens = 0
        self.thinking_tokens = 0   # billed at the OUTPUT rate on Gemini 3.x
        self.grounded_requests = 0
        self.grounding_metadata_present = 0
        self.calls = 0
        self.retries = 0
        self.failures = 0
        self.rate_limited = 0

    def add(self, *, ti: int, to: int, tt: int = 0, grounded: bool, had_meta: bool,
            retries: int, failed: bool, rate_limited: int = 0):
        with self._lock:
            self.in_tokens += ti
            self.out_tokens += to
            self.thinking_tokens += tt
            self.calls += 1
            self.retries += retries
            self.rate_limited += rate_limited
            self.grounded_requests += int(grounded)
            self.grounding_metadata_present += int(had_meta)
            self.failures += int(failed)

    def cost(self) -> dict[str, float]:
        # out_tokens already includes reasoning on Gemini 3.x; thinking_tokens is reported
        # separately for visibility, not added again.
        tok = (self.in_tokens / 1e6) * PRICE_IN_PER_M + (self.out_tokens / 1e6) * PRICE_OUT_PER_M
        billable = max(0, self.grounded_requests - GROUNDING_FREE_PER_MONTH)
        gnd = (billable / 1000.0) * PRICE_GROUNDING_PER_1K_REQ
        return {"tokens_usd": tok, "grounding_usd": gnd, "total_usd": tok + gnd,
                "grounding_free_applied": min(self.grounded_requests, GROUNDING_FREE_PER_MONTH)}


def _status_code(exc: Exception) -> int | None:
    for attr in ("code", "status_code"):
        v = getattr(exc, attr, None)
        if isinstance(v, int):
            return v
    m = re.search(r"\b(4\d\d|5\d\d)\b", str(exc))
    return int(m.group(1)) if m else None


_RETRY_AFTER_RE = re.compile(r"(?:retry|try again)\s+(?:in|after)\s+([\d.]+)\s*(s|sec|second)",
                             re.I)


def _suggested_delay(exc: Exception) -> float | None:
    """
    Vertex quota errors often say "Please retry in 27.5 seconds", and some responses carry a
    Retry-After header. The server knows the window better than our exponential does — use it.
    """
    for attr in ("response", "_response"):
        r = getattr(exc, attr, None)
        hdrs = getattr(r, "headers", None)
        if hdrs:
            for key in ("retry-after", "Retry-After", "x-retry-after"):
                v = hdrs.get(key) if hasattr(hdrs, "get") else None
                if v:
                    try:
                        return float(v)
                    except (TypeError, ValueError):
                        pass
    m = _RETRY_AFTER_RE.search(str(exc))
    return float(m.group(1)) if m else None


def _backoff(attempt: int, code: int | None = None, exc: Exception | None = None) -> float:
    """
    FIX 3: exponential with full jitter. A flat 2s sleep at 50 workers just re-collides two
    seconds later. Jitter is applied INSIDE the cap — capping first and multiplying after lets
    the delay reach 1.5x BACKOFF_CAP_S, which makes the cap a suggestion rather than a cap.

    429 is handled separately. Quota windows are per-minute, so a sub-second-to-few-second retry
    is guaranteed to fail AND adds load to the saturated window. Floor it at
    RATE_LIMIT_MIN_WAIT_S, and prefer whatever delay the server itself suggested.
    """
    jitter = 0.5 + random.random()
    if exc is not None:
        hinted = _suggested_delay(exc)
        if hinted:
            return min(BACKOFF_CAP_S, hinted * jitter + hinted * 0.5)
    wait = (BACKOFF_BASE ** attempt) * jitter
    if code == 429:
        wait = max(RATE_LIMIT_MIN_WAIT_S * jitter, wait)
    return min(BACKOFF_CAP_S, wait)


class Throttle:
    """
    Adaptive concurrency. The honest answer to a sustained 429 is not more retries — it is fewer
    workers. Adjusted between batches so the pool size is stable within a batch.
    """

    def __init__(self, start: int, floor: int = MIN_WORKERS):
        self.start, self.floor, self.workers = start, floor, start
        self._seen_calls = 0
        self._seen_429 = 0
        self.history: list[tuple[int, float]] = []

    def adjust(self, meter: "Meter") -> int:
        calls = meter.calls - self._seen_calls
        limited = meter.rate_limited - self._seen_429
        self._seen_calls, self._seen_429 = meter.calls, meter.rate_limited
        if calls <= 0:
            return self.workers
        rate = limited / calls
        before = self.workers
        if rate > THROTTLE_TRIGGER_RATE:
            self.workers = max(self.floor, int(self.workers * 0.6))
        elif rate == 0 and self.workers < self.start:
            self.workers = min(self.start, int(self.workers * 1.5) + 1)
        if self.workers != before:
            print(f"    throttle: {rate:.0%} rate-limited -> workers {before} -> {self.workers}")
        self.history.append((self.workers, rate))
        return self.workers


def generate_one(task: GenTask, meter: Meter) -> dict[str, Any]:
    cfg_kwargs: dict[str, Any] = {
        "temperature": TEMPERATURE,  # FIX 4: was 1.3
        "max_output_tokens": MAX_OUTPUT_TOKENS,
        "thinking_config": thinking_config(),
        **afc_off(),
    }
    if task.ground:
        cfg_kwargs["tools"] = [gtypes.Tool(google_search=gtypes.GoogleSearch())]
    config = gtypes.GenerateContentConfig(**cfg_kwargs)

    last_err = ""
    n_429 = 0
    deadline = time.time() + RETRY_BUDGET_S   # a wall clock, so a slow wall can't hold a row forever
    for attempt in range(MAX_RETRIES):
        try:
            resp = get_client(GEN_LOCATION).models.generate_content(
                model=GEN_MODEL, contents=task.prompt, config=config
            )
            cand = (getattr(resp, "candidates", None) or [None])[0]
            finish = str(getattr(cand, "finish_reason", "") or "")

            # Distinguish the two ways a thinking model returns nothing useful. Neither is worth
            # five retries: MAX_TOKENS means reasoning ate the budget and will do so again, and
            # SAFETY means the content was blocked, not that the call failed.
            if "MAX_TOKENS" in finish.upper():
                raise _Fatal(
                    f"hit MAX_OUTPUT_TOKENS ({MAX_OUTPUT_TOKENS:,}) — reasoning consumed the "
                    "budget. Raise MAX_OUTPUT_TOKENS or lower THINKING_LEVEL."
                )
            if "SAFETY" in finish.upper() or "RECITATION" in finish.upper():
                raise _Fatal(f"blocked by the model, finish_reason={finish}")

            text = (resp.text or "").strip()
            if not text:
                raise RuntimeError(f"empty response body (finish_reason={finish or 'unknown'})")

            um = getattr(resp, "usage_metadata", None)
            ti = int(getattr(um, "prompt_token_count", 0) or 0)
            to = int(getattr(um, "candidates_token_count", 0) or 0)
            tt = int(getattr(um, "thoughts_token_count", 0) or 0)
            had_meta = bool(
                getattr(resp, "candidates", None)
                and getattr(resp.candidates[0], "grounding_metadata", None)
            )
            meter.add(ti=ti, to=to, tt=tt, grounded=task.ground, had_meta=had_meta,
                      retries=attempt, failed=False, rate_limited=n_429)
            return {
                "id": task.key, "text": text, "words": len(text.split()),
                "grounded": task.ground, "grounding_metadata": had_meta,
                "attempts": attempt + 1, "in_tokens": ti, "out_tokens": to,
                "thinking_tokens": tt, "error": "",
            }

        except _Fatal as exc:
            last_err = f"Fatal: {exc}"
            break
        except Exception as exc:  # noqa: BLE001
            code = _status_code(exc)
            last_err = f"{type(exc).__name__} {code or ''}: {str(exc)[:300]}"
            # FIX 3 continued: 400/403/404 are our bug, not the service's. Don't burn retries.
            if code == 429:
                n_429 += 1
            if code is not None and code not in RETRYABLE_CODES and 400 <= code < 500:
                break
            if attempt < MAX_RETRIES - 1:
                wait = _backoff(attempt, code, exc)
                if time.time() + wait > deadline:
                    last_err += f" | gave up: {RETRY_BUDGET_S:.0f}s retry budget exhausted"
                    break
                time.sleep(wait)

    meter.add(ti=0, to=0, tt=0, grounded=False, had_meta=False, retries=MAX_RETRIES, failed=True,
              rate_limited=n_429)
    return {
        "id": task.key, "text": f"{ERROR_SENTINEL} {last_err}]]", "words": 0,
        "grounded": task.ground, "grounding_metadata": False,
        "attempts": MAX_RETRIES, "in_tokens": 0, "out_tokens": 0, "thinking_tokens": 0,
        "error": last_err,
    }


def checkpoint_path(kind: str) -> Path:
    return WORK_DIR / f"gen_{kind}_{PROMPT_VERSION}.csv"


def error_log_path(kind: str) -> Path:
    stamp = RUN_STARTED_UTC.replace(":", "").replace("-", "")
    return WORK_DIR / f"errors_{kind}_{stamp}.log"


def load_checkpoint(kind: str) -> dict[int, dict[str, Any]]:
    """Successes only. Rows carrying the sentinel are dropped so the resume retries them."""
    p = checkpoint_path(kind)
    if not p.exists():
        return {}
    df = pd.read_csv(p)
    df["text"] = df["text"].fillna("")
    ok = df[~df["text"].str.startswith(ERROR_SENTINEL) & (df["text"].str.strip() != "")]
    dropped = len(df) - len(ok)
    print(f"resume: {len(ok):,} usable rows in checkpoint, {dropped:,} failures queued for retry")
    return {int(r["id"]): dict(r) for _, r in ok.iterrows()}


def run_generation(tasks: list[GenTask], kind: str, resume: bool = True) -> pd.DataFrame:
    """
    FIX 1 lives here. Checkpoints fire on a batch counter, never on len(results) % SAVE_INTERVAL.
    The old form only worked on a cold run, where results grew by exactly BATCH_SIZE. On a resume
    the list starts pre-loaded, the modulo stops landing, and intermediate saves silently cease —
    so a crash halfway through a resumed run loses everything since the resume point.
    """
    done = load_checkpoint(kind) if resume else {}
    todo = [t for t in tasks if t.key not in done]
    print(f"{kind}: {len(done):,} already done, {len(todo):,} to generate")
    if not todo:
        return pd.DataFrame(list(done.values()))

    meter = Meter()
    throttle = Throttle(MAX_WORKERS)
    results: list[dict[str, Any]] = list(done.values())
    errlog = error_log_path(kind)
    t0 = time.time()
    batches = [todo[i:i + BATCH_SIZE] for i in range(0, len(todo), BATCH_SIZE)]

    def save() -> None:
        pd.DataFrame(results).to_csv(checkpoint_path(kind), index=False)

    with tqdm(total=len(todo), desc=f"generate {kind}", unit="row") as bar:
        for bi, batch in enumerate(batches, start=1):
            with ThreadPoolExecutor(max_workers=throttle.workers) as pool:
                futures = {pool.submit(generate_one, t, meter): t for t in batch}
                for fut in as_completed(futures):
                    r = fut.result()
                    results.append(r)
                    if r["error"]:
                        with open(errlog, "a", encoding="utf-8") as fh:
                            fh.write(f"{datetime.now(timezone.utc).isoformat()}\t{kind}\t{r['id']}\t{r['error']}\n")
                    bar.update(1)
            throttle.adjust(meter)
            if bi % SAVE_EVERY_N_BATCHES == 0:   # <-- FIX 1
                save()
                bar.set_postfix_str(f"saved @ batch {bi}/{len(batches)}")
    save()

    elapsed = time.time() - t0
    c = meter.cost()
    print(f"\n{kind}: {len(todo):,} generated in {elapsed/60:.1f} min "
          f"({len(todo)/max(elapsed,1):.1f} rows/s)")
    print(f"  tokens     in {meter.in_tokens:,}  out {meter.out_tokens:,} "
          f"(of which reasoning {meter.thinking_tokens:,}, "
          f"{meter.thinking_tokens / max(meter.out_tokens, 1):.0%})")
    print(f"  grounded   {meter.grounded_requests:,} requests, "
          f"{meter.grounding_metadata_present:,} came back with grounding metadata")
    print(f"  retries    {meter.retries:,}    failures {meter.failures:,}    "
          f"429s {meter.rate_limited:,}")
    if meter.rate_limited:
        print(f"  throttle   workers ended at {throttle.workers} of {MAX_WORKERS}. Sustained 429s "
              "mean the quota is the ceiling — lower MAX_WORKERS rather than raising MAX_RETRIES.")
    print(f"  cost       tokens ${c['tokens_usd']:.2f} + grounding ${c['grounding_usd']:.2f} "
          f"= ${c['total_usd']:.2f}   ({c['grounding_free_applied']:,.0f} grounded requests "
          "covered by the free monthly allowance)")
    if len(todo):
        per_1k = c["total_usd"] / len(todo) * 1000
        print(f"  unit       ${per_1k:.2f} per 1,000 rows  ->  "
              f"${per_1k * 14.235:,.0f} projected for all 14,235")
    if meter.failures:
        print(f"  ⚠️  {meter.failures:,} rows failed. Re-run this cell to retry them: {errlog}")
    return pd.DataFrame(results)


def make_player_tasks(df: pd.DataFrame) -> list[GenTask]:
    return [
        GenTask("player", int(r["player_id"]), build_player_prompt(r),
                USE_GROUNDING and r["_appearances"] >= GROUND_MIN_APPEARANCES)
        for _, r in df.iterrows()
    ]


def make_club_tasks(df: pd.DataFrame) -> list[GenTask]:
    return [
        GenTask("club", int(r["club_id"]), build_club_prompt(r), USE_GROUNDING)
        for _, r in df.iterrows()
    ]


_n_ground = int((players["_appearances"] >= GROUND_MIN_APPEARANCES).sum()) if USE_GROUNDING else 0
print(f"\nprojected full run: {len(players):,} players + {len(clubs):,} clubs")
print(f"  grounded requests: {_n_ground + (len(clubs) if USE_GROUNDING else 0):,}")
_billable = max(0, _n_ground + (len(clubs) if USE_GROUNDING else 0) - GROUNDING_FREE_PER_MONTH)
print(f"  grounding cost:    ${_billable * PRICE_GROUNDING_PER_1K_REQ / 1000:,.2f} "
      f"({GROUNDING_FREE_PER_MONTH:,} free/month applied; the allowance is account-wide, so "
      "treat this as a floor)")

## 6.5 · The corruption screen

Defined here, **before** the sample, and that placement is the point.

In the v1.4 sample two of 26 profiles were broken — Charlie Daniels restarted mid-sentence
inside an unclosed bracket, and Unai Albizua opened by copying the `Full name:` label. Both
have gates. Neither fired, because the gates lived in the QA section and the QA section only
ever ran against the **full** corpus. The sample cell just printed. A check that does not run
at the moment you are deciding whether to spend $175 is not a check.

Both failures are stochastic at roughly 4% each, so a full run should be expected to produce
something like 500 of each. `quarantine_corrupt()` below turns that from a manual cleanup into
one cell: it rewrites corrupt rows in the checkpoint with the error sentinel, which is exactly
what the resume path already looks for, so re-running the generation cell regenerates them.

In [ ]:
PLUMBING_RE = (
    r"(?i)(?:google search|as an ai|language model|this prompt"
    r"|(?:this|the|our|provided|given|supplied)\s+(?:data\s?set|dataset)"
    r"|covered competitions|covered leagues|verified statistics|verified figures"
    r"|specified\s+(?:\w+\s+){0,2}(?:leagues|competitions)"
    r"|(?:covered|given|listed)\s+(?:\w+\s+){0,2}(?:leagues|competitions)"
    r"|the snapshot|at the snapshot|in this profile"
    r"|scope of (?:these|the) statistic|transfer counterparty|fact block|CymbalGoal)"
)
MAX_PLUMBING_RATE = 0.02

LABEL_LEAK_RE = (
    r"(?im)(?:^|(?<=[.!?]\s))\s*(?:full name|first name|last name|born|citizenship|position|"
    r"preferred foot|height|current club|club at snapshot|clubs he played for|"
    r"transfermarkt profile|senior international record|appearances|minutes played|"
    r"goals per 90 minutes|market value at snapshot|peak recorded market value|"
    r"largest recorded transfers|transfer history|agent|contract runs to|"
    r"primary domestic competition|stadium|head coach at snapshot|squad at snapshot|"
    r"average squad age|foreign players|net transfer record|competitions featured in|"
    r"leading scorers|record in the big 5)\s*:"
)


def _restarts(text: str, prefix_chars: int = 55) -> bool:
    """The profile's own opening reappears later — the model started over verbatim."""
    t = " ".join(str(text).split())
    if len(t) < prefix_chars * 2:
        return False
    return t[:prefix_chars] in t[prefix_chars:]


def _has_brackets(text: str) -> bool:
    """
    ANY square or curly bracket is a defect, not just an unbalanced one.

    v1.5 shipped Haaland's profile with a grounding citation marker in it — "on the way to a
    historic continental treble [INDEX_0]." — and the unbalanced-bracket check waved it through
    because the brackets matched. Balance was never the property that mattered. A football
    profile written for supporters has no legitimate use for square brackets at all, so the rule
    is simply that they must not appear. Parentheses are untouched: "O Polvo (The Octopus)" is
    fine.
    """
    return bool(re.search(r"[\[\]{}]", str(text)))


def _unbalanced(text: str) -> bool:
    """Kept as a distinct signal: unbalanced brackets mean a truncated or restarted profile."""
    t = str(text)
    return t.count("[") != t.count("]") or t.count("{") != t.count("}")


def _name_repeats_in_opening(text: str, name: str, window: int = 300) -> bool:
    """
    ADVISORY ONLY — see ADVISORY_FLAGS. The subject's name appears twice early. This was fatal
    through v2.2 and it was wrong: at full scale it fired 8 times in 14,235 rows and every one
    was ordinary prose. Kept as a signal to eyeball, never as a gate.
    """
    if not isinstance(name, str) or len(name.split()) < 2:
        return False
    return " ".join(str(text).split())[:window].count(name) > 1


# Splitting on "period + space" is wrong for this corpus, because a great many subjects END in an
# abbreviation: "Società Sportiva Lazio S.p.A.", "SK Sigma Olomouc, a.s.", "Charly Musonda Jr.".
# A naive split turns one sentence into two, the phantom first half is exactly the subject's name,
# and any "does the profile open twice?" check fires on every one of them. Requiring the next
# sentence to START with a capital fixes it: real prose always does, and a continuation ("…a.s. is
# based in Olomouc") never does. This cost five false positives before it was caught.
_SENT_SPLIT_RE = re.compile(r"(?<=[.!?])\s+(?=[\"'“(]?[A-Z0-9])")


def _duplicated_opening(text: str, name: str) -> bool:
    """
    A restart: the profile introduces its subject twice. This is what the name-repeat check was
    reaching for and kept missing by a mile.

    The discriminator is not *whether* the name recurs but *where*. A restart re-opens the
    biography, so the subject is named at the START of two consecutive sentences, or the first
    "sentence" is a bare stub of the name before the real opening begins:

        "Sturm Graz. Sturm Graz is an Austrian club founded in 1909…"          ← stub
        "Ivan Rakitic was a metronome. Ivan Rakitic was a metronome who…"      ← twice opening
        "Daniels established himself as… Daniels cemented his reputation as…"  ← paraphrased

    Every one of the 8 false positives the old check produced has the second mention buried
    mid-sentence, where it belongs, and is untouched by this:

        "Luton Town is an English club based in the town of Luton…"     ← the town it is named for
        "the club became Manchester United in 1902"                     ← a renaming it must mention
        "The son of iconic Venezuelan playmaker Juan Arango…"           ← a father and son
        "André Luiz Silva do Nascimento, commonly known as André Luiz"  ← the prompt ASKS for this

    Note the last one especially: the old gate blocked the exact disambiguation the prompt
    requires, which is how a check earns the right to be replaced rather than merely loosened.
    """
    if not isinstance(name, str) or not name.strip():
        return False
    name = name.strip()
    t = " ".join(str(text).split())
    sents = _SENT_SPLIT_RE.split(t)
    if len(sents) < 2:
        return False
    first, second = sents[0].strip(), sents[1].strip()

    def opens_with_name(s: str) -> bool:
        return s.lower().startswith(name.lower())

    # BOTH of the first two sentences must introduce the subject by name. Requiring it of the
    # second as well as the first is what separates a restart from a sentence that merely happens
    # to be split badly: "Charly Musonda Jr. | came through at Chelsea" is one sentence wearing a
    # false seam, and its second half does not re-introduce anybody.
    if not (opens_with_name(first) and opens_with_name(second)):
        return False
    # For a multi-word name, opening twice with it IS the restart signature.
    if len(name.split()) >= 2:
        return True
    # For a one-word name, "Danilo is a right-back. Danilo joined City in 2017." is ordinary
    # prose, so require the stub shape: a first sentence that is nothing but the name.
    return len(first.rstrip(".!?")) <= len(name) + 3


# v1.6 prose blemishes no check could see: "Italy top stage", "Europe premier competition",
# "the Scorpions historic run". Narrow by design — a proper noun followed by a superlative
# adjective almost always wants an apostrophe, and this pattern will not fire on ordinary prose.
# Warning only: it is a cosmetic flaw, not a corrupt row.
POSSESSIVE_RE = (
    r"\b(?:Italy|Spain|France|Germany|England|Portugal|Brazil|Argentina|Denmark|Norway|Sweden"
    r"|Belgium|Netherlands|Hungary|Europe|Scorpions|Cherries|Rossoneri|Blaugrana)\s+"
    r"(?:top|premier|greatest|finest|defining|historic|golden|modern|leading|elite|best|first"
    r"|biggest|proudest|only)\b"
)


FLAG_COLUMNS = [
    "carries the error sentinel", "empty or whitespace",
    f"under {MIN_ACCEPTABLE_WORDS} words", "restarts mid-profile (corrupt)",
    "brackets or citation markers (corrupt)", "introduces the subject twice (corrupt)",
    "repeats the name in the opening", "copies a fact-block label (corrupt)",
    "leaks the data plumbing", "contains markdown formatting", "uses emphasis markers",
    "drops thousands separators", "spells figures as words", "drops a possessive apostrophe",
]


def screen(df: pd.DataFrame, names: dict | None = None, col: str = "text") -> pd.DataFrame:
    """Returns the per-row problem flags. Raises nothing — the gate decides what is fatal."""
    if not len(df):
        return pd.DataFrame(columns=FLAG_COLUMNS, dtype=bool)
    s = df[col].fillna("").astype(str)
    nm = df["id"].map(names) if names is not None else pd.Series([""] * len(df), index=df.index)
    return pd.DataFrame({
        "carries the error sentinel": s.str.contains(re.escape(ERROR_SENTINEL), regex=True),
        "empty or whitespace": s.str.strip() == "",
        f"under {MIN_ACCEPTABLE_WORDS} words": s.str.split().str.len().fillna(0) < MIN_ACCEPTABLE_WORDS,
        "restarts mid-profile (corrupt)": s.map(_restarts),
        "brackets or citation markers (corrupt)": s.map(_has_brackets),
        "introduces the subject twice (corrupt)": [
            _duplicated_opening(t, n) for t, n in zip(s, nm.fillna(""))
        ],
        "repeats the name in the opening": [
            _name_repeats_in_opening(t, n) for t, n in zip(s, nm.fillna(""))
        ],
        "copies a fact-block label (corrupt)": s.str.contains(LABEL_LEAK_RE, regex=True),
        "leaks the data plumbing": s.str.contains(PLUMBING_RE, regex=True),
        "contains markdown formatting": s.str.contains(r"(?:^|\n)\s*[-*#]\s|\*\*", regex=True),
        "uses emphasis markers": s.str.contains(r"\*\w|\w\*|(?<![a-zA-Z0-9])_\w+_", regex=True),
        "drops thousands separators": s.str.contains(r"[€£$]\s?\d{5,}", regex=True),
        "spells figures as words": s.str.contains(
            r"(?i)\b(?:twenty|thirty|forty|fifty|sixty|seventy|eighty|ninety|hundred)[- ]"
            r"(?:one|two|three|four|five|six|seven|eight|nine|first|second|third|fourth|fifth|"
            r"sixth|seventh|eighth|ninth)\b", regex=True),
        "drops a possessive apostrophe": s.str.contains(POSSESSIVE_RE, regex=True),
    }, index=df.index)


# "repeats the name in the opening" is ADVISORY, not fatal — it is DEMOTED, and replaced at the
# gate by "introduces the subject twice", which measures position rather than mere recurrence.
#
# The demotion is evidence-driven, not convenience-driven. The old check was fatal through v2.2 on
# the strength of one true positive in a 25-row sample. At full scale it fired 8 times in 14,235
# rows and every one was ordinary prose — including the full-name-then-short-name disambiguation
# the prompt explicitly asks for. Precision 0/8 is not a threshold to tune; it means the check was
# measuring the wrong property.
#
# Crucially, it is REPLACED rather than merely dropped. _duplicated_opening() catches all three
# restart shapes the old check was defending against — stub, verbatim, paraphrased — and none of
# the 8. Relaxing a gate because it is inconvenient is how corpora rot; relaxing one because a
# sharper instrument exists is just calibration. The advisory stays visible so the softer signal
# is still read by a human rather than thrown away.
ADVISORY_FLAGS = ["repeats the name in the opening"]

FATAL_FLAGS = [
    "carries the error sentinel", "empty or whitespace",
    "restarts mid-profile (corrupt)", "brackets or citation markers (corrupt)",
    "introduces the subject twice (corrupt)", "copies a fact-block label (corrupt)",
]


def report_screen(df: pd.DataFrame, label: str, names: dict | None = None) -> pd.Series:
    """Print the flag table and return the fatal mask. Used by the sample AND the export gate."""
    if not len(df):
        print(f"\n{label}: no rows to screen")
        return pd.Series(dtype=bool)
    probs = screen(df, names)
    print(f"\n{label}: {len(df):,} rows")
    for name in probs.columns:
        n = int(probs[name].sum())
        mark = "🔴" if (name in FATAL_FLAGS and n) else ("⚠️ " if n else "✅")
        print(f"  {mark} {name:<44} {n:,}")
    fatal = probs[FATAL_FLAGS].any(axis=1)
    for x in df.loc[fatal, "text"].head(3):
        print(f"       corrupt: {' '.join(str(x).split())[:180]}…")
    # Advisory hits do not block anything, so they must be READ or they are worthless.
    adv = probs[ADVISORY_FLAGS].any(axis=1) & ~fatal
    if adv.any():
        print(f"\n  ⚠️  {int(adv.sum()):,} advisory: name repeats in the opening — eyeball these,")
        print("      they are usually a club named for its city or a father and son sharing a name")
        for x in df.loc[adv, "text"].head(8):
            print(f"       {' '.join(str(x).split())[:150]}…")
    return fatal


# The full run produced 97 corrupt rows in 13,439 players (0.72%) and 7 in 796 clubs (0.88%).
# Of the players, 87 were a wrapper around an otherwise perfect profile — a bare "Full name: "
# prefix or a stray citation token — and 10 were genuinely broken. All 7 clubs were genuinely
# broken; none were repairable. Regenerating 87 good profiles to remove an 11-character prefix is
# both wasteful and risky: the replacement is a different profile, and it may be worse than the
# one you already screened. Repair what is provably a wrapper; regenerate only what is genuinely
# broken.
LEADING_LABEL_RE = re.compile(
    r"^\s*(?:full name|first name|last name|born|citizenship|position|preferred foot|height"
    r"|current club|club at snapshot|clubs he played for|club|stadium|head coach at snapshot"
    r"|senior international record|primary domestic competition|squad at snapshot)\s*:\s*", re.I)
# Any bracketed span is already forbidden, so a SHORT one is a citation artifact. A long or
# unclosed one is a restart and will not match — those must be regenerated, not patched.
CITATION_TOKEN_RE = re.compile(r"\s*\[[^\[\]]{0,30}\]")

# The full run turned up a second, much uglier grounding artifact: the SDK's own metadata object
# serialized into the prose, sometimes with a several-hundred-character redirect URL inside —
#   ... from 2019 to 2023 [PerQueryResult(index="1.2.2", source_title="squawka.com", url="https://…")].
# Length is not the discriminator here; the literal "PerQueryResult" is. That string cannot occur
# in football prose, so a bracketed span that opens with it is machine output by construction and
# can be removed at any length. The [^\[\]]* body means a nested bracket makes it NOT match, which
# is the safe direction: an ambiguous case stays flagged rather than being silently rewritten.
GROUNDING_BLOB_RE = re.compile(r"\s*\[\s*PerQueryResult[^\[\]]*\]")

MIN_REPAIR_RETENTION = 0.90   # a repair that deletes >10% of the PROSE is not a repair


def backup_checkpoint(path: Path) -> Path | None:
    """
    Snapshot a checkpoint before mutating it. repair_corrupt() and quarantine_corrupt() both
    rewrite the file in place, and that file is the only copy of ~$175 of generated text. One
    .bak per calendar minute, so repeated calls do not bury the original under copies.
    """
    if not path.exists():
        return None
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M")
    bak = path.with_suffix(path.suffix + f".bak-{stamp}")
    if not bak.exists():
        bak.write_bytes(path.read_bytes())
        print(f"  backup: {bak.name}  ({bak.stat().st_size/1e6:.1f} MB)")
    return bak


def strip_grounding_blobs(t: str) -> str:
    """Remove serialized grounding metadata. Exempt from the retention guard — see repair_corrupt."""
    return GROUNDING_BLOB_RE.sub("", str(t))


def repair_text(t: str) -> str:
    """Strip only what is demonstrably scaffolding: a leading label, a citation, a metadata blob."""
    t = strip_grounding_blobs(t)
    prev = None
    while prev != t:                       # a row can carry a label AND a citation
        prev = t
        t = LEADING_LABEL_RE.sub("", t, count=1)
    t = CITATION_TOKEN_RE.sub("", t)
    # Nothing else. No whitespace normalisation, no strip() — a repair must change ONLY the
    # defect, so that repair_text(clean) is byte-identical to clean and "did this row change?"
    # stays a question with an exact answer.
    return t


def repair_corrupt(kind: str, names: dict | None = None, apply: bool = False) -> dict:
    """
    Repair the surgically-fixable corruption in a checkpoint. Dry run by default.

    Every repaired row is re-screened afterwards: if it still trips any fatal flag, or if the
    repair removed more than 10% of the text, it is left alone for quarantine_corrupt(). Nothing
    is ever "fixed" on the strength of the pattern alone.
    """
    path = checkpoint_path(kind)
    if not path.exists():
        print(f"no checkpoint for {kind}")
        return {}
    df = pd.read_csv(path)
    df["text"] = df["text"].fillna("")
    before = screen(df, names)[FATAL_FLAGS].any(axis=1)
    if not before.any():
        print(f"{kind}: nothing corrupt to repair")
        return {"corrupt": 0, "repaired": 0, "unrepairable": 0}

    cand = df.copy()
    cand.loc[before, "text"] = cand.loc[before, "text"].map(repair_text)
    # Refuse a "repair" that guts the PROSE. Retention is measured against the text with grounding
    # blobs already removed, because a blob is not prose — one redirect URL can be 400 characters,
    # and charging that against the retention budget would refuse the very repairs that matter most.
    prose = df.loc[before, "text"].map(strip_grounding_blobs)
    shrank = pd.Series(False, index=df.index)
    shrank[before] = cand.loc[before, "text"].str.len() < prose.str.len() * MIN_REPAIR_RETENTION
    cand.loc[shrank, "text"] = df.loc[shrank, "text"]

    after = screen(cand, names)[FATAL_FLAGS].any(axis=1)
    repaired = before & ~after
    left = before & after

    print(f"{kind}: {int(before.sum()):,} corrupt")
    print(f"  repairable in place : {int(repaired.sum()):,}")
    print(f"  needs regeneration  : {int(left.sum()):,}")
    if shrank.any():
        print(f"  repair refused (would delete >10% of the text): {int(shrank.sum()):,}")
    for _, r in df[repaired].head(3).iterrows():
        b = " ".join(str(r["text"]).split())[:70]
        a = " ".join(repair_text(r["text"]).split())[:70]
        print(f"    before: {b}…\n    after : {a}…")

    if apply and repaired.any():
        backup_checkpoint(path)
        df.loc[repaired, "text"] = cand.loc[repaired, "text"]
        df.loc[repaired, "words"] = df.loc[repaired, "text"].str.split().str.len()
        df.to_csv(path, index=False)
        print(f"  ✅ written back to {path.name}. Re-run the QA cell.")
    elif repaired.any():
        print("  (dry run — call again with apply=True to write these back)")
    return {"corrupt": int(before.sum()), "repaired": int(repaired.sum()),
            "unrepairable": int(left.sum())}


def quarantine_corrupt(kind: str, names: dict | None = None) -> int:
    """
    Mark corrupt rows in a checkpoint as failures so the normal resume path regenerates them.
    Corruption is not a transient error, so nothing retries it automatically — but at ~4% a run
    it is far too common to clean up by hand.
    """
    path = checkpoint_path(kind)
    if not path.exists():
        print(f"no checkpoint for {kind}")
        return 0
    df = pd.read_csv(path)
    df["text"] = df["text"].fillna("")
    fatal = screen(df, names)[FATAL_FLAGS].any(axis=1)
    n = int(fatal.sum())
    if n:
        backup_checkpoint(path)
        df.loc[fatal, "text"] = f"{ERROR_SENTINEL} quarantined as corrupt output]]"
        df.to_csv(path, index=False)
    print(f"{kind}: {n:,} corrupt rows quarantined — re-run the generation cell to regenerate them")
    return n


def restore_from_backup(kind: str, names: dict | None = None,
                        source: str | Path | None = None, apply: bool = False) -> dict:
    """
    Bring quarantined rows back from the pre-repair backup and re-judge them under the CURRENT
    gate. Dry run by default.

    Quarantining is not a verdict about the text — it is a verdict about the text under the rules
    in force at the time. When a rule turns out to be wrong (an advisory that used to be fatal) or
    a repair improves (grounding blobs are now strippable at any length), the honest move is to
    re-judge the original rather than pay to generate a replacement. A row comes back only if,
    after repair, it passes the current fatal screen. Anything that still fails stays quarantined.
    """
    path = checkpoint_path(kind)
    if not path.exists():
        print(f"no checkpoint for {kind}")
        return {}
    src = Path(source) if source else _latest_backup(path)
    if src is None or not src.exists():
        print(f"no backup found for {kind} — nothing to restore from")
        return {}

    live = pd.read_csv(path)
    bak = pd.read_csv(src)
    live["text"] = live["text"].fillna("")
    bak["text"] = bak["text"].fillna("")

    quarantined = live["text"].str.contains(re.escape(ERROR_SENTINEL), regex=True)
    if not quarantined.any():
        print(f"{kind}: nothing is quarantined")
        return {"quarantined": 0, "restored": 0, "still_failing": 0}

    # Match on id, never on row position — the backup and the live file need not be aligned.
    old = bak.set_index("id")["text"]
    ids = live.loc[quarantined, "id"]
    have = ids.map(lambda i: i in old.index)
    recovered = ids.map(lambda i: repair_text(old.get(i, "")))

    trial = pd.DataFrame({"id": ids.values, "text": recovered.values})
    still_bad = screen(trial, names)[FATAL_FLAGS].any(axis=1).values
    ok = have.values & ~still_bad

    print(f"\n{kind}: {int(quarantined.sum()):,} quarantined   (backup: {src.name})")
    print(f"  restorable under the current gate : {int(ok.sum()):,}")
    print(f"  still failing, keep for regen     : {int((~ok).sum()):,}")
    for i, (rid, txt) in enumerate(zip(ids[ok].head(3), recovered[ok].head(3))):
        who = (names or {}).get(int(rid), "")
        print(f"    {who}: {' '.join(str(txt).split())[:100]}…")

    if apply and ok.any():
        backup_checkpoint(path)
        idx = live.index[quarantined][ok]
        live.loc[idx, "text"] = recovered[ok].values
        live.loc[idx, "words"] = live.loc[idx, "text"].str.split().str.len()
        if "error" in live.columns:
            live.loc[idx, "error"] = pd.NA
        live.to_csv(path, index=False)
        print(f"  ✅ {int(ok.sum()):,} rows restored to {path.name}. Re-run the QA cell.")
    elif ok.any():
        print("  (dry run — call again with apply=True to write these back)")
    return {"quarantined": int(quarantined.sum()), "restored": int(ok.sum()),
            "still_failing": int((~ok).sum())}


def _latest_backup(path: Path) -> Path | None:
    baks = sorted(path.parent.glob(path.name + ".bak-*"))
    return baks[-1] if baks else None


def show_fatal(kind: str, names: dict | None = None, n: int = 20) -> pd.DataFrame:
    """
    List the rows in the LIVE checkpoint that currently block the export, with the evidence.

    Distinct from inspect_unrepairable(), which reads a backup to recover text quarantine has
    already destroyed. This reads what is on disk right now — the right tool when a NEW check
    starts flagging rows an older one passed, because then the check itself is what is on trial.

    Prints the first two sentences of each row verbatim. That is not decoration: every fatal flag
    added since v2.2 is a judgement about how the profile OPENS, so the opening is the evidence,
    and a truncated middle tells you nothing about whether the flag is right.
    """
    path = checkpoint_path(kind)
    if not path.exists():
        print(f"no checkpoint for {kind}")
        return pd.DataFrame()
    df = pd.read_csv(path)
    df["text"] = df["text"].fillna("")
    probs = screen(df, names)
    fatal = probs[FATAL_FLAGS].any(axis=1)

    print(f"\n{kind}: {int(fatal.sum()):,} rows currently blocking the export")
    counts = probs.loc[fatal, FATAL_FLAGS].sum()
    for flag, c in counts[counts > 0].sort_values(ascending=False).items():
        print(f"  {int(c):>4}  {flag}")

    for i, (_, r) in enumerate(df[fatal].head(n).iterrows(), 1):
        flags = ", ".join(f for f in FATAL_FLAGS if probs.loc[r.name, f])
        t = " ".join(str(r["text"]).split())
        sents = _SENT_SPLIT_RE.split(t)
        who = (names or {}).get(int(r["id"]), "")
        print(f"\n  [{i}] id={int(r['id'])}  {who}")
        print(f"      flags: {flags}")
        print(f"      s1   : {sents[0][:200]}")
        if len(sents) > 1:
            print(f"      s2   : {sents[1][:200]}")
        print(f"      tail : …{t[-110:]}")
    return df[fatal]


def inspect_unrepairable(kind: str, names: dict | None = None, n: int = 10,
                         source: str | Path | None = None) -> pd.DataFrame:
    """
    Show what repair could NOT fix — reading the pre-repair backup, not the live checkpoint.

    quarantine_corrupt() has already overwritten those rows with the error sentinel, so the
    checkpoint no longer holds the evidence. The backup does. Read this before regenerating: if
    the failures share a shape, regeneration will reproduce it and the prompt is what needs
    changing. If they look like ordinary one-off garbage, just regenerate.
    """
    path = checkpoint_path(kind)
    src = Path(source) if source else _latest_backup(path)
    if src is None or not src.exists():
        print(f"no pre-repair backup found for {kind} — nothing to inspect")
        return pd.DataFrame()

    df = pd.read_csv(src)
    df["text"] = df["text"].fillna("")
    probs = screen(df, names)
    before = probs[FATAL_FLAGS].any(axis=1)
    cand = df.copy()
    cand.loc[before, "text"] = cand.loc[before, "text"].map(repair_text)
    left = before & screen(cand, names)[FATAL_FLAGS].any(axis=1)

    print(f"\n{kind}: {int(left.sum()):,} rows repair could not fix   (from {src.name})")
    counts = probs.loc[left, FATAL_FLAGS].sum()
    for flag, c in counts[counts > 0].sort_values(ascending=False).items():
        print(f"  {int(c):>4}  {flag}")

    for i, (_, r) in enumerate(df[left].head(n).iterrows(), 1):
        flags = ", ".join(f for f in FATAL_FLAGS if probs.loc[r.name, f])
        t = " ".join(str(r["text"]).split())
        who = (names or {}).get(int(r["id"]), "")
        print(f"\n  [{i}] id={int(r['id'])}  {who}")
        print(f"      flags: {flags}")
        print(f"      head : {t[:260]}…")
        if len(t) > 380:
            print(f"      tail : …{t[-120:]}")
    return df[left]

## 7 · Stage 4d — Sample first

**Twenty-five rows, roughly a dollar, and the gate on everything downstream.**

The sample is stratified so it exercises the places quality actually breaks: the two Lab 1 target
players, shared names, the superstar end where Search knows too much, the median, and the tail
where Search knows nothing. Read all of them. In particular read the journeymen — Patrick asked
for extra attention on players who were never regulars, and that is where a grounded model is
most tempted to fill silence with invention.

In [ ]:
def build_sample() -> tuple[pd.DataFrame, pd.DataFrame]:
    picks: list[int] = []

    # The two players Lab 1's exact-match story depends on.
    picks += [28003, 449151]

    # Shared names — the disambiguation test.
    for nm in ("Paulinho", "Danilo", "Fernando"):
        picks += players.loc[players["player_name"] == nm, "player_id"].head(2).tolist()

    pool = players[~players["player_id"].isin(picks)]

    # Superstars: Search knows a great deal, so contradiction risk is highest.
    picks += pool.nlargest(5, "market_value_in_eur")["player_id"].tolist()

    # Median regulars: the bulk of the corpus.
    mid = pool[(pool["_appearances"] >= 80) & (pool["_appearances"] <= 200)]
    picks += mid.sample(min(5, len(mid)), random_state=7)["player_id"].tolist()

    # The tail: extra scrutiny, per Patrick.
    tail = pool[(pool["_appearances"] >= 1) & (pool["_appearances"] <= 6)]
    picks += tail.sample(min(5, len(tail)), random_state=7)["player_id"].tolist()

    # Goalkeepers: the style vocabulary has to work for them too, and it usually doesn't.
    gks = pool[(pool["main_position"] == "Goalkeeper") & (pool["_appearances"] > 50)]
    picks += gks.sample(min(3, len(gks)), random_state=7)["player_id"].tolist()

    seen, ordered = set(), []
    for p in picks:
        if p not in seen and p in set(players["player_id"]):
            seen.add(p)
            ordered.append(p)

    p_sample = players[players["player_id"].isin(ordered)]
    c_sample = pd.concat([
        clubs.nlargest(2, "_matches"),                              # a giant
        clubs[clubs["_matches"].between(1, 40)].sample(1, random_state=7),  # a small side
        clubs[clubs["_matches"] == 0].sample(1, random_state=7),    # transfer counterparty only
    ])
    return p_sample, c_sample


sample_players, sample_clubs = build_sample()
print(f"sample: {len(sample_players)} players, {len(sample_clubs)} clubs")
print(sample_players[["player_id", "player_name", "main_position", "_appearances"]]
      .sort_values("_appearances", ascending=False).to_string(index=False))

In [ ]:
# ---- RUN THE SAMPLE --------------------------------------------------------------------------
# resume=True, deliberately. The sample checkpoint is keyed on PROMPT_VERSION, so a prompt edit
# regenerates it automatically — but re-running the notebook with the SAME prompt should not pay
# for the same 30 rows again, nor silently replace the sample you already reviewed.
# The sample cost about a dollar and its only job was to earn the go-ahead for the full run.
# That decision is made, so on a fresh runtime there is no reason to buy it again. Set
# RUN_SAMPLE = True only if you are changing the prompt and need a new read.
RUN_SAMPLE = False

if RUN_SAMPLE or checkpoint_path("sample_players").exists():
    sample_gen_p = run_generation(make_player_tasks(sample_players), "sample_players", resume=True)
else:
    print("RUN_SAMPLE is False and no sample checkpoint exists — skipping (saves ~$1).")
    sample_gen_p = pd.DataFrame(columns=["id", "text", "words", "grounded", "error"])
if RUN_SAMPLE or checkpoint_path("sample_clubs").exists():
    sample_gen_c = run_generation(make_club_tasks(sample_clubs), "sample_clubs", resume=True)
else:
    sample_gen_c = pd.DataFrame(columns=["id", "text", "words", "grounded", "error"])

In [ ]:
# These two lookups are used by the screen, the repair tools, the QA report and the Lab 1 probes,
# so they are defined unconditionally. They lived inside the sample-review block until skipping
# the sample made every later cell fail on a NameError.
_names = {int(k): v for k, v in players.set_index("player_id")["player_name"].to_dict().items()}
_club_names = {int(k): v for k, v in clubs.set_index("club_id")["club_name"].to_dict().items()}
_apps = {int(k): v for k, v in players.set_index("player_id")["_appearances"].to_dict().items()}

if len(sample_gen_p):
    # ---- READ THE SAMPLE. All of it. ---------------------------------------------------------------

    # THE CHECK THAT WAS MISSING. Run the same screen the export gate runs, here, on the sample,
    # before you read a word of it.
    _fatal_p = report_screen(sample_gen_p, "sample players", _names)
    _fatal_c = report_screen(sample_gen_c, "sample clubs", _club_names)
    if _fatal_p.any() or _fatal_c.any():
        print(f"\n🔴 {int(_fatal_p.sum()) + int(_fatal_c.sum())} corrupt row(s) in the sample. At this "
              f"rate a full run yields roughly "
              f"{(int(_fatal_p.sum()) + int(_fatal_c.sum())) / max(len(sample_gen_p) + len(sample_gen_c), 1) * 14235:,.0f} "
              "corrupt profiles.\n   They are tagged CORRUPT below. The export gate will block on them;\n"
              "   quarantine_corrupt() clears them from the checkpoint for regeneration.")
    else:
        print("\n✅ no corrupt rows in the sample")

    _fatal_map = dict(zip(sample_gen_p["id"], _fatal_p))
    for _, r in sample_gen_p.sort_values("words").iterrows():
        pid = int(r["id"])
        tag = "TAIL" if _apps.get(pid, 0) < REGULAR_APPEARANCE_FLOOR else "    "
        if _fatal_map.get(r["id"], False):
            tag = "🔴 CORRUPT"
        print(f"\n{'='*100}\n[{tag}] {_names.get(pid, pid)}  (id {pid}, {_apps.get(pid,0)} apps, "
              f"{r['words']} words, grounded={r['grounding_metadata']})\n{'-'*100}")
        print(r["text"])

    for _, r in sample_gen_c.iterrows():
        cid = int(r["id"])
        nm = clubs.set_index("club_id")["club_name"].get(cid, cid)
        print(f"\n{'='*100}\n[CLUB] {nm}  (id {cid}, {r['words']} words, "
              f"grounded={r['grounding_metadata']})\n{'-'*100}")
        print(r["text"])

    for _lbl, _df in (("players", sample_gen_p), ("clubs", sample_gen_c)):
        if len(_df):
            print(f"\n{_lbl} word count: min {_df['words'].min()}, "
                  f"median {int(_df['words'].median())}, max {_df['words'].max()}  "
                  f"(target {TARGET_WORDS})")

### 🛑 Stop here

Review the sample with Patrick before running the next cell. Specifically:

- Do the tail profiles say anything Search could not have known? That is the hallucination signal.
- Did the shared-name players come back as the right people, and is the opening sentence
  unmistakable about which one?
- Does any profile read like a stat line? If so the semantic query will fail and the prompt needs
  another pass at the style paragraph.
- Did grounding actually fire (`grounded=True` above)? If the tail rows came back without
  grounding metadata, you are paying $35/1,000 for nothing and `GROUND_MIN_APPEARANCES` should
  go up.

When the prompt changes, bump `PROMPT_VERSION` — it names the checkpoint file, so the bump also
guarantees you don't resume onto stale text generated by the old prompt.

### 7.2 — Optional: is `thinking_level = LOW` costing us surface quality?

The v1.2 sample showed three things that all look like carelessness rather than ignorance:
dropped possessive apostrophes ("Europe premier competition", "Spain victorious Euro 2024"),
two profiles that copied a field label into the opening sentence, and one confidently wrong
out-of-scope claim (Krohn-Dehli credited with a winner against Germany at Euro 2012 — he scored
against the Netherlands only).

That is a testable hypothesis, not a diagnosis: **LOW may be under-thinking the writing.** MEDIUM
costs more, because reasoning bills at the output rate, so measure it rather than assume. This
cell regenerates the same sample at MEDIUM and reports the differences that matter.

In [ ]:
RUN_THINKING_AB = False  # ~$1 and a few minutes on the sample

if RUN_THINKING_AB and len(sample_gen_p):
    import copy as _copy

    _orig_level = THINKING_LEVEL
    rows = []
    for level in ("LOW", "MEDIUM"):
        THINKING_LEVEL = level
        got = run_generation(make_player_tasks(sample_players), f"ab_{level.lower()}", resume=False)
        txt = got["text"].astype(str)
        rows.append({
            "thinking_level": level,
            "median words": int(got["words"].median()),
            "reasoning tokens/row": int(got["thinking_tokens"].mean()),
            "label leaks": int(txt.str.contains(LABEL_LEAK_RE, regex=True).sum()),
            "restarts": int(txt.map(_restarts).sum()),
            "dropped possessives": int(txt.str.contains(
                r"(?i)\b(?:Europe|Spain|France|Barcelona|Leipzig|Bournemouth|Madrid|Brazil)\s+"
                r"(?:premier|victorious|historic|modern|golden|elite|attacking)\b",
                regex=True).sum()),
            "plumbing leaks": int(txt.str.contains(PLUMBING_RE, regex=True).sum()),
        })
    THINKING_LEVEL = _orig_level

    ab = pd.DataFrame(rows).set_index("thinking_level")
    print(ab.to_string())
    _extra = ab.loc["MEDIUM", "reasoning tokens/row"] - ab.loc["LOW", "reasoning tokens/row"]
    print(f"\nMEDIUM costs ~{_extra:,} extra reasoning tokens per row "
          f"= ${_extra * 14_235 / 1e6 * PRICE_OUT_PER_M:,.0f} across the full corpus.")
    print("Worth it only if the defect counts actually drop. If they are equal, stay on LOW —")
    print("the fixes for those defects live in the prompt and the gate, not the thinking budget.")

### 7.3 — Calibrate the cost from the sample, don't estimate it

The $175 figure in the header is my arithmetic on assumed token counts and a 100% grounding
rate. The sample knows better: it has real token counts, the real reasoning share, and the real
proportion of rows the model actually chose to search — which has run between 88% and 96%, not
100%, because Gemini 3.x decides for itself. Measure, then decide.

In [ ]:
_EMPTY_CAL = {"in_per_row": 0.0, "out_per_row": 0.0, "searched_rate": 0.0,
              "low_usd": 0.0, "high_usd": 0.0, "rows_measured": 0, "source": "none"}


def calibrate_cost(kinds=("sample_players", "sample_clubs"), total_rows: int = 14_235) -> dict:
    """
    Project cost from whatever real measurements exist.

    Falls back from the sample to the full corpus, which is better data anyway — once the real
    run has happened, projecting from 29 sample rows is strictly worse than projecting from
    14,235 actual ones.

    Always returns the same keys. An early return of {} means every caller has to guard, and
    they will not; a function that sometimes lacks a key it usually has is a trap.
    """
    frames = [pd.read_csv(checkpoint_path(k)) for k in kinds if checkpoint_path(k).exists()]
    source = "sample"
    if not frames:
        frames = [pd.read_csv(checkpoint_path(k)) for k in ("players", "clubs")
                  if checkpoint_path(k).exists()]
        source = "the full corpus"
    if not frames:
        print("No checkpoints found — nothing to measure yet.")
        return dict(_EMPTY_CAL)
    df = pd.concat(frames, ignore_index=True)
    df = df[~df["text"].fillna("").str.startswith(ERROR_SENTINEL)]
    if not len(df) or "in_tokens" not in df.columns:
        print("Checkpoints contain no measurable rows.")
        return dict(_EMPTY_CAL)

    n = len(df)
    in_pr = df["in_tokens"].mean()
    out_pr = df["out_tokens"].mean()
    think_pr = df.get("thinking_tokens", pd.Series([0] * n)).mean()
    searched = df["grounding_metadata"].astype(str).str.lower().eq("true").mean()

    gen_tokens = (in_pr * total_rows / 1e6) * PRICE_IN_PER_M + \
                 (out_pr * total_rows / 1e6) * PRICE_OUT_PER_M
    billable = max(0, total_rows * searched - GROUNDING_FREE_PER_MONTH)
    grounding = billable / 1000 * PRICE_GROUNDING_PER_1K_REQ
    # Every row is offered the tool; only some search. Bound the uncertainty rather than hide it.
    billable_hi = max(0, total_rows - GROUNDING_FREE_PER_MONTH)
    grounding_hi = billable_hi / 1000 * PRICE_GROUNDING_PER_1K_REQ
    embed = (out_pr * total_rows / 1e6) * PRICE_EMBED_PER_M

    print(f"measured on {n:,} rows from {source}")
    print(f"  input tokens/row      {in_pr:>8,.0f}")
    print(f"  output tokens/row     {out_pr:>8,.0f}   (reasoning {think_pr:,.0f}, "
          f"{think_pr/max(out_pr,1):.0%})")
    print(f"  actually searched     {searched:>8.0%}   (the rest answered from model priors)")
    print()
    print(f"projected for {total_rows:,} rows")
    print(f"  generation tokens     ${gen_tokens:>8,.2f}")
    print(f"  grounding             ${grounding:>8,.2f}   (worst case if every row searches: "
          f"${grounding_hi:,.2f})")
    print(f"  embeddings            ${embed:>8,.2f}")
    print(f"  TOTAL                 ${gen_tokens + grounding + embed:>8,.2f}"
          f"   to ${gen_tokens + grounding_hi + embed:,.2f}")
    print()
    print("Retries are not in this figure. Budget a little more for the corrupt rows the export")
    print("gate will bounce back through quarantine_corrupt() — historically a few percent.")
    return {"in_per_row": in_pr, "out_per_row": out_pr, "searched_rate": searched,
            "low_usd": gen_tokens + grounding + embed,
            "high_usd": gen_tokens + grounding_hi + embed,
            "rows_measured": n, "source": source}


_cal = calibrate_cost()   # falls back to the full corpus when the sample is skipped

## 8 · Stage 4e — The full run

Resumable. Re-run the cell after any interruption; it picks up successes and retries failures.

In [ ]:
def guard_cold_start() -> None:
    """
    Refuse to regenerate a corpus that should have been restored — but only when there was
    something to restore.

    A resume and a cold start run the same code with the same flags; the only difference is
    whether the checkpoint file was there. When a runtime dies and the restore quietly fails,
    "resume" spends $156 and several hours without saying anything unusual.

    The discriminator is the backup bucket, not the local directory. Backups in GCS with nothing
    local means the restore failed and this would be an expensive accident. An empty bucket means
    a genuine first run, which is allowed — loudly, with the price attached, because the first
    time is also the time to notice that nothing is being backed up.
    """
    actual = {"players": len(players), "clubs": len(clubs)}
    missing = []
    for kind, expected in actual.items():
        path = checkpoint_path(kind)
        have = 0
        if path.exists():
            try:
                d = pd.read_csv(path)
                have = int((~d["text"].fillna("").astype(str)
                            .str.startswith(ERROR_SENTINEL)).sum())
            except Exception:  # noqa: BLE001
                have = 0
        # Half is the threshold because a partial resume is normal and a cold start is not.
        if expected and have < expected * 0.5:
            missing.append(f"{kind}: {have:,} of {expected:,} present")

    if not missing:
        return

    backups = []
    try:
        listing = _gcs("ls", f"{CKPT_GCS}/")
        if listing.returncode == 0:
            backups = [l.strip() for l in listing.stdout.splitlines()
                       if l.strip().endswith((".csv", ".parquet"))]
    except Exception:  # noqa: BLE001
        backups = []

    if backups and not ALLOW_FULL_REGENERATION:
        raise RuntimeError(
            "Refusing to start a full generation run — this looks like a FAILED RESTORE, not a "
            "first run.\n  "
            + "\n  ".join(missing)
            + "\n\nBackups exist in the bucket but are not on this runtime:\n  "
            + "\n  ".join(backups[:6])
            + f"\n\nBefore spending anything:\n"
              f"  1. restore_checkpoints()\n"
              f"  2. status()\n\n"
              "If you genuinely intend to regenerate from scratch (~$156, several hours), set\n"
              "ALLOW_FULL_REGENERATION = True and re-run this cell."
        )

    if not ALLOW_FULL_REGENERATION:
        print("⚠️  COLD START — no checkpoints locally and no backups in the bucket.")
        for m in missing:
            print(f"      {m}")
        print("    This will generate from scratch: roughly $156 and several hours for the full")
        print(f"    corpus. Checkpoints will be mirrored to {CKPT_GCS} as it goes, so a runtime")
        print("    failure will not cost you the work a second time.")


RUN_FULL = True   # resumes from the checkpoint; guard_cold_start() below refuses
                  # to turn a failed restore into a $156 regeneration

if RUN_FULL:
    guard_cold_start()
    gen_players = run_generation(make_player_tasks(players), "players", resume=True)
    gen_clubs = run_generation(make_club_tasks(clubs), "clubs", resume=True)
else:
    print("RUN_FULL is False. Set it to True after the sample review.")
    gen_players = pd.DataFrame(load_checkpoint("players").values())
    gen_clubs = pd.DataFrame(load_checkpoint("clubs").values())

### 8.2 — Review a FRESH sample of the full output

**This is the check the 26-row sample cannot give you.** Every prompt revision so far has been
tuned against the same stratified 26 players and 4 clubs. That is the right way to iterate — a
fixed comparison set is what made "did v1.4 fix v1.3's defect?" answerable — but it also means
those 30 rows are the ones the prompt is most likely to handle well. The other 14,205 include
players with no appearances at all, clubs that never played a covered fixture, and every odd
shape the source contains.

So after the full run, before spending anything on embeddings, draw a **new** stratified sample
from the completed output, excluding everything already reviewed. Generation is already paid
for, so this costs nothing but a few minutes of reading. Change `REVIEW_SEED` to draw again.

In [ ]:
REVIEW_SEED = 101


def fresh_review(gen_df: pd.DataFrame, n_per_band: int = 3, seed: int = REVIEW_SEED) -> pd.DataFrame:
    """Stratified draw from the full output, excluding rows already reviewed."""
    seen = set(sample_players["player_id"].astype(int)) if len(sample_players) else set()
    m = gen_df.merge(
        players[["player_id", "player_name", "_appearances", "main_position", "_name_is_shared"]],
        left_on="id", right_on="player_id", how="left",
    )
    m = m[~m["id"].astype(int).isin(seen)]
    bands = {
        "no appearances in scope": m["_appearances"].fillna(0) == 0,
        "1-6 appearances": m["_appearances"].between(1, 6),
        "7-24 appearances": m["_appearances"].between(7, 24),
        "25-150 appearances": m["_appearances"].between(25, 150),
        "150+ appearances": m["_appearances"] > 150,
        "goalkeepers": m["main_position"] == "Goalkeeper",
        "shared names": m["_name_is_shared"].fillna(False),
        "shortest profiles": m["words"] <= m["words"].quantile(0.02),
        "longest profiles": m["words"] >= m["words"].quantile(0.98),
    }
    picked = []
    for label, mask in bands.items():
        sub = m[mask.fillna(False)]
        if not len(sub):
            print(f"  (no rows in band: {label})")
            continue
        picked.append(sub.sample(min(n_per_band, len(sub)), random_state=seed).assign(_band=label))
    out = pd.concat(picked).drop_duplicates("id") if picked else pd.DataFrame()
    print(f"fresh review sample: {len(out)} rows across {len(bands)} bands, "
          "none previously reviewed")
    return out


RUN_FRESH_REVIEW = True   # reads the finished corpus; costs nothing

if RUN_FRESH_REVIEW and len(gen_players):
    fresh = fresh_review(gen_players)
    report_screen(fresh, "fresh review sample", _names)
    _ff = dict(zip(fresh["id"], screen(fresh, _names)[FATAL_FLAGS].any(axis=1)))
    for _, r in fresh.sort_values(["_band", "words"]).iterrows():
        pid = int(r["id"])
        mark = "🔴 CORRUPT" if _ff.get(r["id"], False) else r["_band"]
        print(f"\n{'='*100}\n[{mark}] {_names.get(pid, pid)}  (id {pid}, "
              f"{int(r['_appearances'] or 0)} apps, {r['words']} words, "
              f"grounded={r['grounding_metadata']})\n{'-'*100}")
        print(r["text"])
else:
    print("RUN_FRESH_REVIEW is False. Turn it on once the full generation has finished —")
    print("it costs nothing, and it is the only look you get at rows the prompt was not tuned on.")

## 9 · Stage 5 — Embeddings

`gemini-embedding-001` at **3072 dimensions**, asserted per row.

### Two things worth knowing before you run this

**The width is not a free parameter.** AlloyDB's in-database embedding functions —
`google_ml.embedding(model_id, content)` and `ai.text_embedding(model_id, content)` — take exactly
two arguments. There is no `output_dimensionality`. Lab 1 embeds the fan's query text in SQL at
runtime, so it gets 3072 back, and a stored column of any other width fails the `<=>` operator
with a dimension mismatch. The assert below turns that into a generation-time failure instead of a
live-demo one.

**⚠️ Task-type symmetry is an open risk.** We embed documents with `RETRIEVAL_DOCUMENT`, which is
correct information-retrieval practice. AlloyDB's `google_ml.embedding()` embeds the query with
whatever task type the registered model defaults to, and we do not control that from here. If the
two sides disagree, retrieval quality degrades quietly — nothing errors, results just get worse.
The A/B cell below measures it on the sample. **Run that before the full embedding pass**, and if
the difference is material, set `EMBED_TASK_TYPE = None` so both sides use the model default.
This wants confirming against a live AlloyDB instance in the Terraform session either way.

Serialization is pgvector's text literal at 6 significant digits — a measured 2.15x size reduction
(60 KB → 28 KB per row) with no meaningful quality cost, since float32 carries only ~7.2 decimal
digits to begin with.

In [ ]:
EMBED_WORKERS = 32
EMBED_BATCH = 1  # gemini-embedding-001 on Vertex accepts one instance per request


def to_pgvector(vec, sig: int = EMBED_SIG_DIGITS) -> str:
    """pgvector text literal. Contains commas, so the CSV field must be quoted."""
    return "[" + ",".join(f"{float(v):.{sig}g}" for v in vec) + "]"


def embed_one(item: tuple[int, str], task_type: str | None = "__default__") -> dict[str, Any]:
    key, text = item
    tt = EMBED_TASK_TYPE if task_type == "__default__" else task_type
    cfg_kwargs: dict[str, Any] = {"output_dimensionality": EMBED_DIM}
    if tt:
        cfg_kwargs["task_type"] = tt
    cfg = gtypes.EmbedContentConfig(**cfg_kwargs)

    last = ""
    for attempt in range(MAX_RETRIES):
        try:
            resp = get_client(EMBED_LOCATION).models.embed_content(
                model=EMBED_MODEL, contents=text, config=cfg)
            vec = list(resp.embeddings[0].values)
            # FIX 6: fail here, not at import.
            if len(vec) != EMBED_DIM:
                raise RuntimeError(f"got {len(vec)} dims, expected {EMBED_DIM}")
            tok = int(getattr(getattr(resp, "usage_metadata", None), "total_token_count", 0) or 0)
            return {"id": key, "embedding": to_pgvector(vec), "dims": len(vec),
                    "tokens": tok, "error": ""}
        except Exception as exc:  # noqa: BLE001
            code = _status_code(exc)
            last = f"{type(exc).__name__} {code or ''}: {str(exc)[:200]}"
            if code is not None and code not in RETRYABLE_CODES and 400 <= code < 500:
                break
            if attempt < MAX_RETRIES - 1:
                time.sleep(_backoff(attempt, code, exc))
    return {"id": key, "embedding": "", "dims": 0, "tokens": 0, "error": last}


def embed_checkpoint(kind: str) -> Path:
    return WORK_DIR / f"emb_{kind}_{PROMPT_VERSION}.parquet"


def run_embeddings(pairs: list[tuple[int, str]], kind: str, resume: bool = True) -> pd.DataFrame:
    """
    SUPERSEDED by run_embeddings_sharded(). Kept because the checkpoint format is identical and
    it is occasionally useful to run a single region with no pacing.

    Two defects, both measured on the real project: the pool is created INSIDE the batch loop, so
    every 100 rows drain to zero before the next batch starts; and it calls embed_one(), which
    inherits generation's 20-second floor on 429 backoff. Against a ~2 req/s quota that produced
    0.31 rows/s where 2.15 was available.
    """
    p = embed_checkpoint(kind)
    done: dict[int, dict] = {}
    if resume and p.exists():
        prev = pd.read_parquet(p)
        prev = prev[(prev["embedding"].str.len() > 0) & (prev["dims"] == EMBED_DIM)]
        done = {int(r["id"]): dict(r) for _, r in prev.iterrows()}
        print(f"resume: {len(done):,} embeddings already present")

    todo = [(k, t) for k, t in pairs if k not in done]
    print(f"{kind}: {len(todo):,} to embed")
    if not todo:
        return pd.DataFrame(list(done.values()))

    results = list(done.values())
    tokens = 0
    t0 = time.time()
    batches = [todo[i:i + BATCH_SIZE] for i in range(0, len(todo), BATCH_SIZE)]

    with tqdm(total=len(todo), desc=f"embed {kind}", unit="row") as bar:
        for bi, batch in enumerate(batches, start=1):
            with ThreadPoolExecutor(max_workers=EMBED_WORKERS) as pool:
                for fut in as_completed([pool.submit(embed_one, it) for it in batch]):
                    r = fut.result()
                    tokens += r["tokens"]
                    results.append(r)
                    bar.update(1)
            if bi % SAVE_EVERY_N_BATCHES == 0:  # FIX 1 again
                pd.DataFrame(results).to_parquet(p, index=False)
    pd.DataFrame(results).to_parquet(p, index=False)

    df = pd.DataFrame(results)
    bad = int((df["dims"] != EMBED_DIM).sum())
    print(f"\n{kind}: {len(todo):,} embedded in {(time.time()-t0)/60:.1f} min")
    print(f"  tokens {tokens:,}  ≈ ${tokens/1e6*PRICE_EMBED_PER_M:.2f}")
    print(f"  wrong-width or failed rows: {bad:,}")
    return df

In [ ]:
# ---- A/B the task type on the sample before committing 14,235 rows ---------------------------
def task_type_ab(sample_texts: dict[int, str], probe: str, n: int = 20) -> None:
    ids = list(sample_texts)[:n]
    q_doc = embed_one((-1, probe), task_type="RETRIEVAL_QUERY")
    q_def = embed_one((-1, probe), task_type=None)

    def _vec(lit: str) -> np.ndarray:
        v = np.fromstring(lit.strip("[]"), sep=",", dtype=np.float32)
        return v / (np.linalg.norm(v) + 1e-12)

    for label, doc_tt, q in (("RETRIEVAL_DOCUMENT ↔ RETRIEVAL_QUERY", "RETRIEVAL_DOCUMENT", q_doc),
                             ("model default ↔ model default", None, q_def)):
        docs = [embed_one((i, sample_texts[i]), task_type=doc_tt) for i in ids]
        M = np.vstack([_vec(d["embedding"]) for d in docs if d["embedding"]])
        sims = M @ _vec(q["embedding"])
        order = np.argsort(-sims)[:5]
        print(f"\n{label}")
        print(f"  spread (top1 − top5): {sims[order[0]] - sims[order[-1]]:.4f}   "
              f"top1 {sims[order[0]]:.4f}")
        for rank, i in enumerate(order, 1):
            pid = ids[i]
            print(f"   {rank}. {sims[i]:.4f}  {_names.get(pid, pid)}")


def task_type_mismatch(sample_texts: dict[int, str], probe: str, n: int = 20) -> bool:
    """
    Test the pairing that actually SHIPS, which task_type_ab() does not.

    task_type_ab() compares two *matched* pairs — RETRIEVAL_DOCUMENT docs against a
    RETRIEVAL_QUERY query, and default docs against a default query — and both came back with
    identical rankings. That is reassuring and it is not the question. In production the documents
    are embedded here with RETRIEVAL_DOCUMENT and the query is embedded inside PostgreSQL by
    google_ml.embedding(), which takes no task-type argument and therefore uses whatever the
    registered model defaults to. The shipping pairing is DOC × default, and nothing had measured
    it.

    Returns True if the mismatch preserves the ranking.

    Caveat worth keeping: task_type=None here is a proxy for "whatever AlloyDB sends," not proof
    of it. This narrows the risk; only a live instance closes it.
    """
    ids = list(sample_texts)[:n]

    def _vec(lit: str) -> np.ndarray:
        v = np.fromstring(lit.strip("[]"), sep=",", dtype=np.float32)
        return v / (np.linalg.norm(v) + 1e-12)

    docs = [embed_one((i, sample_texts[i]), task_type="RETRIEVAL_DOCUMENT") for i in ids]
    keep = [(i, d) for i, d in zip(ids, docs) if d["embedding"]]
    M = np.vstack([_vec(d["embedding"]) for _, d in keep])
    kept_ids = [i for i, _ in keep]

    orders = {}
    for label, tt in (("matched   — RETRIEVAL_DOCUMENT docs x RETRIEVAL_QUERY query", "RETRIEVAL_QUERY"),
                      ("MISMATCH  — RETRIEVAL_DOCUMENT docs x default query (what ships)", None)):
        q = embed_one((-1, probe), task_type=tt)
        sims = M @ _vec(q["embedding"])
        order = np.argsort(-sims)[:5]
        orders[label] = [kept_ids[i] for i in order]
        print(f"\n{label}")
        print(f"  top1 {sims[order[0]]:.4f}   spread (top1 - top5) {sims[order[0]] - sims[order[-1]]:.4f}")
        for rank, i in enumerate(order, 1):
            print(f"   {rank}. {sims[i]:.4f}  {_names.get(kept_ids[i], kept_ids[i])}")

    a, b = list(orders.values())
    same = a == b
    print("\n" + ("SAFE — the mismatch preserves the ranking exactly. Absolute cosines shift, but"
                  "\n  ranking is what retrieval depends on, so RETRIEVAL_DOCUMENT is fine to ship."
                  if same else
                  "DIVERGES — the mismatch reorders results. Set EMBED_TASK_TYPE = None and re-embed"
                  "\n  so the stored vectors match whatever AlloyDB produces at query time."))
    if not same:
        print(f"  matched : {[_names.get(i, i) for i in a]}")
        print(f"  mismatch: {[_names.get(i, i) for i in b]}")
    print("\n  Caveat: task_type=None is a proxy for AlloyDB's default, not proof of it.")
    return same


RUN_TASK_TYPE_AB = True   # a few cents; also runs the DOC x default mismatch check
if RUN_TASK_TYPE_AB and len(sample_gen_p):
    task_type_ab(
        {int(r["id"]): r["text"] for _, r in sample_gen_p.iterrows()},
        "diminutive Argentine playmaker with a magical left foot",
    )
    print("\nThat compares two MATCHED pairs, which is not what ships. The pairing that ships is")
    print("RETRIEVAL_DOCUMENT documents against whatever google_ml.embedding() sends, so:")
    task_type_mismatch(
        {int(r["id"]): r["text"] for _, r in sample_gen_p.iterrows()},
        "diminutive Argentine playmaker with a magical left foot",
    )

### Embeddings, paced rather than sprinted

The first embedding runner inherited the generation harness's shape, and two things that are
right for generation are wrong here.

**A barrier every 100 rows.** `ThreadPoolExecutor` was created *inside* the batch loop, so the
pool drained completely before the next batch started. A generation call takes seconds and the
variance between calls is small, so the barrier costs little. An embedding call takes a couple of
hundred milliseconds, so one straggler retrying against a quota stalls thirty-one idle workers
for as long as it takes. With 143 batches, that compounds into hours.

**A 20-second floor on 429 backoff.** Correct for generation, where quotas are per-minute and a
request is expensive, so waiting out the minute is genuinely cheaper than colliding again. For
embeddings it is catastrophic: each worker independently sleeps 20 seconds for a request that
would have taken 0.2, and thirty-two workers doing that in parallel is thirty-two sleeping
threads and zero throughput.

The fix for a hard rate limit is not to sprint and sleep. It is to **pace just under the limit**
and stay there. `Pacer` is a token bucket that halves its rate on a 429 and creeps back up on
success, so the run converges on whatever the real quota is instead of oscillating between
flooding it and sleeping off the consequences.

In [ ]:
EMBED_CHECKPOINT_EVERY_S = 120.0   # wall clock, not row count — the frame is ~380 MB by the end
EMBED_START_RPS = 2.0    # measured: us-central1 sustained ~2.15 successful req/s on this
                         # project. Starting above the real ceiling just buys an immediate
                         # breach and a halving; start under it and let the pacer climb.
EMBED_MIN_RPS = 0.5
EMBED_MAX_RPS = 96.0
EMBED_LOCAL_BACKOFF_CAP_S = 8.0    # the Pacer does the rate control; this is only jitter


class Pacer:
    """
    Shared adaptive rate limiter, shaped like TCP congestion control: additive increase,
    multiplicative decrease.

    The naive version of this halved the rate on every 429 and it was much worse than useless.
    With 32 workers in flight, one quota breach returns up to 32 rejections at once, so a single
    breach halved the rate 32 times and pinned it at the floor within a second. Whatever the
    controller does on rejection must therefore fire **once per breach, not once per rejection** —
    hence the cooldown. That is the same reason TCP reduces once per round trip rather than once
    per dropped packet.

    Increase is additive because multiplicative growth overshoots the ceiling it just found and
    triggers the next breach immediately, which is how a controller ends up oscillating instead of
    settling.
    """

    PENALTY_COOLDOWN_S = 2.0   # one reduction per breach, not one per rejection
    AI_STEP = 0.5              # rps gained per second of clean running

    def __init__(self, rps: float = EMBED_START_RPS):
        self._rps = float(rps)
        self._next = time.monotonic()
        self._last_penalty = 0.0
        self._lock = threading.Lock()
        self.sent = 0
        self.throttled = 0
        self.reductions = 0

    @property
    def rps(self) -> float:
        with self._lock:
            return self._rps

    def wait(self) -> None:
        with self._lock:
            now = time.monotonic()
            self._next = max(now, self._next) + 1.0 / self._rps
            delay = self._next - now
            self.sent += 1
        if delay > 0:
            time.sleep(delay)

    def penalize(self) -> None:
        with self._lock:
            self.throttled += 1
            now = time.monotonic()
            if now - self._last_penalty < self.PENALTY_COOLDOWN_S:
                return                      # same breach, already accounted for
            self._last_penalty = now
            self.reductions += 1
            self._rps = max(EMBED_MIN_RPS, self._rps * 0.5)
            self._next = now                # drop the queued backlog built at the old rate

    def reward(self) -> None:
        with self._lock:
            # Additive: one AI_STEP per second of clean running, regardless of how many workers
            # report success in that second.
            self._rps = min(EMBED_MAX_RPS, self._rps + self.AI_STEP / max(self._rps, 1e-9))


def embed_one_paced(item: tuple[int, str], pacer: Pacer,
                    task_type: str | None = "__default__",
                    max_retries: int = 6,
                    location: str | None = None) -> dict[str, Any]:
    """As embed_one, but rate-limited globally and with a short local backoff."""
    key, text = item
    tt = EMBED_TASK_TYPE if task_type == "__default__" else task_type
    cfg_kwargs: dict[str, Any] = {"output_dimensionality": EMBED_DIM}
    if tt:
        cfg_kwargs["task_type"] = tt
    cfg = gtypes.EmbedContentConfig(**cfg_kwargs)

    last = ""
    for attempt in range(max_retries):
        pacer.wait()
        try:
            resp = get_client(location or EMBED_LOCATION).models.embed_content(
                model=EMBED_MODEL, contents=text, config=cfg)
            vec = list(resp.embeddings[0].values)
            if len(vec) != EMBED_DIM:
                raise RuntimeError(f"got {len(vec)} dims, expected {EMBED_DIM}")
            tok = int(getattr(getattr(resp, "usage_metadata", None), "total_token_count", 0) or 0)
            pacer.reward()
            return {"id": key, "embedding": to_pgvector(vec), "dims": len(vec),
                    "tokens": tok, "error": ""}
        except Exception as exc:  # noqa: BLE001
            code = _status_code(exc)
            last = f"{type(exc).__name__} {code or ''}: {str(exc)[:200]}"
            if code in (429, 503):
                pacer.penalize()
            elif code is not None and 400 <= code < 500 and code not in RETRYABLE_CODES:
                break
            if attempt < max_retries - 1:
                time.sleep(min(EMBED_LOCAL_BACKOFF_CAP_S,
                               0.5 * (2 ** attempt)) * (0.5 + random.random()))
    return {"id": key, "embedding": "", "dims": 0, "tokens": 0, "error": last}


def run_embeddings_fast(pairs: list[tuple[int, str]], kind: str,
                        resume: bool = True, start_rps: float = EMBED_START_RPS) -> pd.DataFrame:
    """
    One pool for the whole run, no per-batch barrier, checkpointed on a timer.

    Checkpointing is time-based rather than row-based because the results frame reaches roughly
    380 MB of vector literals; rewriting it every few hundred rows costs more than the work it
    protects.
    """
    p = embed_checkpoint(kind)
    done: dict[int, dict] = {}
    if resume and p.exists():
        prev = pd.read_parquet(p)
        prev = prev[(prev["embedding"].str.len() > 0) & (prev["dims"] == EMBED_DIM)]
        done = {int(r["id"]): dict(r) for _, r in prev.iterrows()}
        print(f"resume: {len(done):,} embeddings already present")

    todo = [(k, t) for k, t in pairs if k not in done]
    print(f"{kind}: {len(todo):,} to embed")
    if not todo:
        return pd.DataFrame(list(done.values()))

    pacer = Pacer(start_rps)
    results = list(done.values())
    tokens = 0
    failures = 0
    t0 = time.time()
    last_save = t0

    with tqdm(total=len(todo), desc=f"embed {kind}", unit="row") as bar:
        with ThreadPoolExecutor(max_workers=EMBED_WORKERS) as pool:
            futs = [pool.submit(embed_one_paced, it, pacer) for it in todo]
            for fut in as_completed(futs):
                r = fut.result()
                tokens += r["tokens"]
                failures += 1 if r["error"] else 0
                results.append(r)
                bar.update(1)
                bar.set_postfix_str(
                    f"{pacer.rps:5.1f} req/s · {pacer.throttled} throttled · {failures} failed")
                now = time.time()
                if now - last_save >= EMBED_CHECKPOINT_EVERY_S:
                    pd.DataFrame(results).to_parquet(p, index=False)
                    last_save = now
    pd.DataFrame(results).to_parquet(p, index=False)

    df = pd.DataFrame(results)
    bad = int((df["dims"] != EMBED_DIM).sum())
    mins = (time.time() - t0) / 60
    print(f"\n{kind}: {len(todo):,} embedded in {mins:.1f} min "
          f"({len(todo)/max(mins*60, 1e-9):.1f} rows/s)")
    print(f"  tokens {tokens:,}  ≈ ${tokens/1e6*PRICE_EMBED_PER_M:.2f}")
    print(f"  throttle events {pacer.throttled:,}, settled at {pacer.rps:.1f} req/s")
    print(f"  wrong-width or failed rows: {bad:,}")
    if bad:
        print("  Re-run this cell — resume skips everything that succeeded.")
    return df

### Sharding embeddings across regions

Measured on the real project: 8 workers gave 0.24 rows/s and 32 gave 0.31. Four times the
concurrency bought 29% more throughput, which is the signature of a hard server-side limit —
confirmed by the error body, `aiplatform.googleapis.com/online_prediction_requests_per_base_model`.

That quota is **per region per base model**, and a document embedding does not care where it was
computed. The same model on the same text returns the same vector from `us-east1` as from
`us-central1`, so spreading the work across regions multiplies the available quota without
changing a single output. `probe_regions()` verifies that claim rather than assuming it — it
embeds identical text everywhere and reports the cosine against the reference region. Anything
below 1.0 and the region is excluded.

This is independent of where the AlloyDB cluster lives. The cluster's region constrains the
**query-side** embedding at lab time; these are document embeddings computed offline and loaded
as ordinary data.

In [ ]:
EMBED_REGION_CANDIDATES = ["us-central1", "us-east1", "us-east4", "us-west1",
                           "europe-west4", "asia-southeast1"]


def probe_regions(candidates: list[str] | None = None,
                  reference: str = EMBED_LOCATION,
                  text: str = "A left-footed playmaker who dictated tempo from deep.") -> list[str]:
    """
    Which regions serve the embedding model, and do they agree with the reference region?

    Returns the usable list, reference first. A region that answers but disagrees is excluded —
    a corpus embedded inconsistently is worse than a corpus embedded slowly.
    """
    cands = candidates or EMBED_REGION_CANDIDATES
    if reference not in cands:
        cands = [reference] + cands

    def _vec(loc: str) -> np.ndarray | None:
        try:
            cfg = gtypes.EmbedContentConfig(output_dimensionality=EMBED_DIM,
                                            task_type=EMBED_TASK_TYPE)
            r = get_client(loc).models.embed_content(model=EMBED_MODEL, contents=text, config=cfg)
            v = np.array(list(r.embeddings[0].values), dtype=np.float32)
            return v / (np.linalg.norm(v) + 1e-12)
        except Exception as exc:  # noqa: BLE001
            print(f"  {loc:<16} unavailable — {type(exc).__name__} {_status_code(exc) or ''}")
            return None

    ref = _vec(reference)
    if ref is None:
        print(f"reference region {reference} failed; cannot compare. Using it alone.")
        return [reference]
    usable = [reference]
    print(f"  {reference:<16} reference")
    for loc in [c for c in cands if c != reference]:
        v = _vec(loc)
        if v is None:
            continue
        cos = float(ref @ v)
        verdict = "identical — safe to shard" if cos > 0.99999 else f"DIFFERS (cos {cos:.6f}) — excluded"
        print(f"  {loc:<16} cosine {cos:.6f}   {verdict}")
        if cos > 0.99999:
            usable.append(loc)
    print(f"\nusable regions: {', '.join(usable)}  "
          f"→ roughly {len(usable)}x the per-region quota")
    return usable


def fill_embedding_gaps(pairs: list[tuple[int, str]], kind: str,
                        regions: list[str] | None = None, max_passes: int = 6,
                        start_rps: float = 1.0) -> pd.DataFrame:
    """
    Keep re-running until every row has a vector or a pass makes no progress.

    A single pass under a tight quota leaves a tail: the pacer converges, but rows that exhaust
    their retries during the noisy early phase are simply dropped, and one pass at 20% loss is
    not an outcome, it is a partial result wearing an outcome's clothes. Resume already excludes
    failed rows, so each pass costs only the gap.

    Deliberately starts slower than a first pass. The remaining rows are the ones that lost the
    race last time; racing them again the same way reproduces the same result.
    """
    prev_missing = None
    for attempt in range(1, max_passes + 1):
        df = run_embeddings_sharded(pairs, kind, regions=regions, resume=True, start_rps=start_rps)
        good = set(df.loc[(df["dims"] == EMBED_DIM) & (df["embedding"].str.len() > 0), "id"]
                   .astype(int))
        missing = [k for k, _ in pairs if k not in good]
        pct = 100 * len(good) / max(len(pairs), 1)
        print(f"\npass {attempt}: {len(good):,}/{len(pairs):,} ({pct:.1f}%), "
              f"{len(missing):,} still missing")
        if not missing:
            print("  complete.")
            return df
        if prev_missing is not None and len(missing) >= prev_missing:
            print("  no progress on this pass — stopping rather than burning quota.")
            print("  Remaining rows are likely failing for a reason retries cannot fix; check the")
            print("  error column on the checkpoint before running again.")
            return df
        prev_missing = len(missing)
    print(f"\nstopped after {max_passes} passes with {len(missing):,} rows still missing.")
    return df


def run_embeddings_sharded(pairs: list[tuple[int, str]], kind: str,
                           regions: list[str] | None = None, resume: bool = True,
                           start_rps: float = EMBED_START_RPS) -> pd.DataFrame:
    """
    As run_embeddings_fast, but round-robins across regions with an independent Pacer each.

    One Pacer per region matters: quota is enforced per region, so a breach in us-central1 says
    nothing about us-east1, and a shared controller would slow every region on one region's
    rejection.
    """
    regs = regions or [EMBED_LOCATION]
    p = embed_checkpoint(kind)
    done: dict[int, dict] = {}
    if resume and p.exists():
        prev = pd.read_parquet(p)
        prev = prev[(prev["embedding"].str.len() > 0) & (prev["dims"] == EMBED_DIM)]
        done = {int(r["id"]): dict(r) for _, r in prev.iterrows()}
        print(f"resume: {len(done):,} embeddings already present")

    todo = [(k, t) for k, t in pairs if k not in done]
    print(f"{kind}: {len(todo):,} to embed across {len(regs)} region(s): {', '.join(regs)}")
    if not todo:
        return pd.DataFrame(list(done.values()))

    pacers = {r: Pacer(start_rps) for r in regs}
    results = list(done.values())
    tokens = 0
    failures = 0
    t0 = time.time()
    last_save = t0
    last_push = t0

    def _work(idx_item: tuple[int, tuple[int, str]]) -> dict[str, Any]:
        idx, item = idx_item
        loc = regs[idx % len(regs)]
        return embed_one_paced(item, pacers[loc], location=loc)

    with tqdm(total=len(todo), desc=f"embed {kind}", unit="row") as bar:
        # Workers past the pace rate just queue on pacer.wait(). Cap the pool so a
        # six-region run does not spawn 192 mostly-sleeping threads.
        with ThreadPoolExecutor(max_workers=min(EMBED_WORKERS * len(regs), 96)) as pool:
            futs = [pool.submit(_work, (i, it)) for i, it in enumerate(todo)]
            for fut in as_completed(futs):
                r = fut.result()
                tokens += r["tokens"]
                failures += 1 if r["error"] else 0
                results.append(r)
                bar.update(1)
                total_rps = sum(pc.rps for pc in pacers.values())
                thr = sum(pc.throttled for pc in pacers.values())
                bar.set_postfix_str(f"{total_rps:5.1f} req/s total · {thr} throttled · {failures} failed")
                now = time.time()
                if now - last_save >= EMBED_CHECKPOINT_EVERY_S:
                    pd.DataFrame(results).to_parquet(p, index=False)
                    last_save = now
                    # And mirror it off the runtime, less often. The absence of this step is
                    # what cost a multi-hour embedding run when the Colab runtime dropped.
                    if now - last_push >= GCS_PUSH_EVERY_S:
                        push_checkpoints(verbose=False)
                        last_push = now
    pd.DataFrame(results).to_parquet(p, index=False)
    push_checkpoints(verbose=False)

    df = pd.DataFrame(results)
    bad = int((df["dims"] != EMBED_DIM).sum())
    mins = (time.time() - t0) / 60
    print(f"\n{kind}: {len(todo):,} embedded in {mins:.1f} min "
          f"({len(todo)/max(mins*60, 1e-9):.1f} rows/s)")
    print(f"  tokens {tokens:,}  ≈ ${tokens/1e6*PRICE_EMBED_PER_M:.2f}")
    for r in regs:
        print(f"  {r:<16} settled at {pacers[r].rps:5.1f} req/s, "
              f"{pacers[r].reductions} reductions, {pacers[r].throttled} rejections")
    print(f"  wrong-width or failed rows: {bad:,}")
    if bad:
        print("  Re-run this cell — resume skips everything that succeeded.")
    return df

In [ ]:
RUN_EMBEDDINGS = True   # paced and sharded; resumes from whatever is already done

if RUN_EMBEDDINGS:
    ok_p = gen_players[~gen_players["text"].str.startswith(ERROR_SENTINEL)]
    ok_c = gen_clubs[~gen_clubs["text"].str.startswith(ERROR_SENTINEL)]
    # Sharded and paced. run_embeddings() below is kept for reference but is NOT the path: it
    # barriers every 100 rows and inherits generation's 20-second 429 floor, which together turned
    # a 45-minute job into a 10-hour one. See the section above.
    EMBED_REGIONS = probe_regions()
    emb_players = run_embeddings_sharded(
        list(zip(ok_p["id"].astype(int), ok_p["text"])), "players", regions=EMBED_REGIONS)
    emb_clubs = run_embeddings_sharded(
        list(zip(ok_c["id"].astype(int), ok_c["text"])), "clubs", regions=EMBED_REGIONS)
else:
    print("RUN_EMBEDDINGS is False.")
    emb_players = pd.read_parquet(embed_checkpoint("players")) if embed_checkpoint("players").exists() else pd.DataFrame()
    emb_clubs = pd.read_parquet(embed_checkpoint("clubs")) if embed_checkpoint("clubs").exists() else pd.DataFrame()

## 10 · QA

These profiles go in front of a thousand developers. Grounding reduces hallucination; it does not
eliminate it, and nothing about a confident 250-word paragraph signals that it invented a cup
final.

The first check is a **gate**, not a report. It runs again inside the exporter and it raises.

It uses the SAME `screen()` the sample review runs, deliberately. In v1.4 two corrupt profiles
reached a human unflagged because the detectors lived here and the sample cell had its own
(nonexistent) checking. One screen, two call sites.

In [ ]:
def assert_exportable(df: pd.DataFrame, col: str = "text", label: str = "rows",
                      names: dict | None = None) -> pd.DataFrame:
    """FIX 2, plus everything the samples taught us since."""
    probs = screen(df, names, col)
    fatal = report_screen(df, label, names)

    leak_rate = probs["leaks the data plumbing"].mean() if len(df) else 0.0
    if leak_rate > MAX_PLUMBING_RATE:
        raise RuntimeError(
            f"EXPORT BLOCKED — {leak_rate:.1%} of {label} rows refer to the data itself "
            f"(ceiling {MAX_PLUMBING_RATE:.0%}). This is a prompt regression, not a data problem: "
            "check the fact-block labels in Stage 4a and the 'writing for a supporter' rule in the "
            "prompt, then regenerate. Bump PROMPT_VERSION so the checkpoint does not resume onto "
            "the old text."
        )
    if fatal.any():
        ids = df.loc[fatal, "id"].head(10).tolist()
        raise RuntimeError(
            f"EXPORT BLOCKED — {int(fatal.sum())} {label} rows are unusable (ids {ids}…). "
            "Corruption is not a transient error, so nothing retries it automatically. Run "
            "quarantine_corrupt('players') / quarantine_corrupt('clubs') to mark them in the "
            "checkpoint, then re-run the generation cell. Nothing is written until this is clean."
        )
    return df.loc[probs.any(axis=1)]


_club_names = {int(k): x for k, x in clubs.set_index("club_id")["club_name"].to_dict().items()}

if len(gen_players):
    flagged_p = assert_exportable(gen_players, "text", "players", _names)
if len(gen_clubs):
    flagged_c = assert_exportable(gen_clubs, "text", "clubs", _club_names)

In [ ]:
# ---- Length distribution ----------------------------------------------------------------------
if len(gen_players):
    w = gen_players["words"]
    qs = w.quantile([0, .01, .05, .25, .5, .75, .95, .99, 1]).round(0).astype(int)
    print("player profile word counts")
    print(qs.to_string())
    lo, hi = int(qs.loc[0.05]), int(qs.loc[0.95])
    hist, edges = np.histogram(w, bins=20)
    for h, e in zip(hist, edges):
        print(f"  {int(e):>4} |{'█' * int(40 * h / max(hist.max(), 1))} {h:,}")
    print(f"\noutliers: {(w < lo).sum():,} below p5 ({lo}w), {(w > hi).sum():,} above p95 ({hi}w)")
    print(f"target was {TARGET_WORDS}; median is {int(w.median())}")

    # Gemini 3.x decides for itself whether to search. "grounded" means we offered the tool;
    # grounding_metadata means it actually used it. In the v1.1 sample Lionel Messi — the single
    # most important profile in the corpus — came back WITHOUT it, answered from model priors.
    if "grounding_metadata" in gen_players:
        rate = gen_players["grounding_metadata"].mean()
        print(f"\nactually searched: {gen_players['grounding_metadata'].sum():,} of "
              f"{len(gen_players):,} ({rate:.1%})")
        print("The rest answered from model priors. Fine for household names, worth a look for")
        print("anyone else — and it means the real grounding bill is lower than the projection.")
        _ung = gen_players[~gen_players["grounding_metadata"]].merge(
            players[["player_id", "player_name", "_appearances"]],
            left_on="id", right_on="player_id", how="left")
        if len(_ung):
            print(_ung.nsmallest(min(10, len(_ung)), "_appearances")
                  [["player_name", "_appearances", "words"]].to_string(index=False))

    # Opening variety: a corpus that starts every profile the same way adds a constant component
    # to every embedding, which slightly flattens the vector space.
    _open3 = gen_players["text"].astype(str).str.split().str[:6].str.join(" ").str.lower()
    _open_tail = _open3.str.replace(r"^\S+\s+\S+\s+", "", regex=True)
    top = _open_tail.value_counts().head(3)
    print(f"\nmost repeated opening construction: {top.iloc[0]/len(gen_players):.0%} of profiles")
    for phrase, n in top.items():
        print(f"   {n:>5,}  …{phrase}")

In [ ]:
# ---- Thin and generic profiles, with the tail under a microscope -----------------------------
if len(gen_players):
    q = gen_players.merge(
        players[["player_id", "player_name", "_appearances", "main_position", "_name_is_shared"]],
        left_on="id", right_on="player_id", how="left",
    )
    q["name_present"] = [
        bool(nm) and str(nm).split()[-1].lower() in str(t).lower()
        for nm, t in zip(q["player_name"], q["text"])
    ]
    q["is_tail"] = q["_appearances"] < REGULAR_APPEARANCE_FLOOR

    print(f"profiles missing the player's own surname: {(~q['name_present']).sum():,}  "
          "(these will not be findable by exact-name search — the whole point of Lab 1 Task 3)")
    if (~q["name_present"]).any():
        print(q.loc[~q["name_present"], ["id", "player_name", "words"]].head(15).to_string(index=False))

    # The first sentence must carry name + club + era. In the v1.0 sample Olise opened with
    # neither club nor era, which is precisely what shared-name disambiguation depends on.
    _club_vocab = {str(c) for c in clubs["club_name"].dropna()}
    _club_words = {w for c in _club_vocab for w in re.findall(r"[A-Z][\wÀ-ÿ'’-]{3,}", c)
                   if w.lower() not in {"club", "football", "sport", "sporting", "calcio",
                                        "deportivo", "real", "athletic", "atletico", "united",
                                        "city", "town"}}

    def _first_sentence(t: str) -> str:
        return re.split(r"(?<=[.!?])\s", str(t).strip())[0]

    q["opens_with_club"] = [
        any(w in _first_sentence(t) for w in _club_words) for t in q["text"]
    ]
    q["opens_with_era"] = [bool(re.search(r"\b(19|20)\d{2}\b", _first_sentence(t))) for t in q["text"]]
    n_noclub = int((~q["opens_with_club"]).sum())
    print(f"\nfirst sentence missing a club name: {n_noclub:,} ({n_noclub/max(len(q),1):.1%})")
    print(f"first sentence missing a year:      {int((~q['opens_with_era']).sum()):,}")
    print("Both matter most for the 155 shared-name players — that opening line is what lets a")
    print("student searching \"Paulinho\" tell six players apart.")
    _bad_open = q[~q["opens_with_club"] & q["_name_is_shared"]] if "_name_is_shared" in q else q[~q["opens_with_club"]]
    if len(_bad_open):
        print(f"⚠️  {len(_bad_open):,} of those are shared-name players — worst case:")
        for t in _bad_open["text"].head(3):
            print(f"     {_first_sentence(t)[:150]}")

    print(f"\nnon-regulars (<{REGULAR_APPEARANCE_FLOOR} apps): {q['is_tail'].sum():,} of {len(q):,}")
    print(q.groupby("is_tail")["words"].describe()[["count", "mean", "min", "max"]].round(0).to_string())
    print("\nIf the tail's mean word count sits close to the regulars', the model is padding —")
    print("a player with four substitute appearances does not have 250 words of real material.")

    print("\n--- five tail profiles, read them ---")
    for _, r in q[q["is_tail"]].sample(min(5, int(q["is_tail"].sum())), random_state=3).iterrows():
        print(f"\n[{r['player_name']} — {r['_appearances']} apps, {r['words']} words]")
        print(r["text"][:900])

In [ ]:
# ---- Numeric hallucination check ---------------------------------------------------------------
# Rule: a stated figure BELOW the verified one is impossible and is a hard finding. A figure ABOVE
# it may be legitimate — the verified block covers only Big 5 + UCL/UEL 2012–2025, so real career
# totals are often larger. Flag the impossible ones; sample the rest for a human read.
NUM_RE = re.compile(r"(\d[\d,]*)\s+(?:career\s+|league\s+)?(goals|assists|appearances)\b", re.I)

if len(gen_players):
    rows = []
    for _, r in q.iterrows():
        pid = int(r["id"])
        if pid not in career.index:
            continue
        v = career.loc[pid]
        verified = {"goals": int(v.goals), "assists": int(v.assists), "appearances": int(v.appearances)}
        for num, kind in NUM_RE.findall(str(r["text"])):
            stated = int(num.replace(",", ""))
            if stated < verified[kind.lower()]:
                rows.append({
                    "player": r["player_name"], "id": pid, "metric": kind.lower(),
                    "stated": stated, "verified": verified[kind.lower()],
                    "apps": int(r["_appearances"]),
                })
    # Denominator check: "N of his M goals" must use the authoritative M, not some other total.
    DENOM_RE = re.compile(r"(\d[\d,]*)\s+of\s+(?:his|her|their)?\s*(\d[\d,]*)\s+"
                          r"(?:total\s+|career\s+|overall\s+)?(?:club\s+|European\s+|top-flight\s+)?goals?",
                          re.I)
    bad_denoms = []
    for _, r in q.iterrows():
        pid = int(r["id"])
        if pid not in career.index:
            continue
        verified_goals = int(career.loc[pid, "goals"])
        for _num, denom in DENOM_RE.findall(str(r["text"])):
            d = int(denom.replace(",", ""))
            if d != verified_goals:
                bad_denoms.append({"player": r["player_name"], "id": pid,
                                   "stated_total": d, "verified": verified_goals})
    if bad_denoms:
        bd = pd.DataFrame(bad_denoms)
        print(f"\n⚠️  {len(bd):,} profiles relate a figure to a goal total that is not the "
              "verified one:")
        print(bd.head(15).to_string(index=False))
        print("Usually means the fact block handed the model two different totals. Check the")
        print("game_events vs appearances agreement reported in Stage 4a.")
    else:
        print("\n✅ every 'N of M goals' construction uses the verified goal total")

    hard = pd.DataFrame(rows)
    print(f"impossible numeric claims (stated < verified): {len(hard):,}")
    if len(hard):
        print(hard.sort_values("apps").head(25).to_string(index=False))
        print("\nRemediation: these are almost always the model reciting a figure for a DIFFERENT")
        print("player of the same name, or restating a single-season number as a career total.")
        print("If they cluster on shared names, the identity block needs strengthening. If they")
        print("cluster on the tail, raise GROUND_MIN_APPEARANCES.")
    else:
        print("✅ none — injecting the numbers is doing its job.")

In [ ]:
# ---- Near-duplicate detection: does the tail collapse into one text? --------------------------
def load_matrix(emb_df: pd.DataFrame) -> tuple[np.ndarray, list[int]]:
    e = emb_df[emb_df["embedding"].str.len() > 0]
    ids = e["id"].astype(int).tolist()
    M = np.vstack([np.fromstring(s.strip("[]"), sep=",", dtype=np.float32) for s in e["embedding"]])
    M /= (np.linalg.norm(M, axis=1, keepdims=True) + 1e-12)
    return M, ids


if len(emb_players):
    M, ids = load_matrix(emb_players)
    print(f"embedding matrix {M.shape}, {M.nbytes/1e6:.0f} MB")
    idx = np.array([i for i, p in enumerate(ids) if _apps.get(p, 0) < REGULAR_APPEARANCE_FLOOR])
    if len(idx) > 1:
        sub = M[idx][: min(3000, len(idx))]
        S = sub @ sub.T
        np.fill_diagonal(S, 0.0)
        pairs = int((S > NEAR_DUP_COSINE).sum() // 2)
        print(f"tail near-duplicate pairs above cosine {NEAR_DUP_COSINE}: {pairs:,}")
        print(f"mean pairwise cosine within the tail: {S[np.triu_indices_from(S, 1)].mean():.3f}")
        print("A high mean means tail profiles are near-interchangeable, so their embeddings")
        print("cluster and can crowd genuine results. Remediation: shorten the tail target, or")
        print("drop grounding there and lean on the (varied) verified facts instead.")

## 11 · The two Lab 1 queries

Lab 1's entire story is two mirror-image search failures. If the generated text does not produce
both, the lab does not work, and it is far cheaper to learn that here than on stage.

The probe below does not assert a single expected answer, deliberately. **Query 1 is the fragile
one:** whether a bare `"Messi"` fails under vector search is an empirical question about the
embedding model, not something to design the corpus around. If it turns out vector search finds
Lionel Messi perfectly well, the honest fix is to pick a different exact-match example for Lab 1 —
not to degrade the profiles until the demo breaks. So this cell ranks several candidates and
reports what actually happens.

In [ ]:
KEYWORD_MISS_PROBES = [
    "diminutive Argentine playmaker with a magical left foot",
    "towering commanding centre-back who wins everything in the air",
    "a goalkeeper who plays like an extra defender",
]
EXACT_MATCH_CANDIDATES = ["Messi", "Paulinho", "Danilo", "Fernando"]


def vector_top(query: str, M: np.ndarray, ids: list[int], k: int = 10) -> pd.DataFrame:
    q = embed_one((-1, query), task_type="RETRIEVAL_QUERY")
    qv = np.fromstring(q["embedding"].strip("[]"), sep=",", dtype=np.float32)
    qv /= np.linalg.norm(qv) + 1e-12
    sims = M @ qv
    top = np.argsort(-sims)[:k]
    return pd.DataFrame({
        "rank": range(1, len(top) + 1),
        "player": [_names.get(ids[i], ids[i]) for i in top],
        "cosine": [round(float(sims[i]), 4) for i in top],
        "apps": [_apps.get(ids[i], 0) for i in top],
    })


def keyword_hits(query: str, texts: pd.Series) -> int:
    """Crude BM25 stand-in: how many profiles contain every content word of the query."""
    words = [w for w in re.findall(r"[a-z]+", query.lower()) if len(w) > 3]
    mask = pd.Series(True, index=texts.index)
    for w in words:
        mask &= texts.str.lower().str.contains(rf"\b{w}", regex=True, na=False)
    return int(mask.sum())


if len(emb_players):
    M, ids = load_matrix(emb_players)
    txt = gen_players.set_index("id")["text"]

    print("=" * 90)
    print("FAILURE MODE 2 — the semantic miss. Keyword search should find ~nothing; vector should")
    print("find the right player. This is the one the prompt was written to satisfy.")
    print("=" * 90)
    for probe in KEYWORD_MISS_PROBES:
        print(f"\n  “{probe}”")
        print(f"  keyword search (all content words present): {keyword_hits(probe, txt)} profiles")
        print(vector_top(probe, M, ids, 5).to_string(index=False))

    print("\n" + "=" * 90)
    print("FAILURE MODE 1 — the exact-match miss. We WANT a name where vector search does badly")
    print("and BM25 nails it. Pick whichever candidate below actually misbehaves.")
    print("=" * 90)
    for name in EXACT_MATCH_CANDIDATES:
        exact = players[players["player_name"].str.contains(name, case=False, na=False)]
        print(f"\n  “{name}” — {len(exact)} players match the name literally: "
              f"{', '.join(exact['player_name'].head(8))}")
        top = vector_top(name, M, ids, 10)
        print(top.to_string(index=False))
        found = top[top["player"].str.contains(name, case=False, na=False)]
        verdict = ("vector search finds them — NOT a usable exact-match failure"
                   if len(found) and found["rank"].min() == 1
                   else f"vector search ranks the literal match at {found['rank'].min() if len(found) else '>10'}"
                        " — this one works for the lab")
        print(f"  → {verdict}")

## 12 · Stage 6 — Export

**This stage writes the pass-2 profile files and nothing else.** The eight relational tables were
exported and staged by Stage 6a and are already in the bucket, FK-clean and acceptance-checked.
Re-exporting them here would at best waste time and at worst overwrite verified data with a
differently-serialized copy.

**Gzipped CSV, no header row.** An AlloyDB CSV import requirement. Parquet is not an AlloyDB
import format and there is no `pg_parquet` extension, so this is not a preference.

**Two-pass load.** `profile_text` and `profile_embedding` are generated separately from the
relational extract and will be regenerated independently — every prompt revision reruns this stage
and nothing else. So they ship as their own file keyed on `player_id` / `club_id`, applied by
`UPDATE` after the relational load. A profile regeneration never forces a full reload.

**`QUOTE_MINIMAL`, not `QUOTE_ALL`.** This changed from the first draft and the distinction is
load-bearing: in PostgreSQL CSV input an **unquoted empty field is NULL, while a quoted `""` is an
empty string.** Quoting everything would therefore make it impossible to ever express NULL in this
file. The embedding literal and any profile text containing a comma, quote, or newline still get
quoted automatically, which is all the quoting that is actually required.

**⚠️ The id column is cast to nullable `Int64` and its dtype is asserted before writing.** pandas
widens `int64` to `float64` the moment a column takes a NULL, and a `player_id` serialized as
`69261.0` is rejected outright by PostgreSQL `INTEGER`. Stage 3 hit this on 22 columns across 7
tables. The check below reads the written bytes back and fails on any id containing a decimal
point, because asserting the dtype and asserting the output are not the same assertion.

**Rows with no usable profile are omitted entirely** rather than written as an empty field. An
omitted row leaves `profile_text` and `profile_embedding` NULL, which is what the schema comments
tell students to filter on, and it means the NULL semantics never depend on quoting behaviour.

In [ ]:
def verify_pass2_file(path: Path, expect_rows: int | None = None) -> list[list[str]]:
    """
    Read a written pass-2 file back and check it the way AlloyDB will see it. Separate from the
    writer on purpose: asserting a pandas dtype and asserting the bytes on disk are two different
    assertions, and only the second one is what the load actually consumes. Also usable standalone
    in the Terraform session to re-check a downloaded file.
    """
    with gzip.open(path, "rt", encoding="utf-8", newline="") as fh:
        rows = list(csv.reader(fh))

    problems: list[str] = []
    if expect_rows is not None and len(rows) != expect_rows:
        problems.append(f"expected {expect_rows} rows, read back {len(rows)}")
    if any(len(r) != 3 for r in rows):
        problems.append("not every row has exactly 3 fields")
    # Numeric-but-not-integer is the float case below; only a non-numeric first field means a
    # header row leaked. Keeping these distinct matters — this text is what someone reads at 6am
    # when a provision fails.
    if rows and not re.fullmatch(r"-?\d+(\.\d+)?([eE][-+]?\d+)?", rows[0][0]):
        problems.append(f"first field is not numeric — header row leaked? got {rows[0][0]!r}")
    bad_ids = [r[0] for r in rows if "." in r[0] or "e" in r[0].lower()]
    if bad_ids:
        problems.append(f"{len(bad_ids)} ids serialized as floats, e.g. {bad_ids[:3]} "
                        "— PostgreSQL INTEGER rejects these at load time")
    bad_dims = [r[0] for r in rows if r[2].count(",") + 1 != EMBED_DIM]
    if bad_dims:
        problems.append(f"{len(bad_dims)} embeddings are not {EMBED_DIM}-wide, e.g. {bad_dims[:3]}")
    if any(ERROR_SENTINEL in r[1] for r in rows):
        problems.append("error sentinel present in the written file")
    if any(not r[1].strip() for r in rows):
        problems.append("a profile_text field is empty — omit the row instead, so it loads as NULL")

    if problems:
        path.unlink(missing_ok=True)
        raise RuntimeError(f"{path.name} failed verification, file deleted:\n  - "
                           + "\n  - ".join(problems))
    return rows


def export_pass2(gen_df: pd.DataFrame, emb_df: pd.DataFrame, key_col: str, out_name: str) -> dict:
    if not len(gen_df) or not len(emb_df):
        raise RuntimeError(f"{out_name}: nothing to export — run generation and embeddings first.")

    assert_exportable(gen_df, "text", out_name)  # GATE, again, at the door

    m = gen_df[["id", "text"]].merge(
        emb_df.loc[emb_df["dims"] == EMBED_DIM, ["id", "embedding"]], on="id", how="inner"
    )
    m = m[(m["text"].str.strip() != "") & (m["embedding"].str.len() > 0)]

    # The Int64 trap. Cast, then PROVE it — a float id fails the load, loudly, at provision time.
    m["id"] = pd.to_numeric(m["id"], errors="coerce").astype("Int64")
    if m["id"].isna().any():
        raise RuntimeError(f"{out_name}: {int(m['id'].isna().sum())} rows have a null key.")
    if str(m["id"].dtype) != "Int64":
        raise RuntimeError(f"{out_name}: key dtype is {m['id'].dtype}, expected Int64.")
    m = m.sort_values("id")

    # Belt and braces: the sentinel must not exist in the bytes we are about to write.
    if m["text"].str.contains(re.escape(ERROR_SENTINEL), regex=True).any():
        raise RuntimeError(f"{out_name}: error sentinel survived to export. Refusing to write.")
    if (m["embedding"].str.count(",") != EMBED_DIM - 1).any():
        raise RuntimeError(f"{out_name}: an embedding literal is not {EMBED_DIM}-wide.")

    path = EXPORT_DIR / out_name
    with gzip.open(path, "wt", encoding="utf-8", newline="") as fh:
        w = csv.writer(fh, quoting=csv.QUOTE_MINIMAL, lineterminator="\n")
        for r in m.itertuples(index=False):
            w.writerow([int(r.id), r.text, r.embedding])   # int(), never a numpy float

    rows = verify_pass2_file(path, expect_rows=len(m))

    raw = sum(len(r[0]) + len(r[1]) + len(r[2]) + 8 for r in rows)
    sha = hashlib.sha256(path.read_bytes()).hexdigest()
    info = {
        "file": out_name, "rows": len(rows), "key": key_col,
        "bytes_gz": path.stat().st_size, "bytes_raw_est": raw, "sha256": sha,
        "column_order": [key_col, "profile_text", "profile_embedding"],
        "header": False, "quoting": "QUOTE_MINIMAL", "compression": "gzip",
        "null_convention": "row omitted entirely; unquoted empty field is NULL",
    }
    print(f"{out_name}: {len(rows):,} rows, {path.stat().st_size/1e6:.1f} MB gz "
          f"({raw/1e9:.2f} GB raw), sha256 {sha[:16]}…  ✅ verified after write")
    return info


exports: list[dict] = []
if len(gen_players) and len(emb_players):
    exports.append(export_pass2(gen_players, emb_players, "player_id", "players_profiles.csv.gz"))
if len(gen_clubs) and len(emb_clubs):
    exports.append(export_pass2(gen_clubs, emb_clubs, "club_id", "clubs_profiles.csv.gz"))

if exports:
    missing_p = len(players) - next((e["rows"] for e in exports if "players" in e["file"]), 0)
    missing_c = len(clubs) - next((e["rows"] for e in exports if "clubs" in e["file"]), 0)
    print(f"\nrows deliberately left NULL in AlloyDB: {missing_p:,} players, {missing_c:,} clubs")

### 12.2 — The load SQL

Emitted here rather than hand-written in the Terraform session, so the column order, the CSV
dialect, and the file names come from the same place that produced the files. The startup VM runs
this **after** the relational load.

Note `\copy … FROM PROGRAM`: streaming from GCS through `gcloud storage cat` avoids the import
API, which handles one table per call and one operation at a time. And `gcloud storage`, never
`gsutil`.

In [ ]:
LOAD_SQL = f"""\
-- CymbalGoal profile load — PASS 2
-- Generated by the Stage 4/5/6 notebook. Do not hand-edit.
-- Snapshot {SNAPSHOT_DATE} ({SNAPSHOT_SHA})   prompt {PROMPT_VERSION}   embed {EMBED_MODEL}@{EMBED_DIM}
--
-- Runs on the startup VM AFTER the eight relational tables are loaded (Stage 6a staged those
-- separately; this script does not touch them) and BEFORE the ScaNN indexes are built. Rows absent from these files keep profile_text and profile_embedding NULL,
-- which is intentional and is what the schema comments tell students to filter on.

\\set ON_ERROR_STOP on
BEGIN;

CREATE TEMP TABLE _p (player_id INTEGER, profile_text TEXT, profile_embedding VECTOR({EMBED_DIM}));
\\copy _p FROM PROGRAM 'gcloud storage cat {GCS_PREFIX}/players_profiles.csv.gz | gunzip' WITH (FORMAT csv)

CREATE TEMP TABLE _c (club_id INTEGER, profile_text TEXT, profile_embedding VECTOR({EMBED_DIM}));
\\copy _c FROM PROGRAM 'gcloud storage cat {GCS_PREFIX}/clubs_profiles.csv.gz | gunzip' WITH (FORMAT csv)

UPDATE players p
   SET profile_text = t.profile_text,
       profile_embedding = t.profile_embedding
  FROM _p t
 WHERE p.player_id = t.player_id;

UPDATE clubs c
   SET profile_text = t.profile_text,
       profile_embedding = t.profile_embedding
  FROM _c t
 WHERE c.club_id = t.club_id;

-- Fail the provision loudly rather than starting a lab with a half-populated corpus.
DO $$
DECLARE n_players BIGINT; n_clubs BIGINT;
BEGIN
  SELECT count(*) INTO n_players FROM players WHERE profile_embedding IS NOT NULL;
  SELECT count(*) INTO n_clubs   FROM clubs   WHERE profile_embedding IS NOT NULL;
  RAISE NOTICE 'profiles loaded: % players, % clubs', n_players, n_clubs;
  IF n_players < {int(exports[0]['rows'] * 0.99) if exports else 0} THEN
    RAISE EXCEPTION 'player profile load short: got %, expected ~{exports[0]['rows'] if exports else 0}', n_players;
  END IF;
END $$;

COMMIT;
ANALYZE players;
ANALYZE clubs;
"""

(ARTIFACT_DIR / "load_profiles.sql").write_text(LOAD_SQL, encoding="utf-8")
print(LOAD_SQL)

## 13 · Stage 6b — Stage to GCS and update the manifest

Lab verification steps assert against the manifest, so it records per-file sizes, checksums, row
counts, the prompt version, and the generation model.

**On reproducibility, plainly:** temperature 0 does not make this reproducible. Grounding queries
live Google Search, so the retrieved context drifts week to week regardless of sampling
temperature. Temperature 0 removes sampling variance, which is worth having on its own merits, but
it is not determinism. **The pinned artifact is the generated corpus, not the generation process.**
Regenerating in six months will produce different profiles. Every lab verification step must
assert against the staged files, never against anything reproducible on demand. The prompt version
and model recorded below exist so that a future regeneration is at least *explicable*.

In [ ]:
UPLOAD = True   # the export gate still has to pass first

if UPLOAD and exports:
    # check=True on purpose: a genuine upload failure must stop the run rather than let the
    # handoff check report a bucket that quietly lacks half its files. A MISSING gcloud is a
    # different thing — that is an environment problem, not a data problem, and it gets a
    # sentence rather than a traceback.
    try:
        for e in exports:
            src = EXPORT_DIR / e["file"]
            subprocess.run(["gcloud", "storage", "cp", str(src), f"{GCS_PREFIX}/{e['file']}"],
                           check=True)
        subprocess.run(["gcloud", "storage", "cp", str(ARTIFACT_DIR / "load_profiles.sql"),
                        f"{GCS_PREFIX}/load_profiles.sql"], check=True)
        print(subprocess.run(["gcloud", "storage", "ls", "-l", f"{GCS_PREFIX}/"],
                             capture_output=True, text=True).stdout)
    except FileNotFoundError:
        print("gcloud is not on PATH, so nothing was staged. The files are built and valid in")
        print(f"  {EXPORT_DIR}")
        print("Upload them from an environment that has gcloud, then re-run the handoff check.")
elif UPLOAD:
    print("UPLOAD is True but there is nothing to upload — the export produced no files.")
else:
    print("UPLOAD is False — nothing staged. Files are in", EXPORT_DIR)

In [ ]:
mpath = ARTIFACT_DIR / "manifest.json"
manifest = json.loads(mpath.read_text()) if mpath.exists() else {}

manifest.setdefault("snapshot", {"date": SNAPSHOT_DATE, "sha256_prefix": SNAPSHOT_SHA})
manifest["profiles"] = {
    "generated_utc": RUN_STARTED_UTC,
    "completed_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "prompt_version": PROMPT_VERSION,
    "generation_model": GEN_MODEL,
    "generation_location": GEN_LOCATION,
    "embedding_location": EMBED_LOCATION,
    "generation_temperature": TEMPERATURE,
    "thinking_level": THINKING_LEVEL,
    "max_output_tokens": MAX_OUTPUT_TOKENS,
    "exclude_off_pitch_controversy": EXCLUDE_OFF_PITCH_CONTROVERSY,
    "grounding": {
        "enabled": USE_GROUNDING,
        "tool": "google_search",
        "min_appearances": GROUND_MIN_APPEARANCES,
    },
    "embedding_model": EMBED_MODEL,
    "embedding_dimensions": EMBED_DIM,
    "embedding_task_type": EMBED_TASK_TYPE,
    "embedding_precision_sig_digits": EMBED_SIG_DIGITS,
    "target_words": TARGET_WORDS,
    "profile_columns_are_pass_2": True,
    "load_script": "load_profiles.sql",
    "gcs_prefix": GCS_PREFIX,
    "files": exports,
    "reproducible": False,
    "reproducibility_note": (
        "Grounded generation queries live Google Search; retrieved context drifts over time, so "
        "regeneration will not reproduce this corpus even at temperature 0. The staged files are "
        "the pinned artifact. Lab verification must assert against them, never against a rerun."
    ),
}
mpath.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest["profiles"], indent=2))

## 14 · Run report

Paste this back into the planning conversation.

In [ ]:
def run_report() -> str:
    """
    A report that cannot lie by omission. The v1.6 version gated every section on whether data
    existed, so running the notebook end to end with RUN_FULL still False produced a file with
    correct provenance, correct caveats, and no indication whatsoever that nothing had been
    generated. It looked like a finished report for an empty corpus. State comes first now.
    """
    n_p, n_c = len(gen_players), len(gen_clubs)
    # Count USABLE embeddings, not rows. The checkpoint keeps failed rows so resume can retry
    # them, so len() overstates completion — it reported 13,439/13,439 "ok" while 2,722 of those
    # rows carried an empty vector and the export correctly dropped them. A report that disagrees
    # with the export gate about what is finished is worse than no report.
    def _usable(df):
        if not len(df) or "embedding" not in df.columns:
            return 0
        return int(((df["dims"] == EMBED_DIM) & (df["embedding"].str.len() > 0)).sum())

    n_ep, n_ec = _usable(emb_players), _usable(emb_clubs)
    want_p, want_c = 13_439, 796

    L = ["# CymbalGoal Stage 4/5/6 — run report", ""]
    L += ["## State", ""]
    if not n_p and not n_c:
        L += ["> 🔴 **NO GENERATION HAS RUN.** `RUN_FULL` is still `False`, so this report describes",
              "> nothing. Set `RUN_FULL = True` and re-run the generation cell. Everything below is",
              "> configuration, not results.", ""]
    else:
        done = "complete" if (n_p >= want_p and n_c >= want_c) else "PARTIAL"
        L += [f"| Stage | Rows | Expected | State |", "| :-- | --: | --: | :-- |",
              f"| Player profiles | {n_p:,} | {want_p:,} | {done} |",
              f"| Club profiles | {n_c:,} | {want_c:,} | |",
              f"| Player embeddings | {n_ep:,} | {want_p:,} | "
              f"{'—' if not n_ep else 'ok'} |",
              f"| Club embeddings | {n_ec:,} | {want_c:,} | "
              f"{'—' if not n_ec else 'ok'} |", ""]
        if n_p < want_p or n_c < want_c:
            L += ["> ⚠️ Generation is incomplete. Re-run the generation cell to continue; it resumes.",
                  ""]
        if not n_ep:
            L += ["> ⚠️ No embeddings yet. Set `RUN_EMBEDDINGS = True`.", ""]
    if not exports:
        L += ["> ⚠️ Nothing exported. The pass-2 CSVs have not been written.", ""]

    L += [f"- snapshot **{SNAPSHOT_DATE}** ({SNAPSHOT_SHA})",
          f"- prompt **{PROMPT_VERSION}**, model **{GEN_MODEL}** at temperature {TEMPERATURE}, "
          f"thinking {THINKING_LEVEL}",
          f"- embeddings **{EMBED_MODEL}** at **{EMBED_DIM}** dims, task type {EMBED_TASK_TYPE}",
          f"- generation endpoint `{GEN_LOCATION}`, embedding endpoint `{EMBED_LOCATION}`",
          f"- grounding **{'on' if USE_GROUNDING else 'off'}**, "
          f"min appearances {GROUND_MIN_APPEARANCES}",
          f"- off-pitch controversy excluded: {EXCLUDE_OFF_PITCH_CONTROVERSY}", ""]

    if n_p:
        w = gen_players["words"]
        searched = gen_players["grounding_metadata"].mean() if "grounding_metadata" in gen_players else float("nan")
        L += ["## Corpus", "",
              f"- words: median **{int(w.median())}**, p5 {int(w.quantile(.05))}, "
              f"p95 {int(w.quantile(.95))}, min {int(w.min())}, max {int(w.max())} "
              f"(target {TARGET_WORDS})",
              f"- actually searched: **{searched:.1%}** of player rows "
              "(the rest answered from model priors)", ""]
        if "thinking_tokens" in gen_players:
            tt, ot = gen_players["thinking_tokens"].sum(), gen_players["out_tokens"].sum()
            L += [f"- reasoning was **{tt/max(ot,1):.0%}** of output tokens at "
                  f"`thinking_level={THINKING_LEVEL}`", ""]
    if exports:
        L += ["## Staged files", "",
              "| file | rows | gz size | sha256 |", "| :-- | --: | --: | :-- |"]
        L += [f"| `{e['file']}` | {e['rows']:,} | {e['bytes_gz']/1e6:.1f} MB | `{e['sha256'][:16]}…` |"
              for e in exports]
        L += ["", f"Destination `{GCS_PREFIX}/`", ""]

    # These were written when both were open. They are not any more, and a caveat that outlived
    # its question is just a lie with a cautious tone.
    L += ["## Caveats", "",
          "- Not reproducible. Grounded generation drifts; the staged corpus is the pinned artifact.",
          "- ✅ Task-type symmetry RESOLVED: cosine(RETRIEVAL_QUERY, model default) = 1.0, so "
          "Vertex's default IS RETRIEVAL_QUERY and AlloyDB's `google_ml.embedding()` pairs "
          "correctly with our RETRIEVAL_DOCUMENT documents. Residual: confirm AlloyDB does not "
          "register an explicit task type.",
          "- ⚠️ Lab 1's NAME-based exact-match example is dead — vector search returns Messi, "
          "Paulinho, Danilo and Fernando at rank 1. Use a transfer fee (`€117,000,000`, Ronaldo). "
          "Residual: BM25 tokenization of a euro-and-commas string is untested.", ""]
    return "\n".join(L)


report = run_report()
(ARTIFACT_DIR / "stage4_run_report.md").write_text(report, encoding="utf-8")
print(report)

## 15 · Handoff readiness

The definition of done for this stage, asserted rather than assumed. Patrick carries the manifest
into the Terraform session; anything missing here becomes a provisioning failure there, where it
is far more expensive to diagnose. Run this last.

In [ ]:
def handoff_ready(check_gcs: bool = True) -> bool:
    checks: list[tuple[str, bool, str]] = []

    for fname, expected in (("players_profiles.csv.gz", 13_439), ("clubs_profiles.csv.gz", 796)):
        f = EXPORT_DIR / fname
        if not f.exists():
            checks.append((f"{fname} written", False, "missing"))
            continue
        rows = verify_pass2_file(f)  # re-runs every byte-level invariant
        pct = len(rows) / expected * 100
        checks.append((f"{fname} written and verified", True,
                       f"{len(rows):,} rows ({pct:.1f}% of {expected:,}), "
                       f"{f.stat().st_size/1e6:.1f} MB"))
        checks.append((f"{fname} coverage above 95%", pct >= 95, f"{pct:.1f}%"))

    sql = ARTIFACT_DIR / "load_profiles.sql"
    checks.append(("load_profiles.sql emitted", sql.exists(),
                   f"{sql.stat().st_size:,} B" if sql.exists() else "missing"))
    if sql.exists():
        body = sql.read_text()
        checks.append(("load SQL uses gcloud storage, never gsutil",
                       "gcloud storage" in body and "gsutil" not in body, ""))
        checks.append((f"load SQL declares VECTOR({EMBED_DIM})", f"VECTOR({EMBED_DIM})" in body, ""))
        checks.append(("load SQL asserts row counts so a short load fails loudly",
                       "RAISE EXCEPTION" in body, ""))

    mpath = ARTIFACT_DIR / "manifest.json"
    if mpath.exists():
        mf = json.loads(mpath.read_text())
        prof = mf.get("profiles", {})
        checks.append(("manifest records the profile block", bool(prof), ""))
        for field in ("prompt_version", "generation_model", "embedding_model",
                      "embedding_dimensions", "files"):
            checks.append((f"manifest.profiles.{field}", field in prof,
                           str(prof.get(field, ""))[:44]))
        checks.append(("manifest still carries Stage 6a's staged_files",
                       bool(mf.get("staged_files")),
                       f"{len(mf.get('staged_files', {}))} relational tables"))
        checks.append(("manifest is honest about reproducibility",
                       prof.get("reproducible") is False, ""))
    else:
        checks.append(("manifest.json present", False, "missing"))

    if check_gcs:
        try:
            out = subprocess.run(["gcloud", "storage", "ls", f"{GCS_PREFIX}/"],
                                 capture_output=True, text=True, timeout=60)
            listing = out.stdout if out.returncode == 0 else ""
            reachable = out.returncode == 0
        except (FileNotFoundError, subprocess.TimeoutExpired) as exc:
            listing, reachable = "", False
            checks.append(("bucket reachable", False,
                           f"could not run gcloud ({type(exc).__name__}) — skipping GCS checks"))
        if not reachable:
            check_gcs = False
    if check_gcs:
        for want in ("players_profiles.csv.gz", "clubs_profiles.csv.gz", "load_profiles.sql"):
            checks.append((f"{want} staged to GCS", want in listing,
                           "" if want in listing else "not in bucket listing"))
        # Stage 6a's work is not ours to touch. Prove we didn't.
        for want in ("players.csv.gz", "schema.sql"):
            checks.append((f"Stage 6a's {want} still intact", want in listing,
                           "" if want in listing else "MISSING — did something overwrite it?"))

    print("HANDOFF READINESS\n")
    for name, ok, detail in checks:
        print(f"  {'OK ' if ok else 'X  '} {name:<52} {detail}")
    passed = all(ok for _, ok, _ in checks)
    print("\n" + ("Ready. Carry artifacts/manifest.json into the Terraform session."
                   if passed else
                   "NOT ready. Resolve the items above before the Terraform handoff."))
    if passed:
        print("\nStill open, and neither is fixable here:")
        print("  - BM25 tokenization of '€117,000,000' — Lab 1's exact-match example depends on")
        print("    pg_textsearch matching it. Fallback: 'Evian Grand Geneve FC' (word tokens).")
        print("  - Whether AlloyDB registers gemini-embedding-001 with an EXPLICIT task type.")
        print("    Vertex's default is RETRIEVAL_QUERY (measured), which is what we want.")
    return passed


RUN_HANDOFF_CHECK = True   # the definition of done

if RUN_HANDOFF_CHECK:
    handoff_ready()
else:
    print("RUN_HANDOFF_CHECK is False. Run it after uploading — it is the definition of done.")

## 16 · Handback bundle

One cell that runs every check worth a second pair of eyes and writes the combined output to a
single file. Scrolling a notebook and copying five separate outputs is error-prone in the boring
way — a section gets missed, and the one that gets missed is the one that mattered. Each section
is independently guarded, so a failure in one is reported in place instead of costing the rest.

In [ ]:
def handback_bundle(path: str | Path | None = None, include_screen: bool = True) -> Path:
    """
    Collect the run's reviewable output into one file: final state, the two Lab 1 probes, the
    embedding task-type A/B, and the handoff check.

    Cost is a handful of embedding calls — roughly a cent. Everything else reads what is already
    on disk.
    """
    needed = ["run_report", "report_screen", "handoff_ready", "final_acceptance",
              "restore_stage3_manifest", "inspect_plumbing_leaks", "hallucination_spot_check",
              "discover_exact_match_candidates", "lab1_rare_token_probes", "task_type_ab"]
    absent = [f for f in needed if not callable(globals().get(f))]
    if absent:
        raise RuntimeError(
            "handback_bundle() is missing dependencies: " + ", ".join(absent) +
            "\nRun the cells above first. This is a preflight rather than N buried "
            "'SECTION FAILED' messages, which is what a mid-file call produced."
        )

    out = Path(path) if path else (ARTIFACT_DIR / "handback.txt")
    buf = io.StringIO()

    def section(title: str, fn) -> None:
        buf.write("\n\n" + "=" * 90 + f"\n{title}\n" + "=" * 90 + "\n")
        try:
            inner = io.StringIO()
            with redirect_stdout(inner):
                result = fn()
            buf.write(inner.getvalue())
            if isinstance(result, str):
                buf.write(result)
        except Exception:
            # A broken section must not cost the others, and must not be silently absent.
            buf.write("SECTION FAILED — reported here rather than lost:\n")
            buf.write(traceback.format_exc())

    buf.write(f"CymbalGoal Stage 4/5/6 handback — prompt {PROMPT_VERSION}, "
              f"gen {GEN_MODEL}, embed {EMBED_MODEL} @ {EMBED_DIM}d\n")
    buf.write(f"generated {datetime.now(timezone.utc).isoformat(timespec='seconds')}\n")

    section("1 · RUN REPORT", run_report)

    if include_screen:
        section("2 · FINAL SCREEN — players", lambda: report_screen(gen_players, "players", _names))
        section("3 · FINAL SCREEN — clubs", lambda: report_screen(gen_clubs, "clubs", _club_names))

    def _lab1():
        if not len(emb_players):
            print("No player embeddings on disk — run the embedding cell first.")
            return
        M, ids = load_matrix(emb_players)
        txt = gen_players.set_index("id")["text"]
        print("FAILURE MODE 2 — the semantic miss.")
        print("Keyword search should find ~nothing; vector search should find the right player.\n")
        for probe in KEYWORD_MISS_PROBES:
            print(f"  “{probe}”")
            print(f"  keyword search (all content words present): {keyword_hits(probe, txt)} profiles")
            print(vector_top(probe, M, ids, 5).to_string(index=False) + "\n")
        print("\nFAILURE MODE 1 — the exact-match miss.")
        print("We WANT a name where vector search does badly and BM25 nails it.\n")
        for name in EXACT_MATCH_CANDIDATES:
            exact = players[players["player_name"].str.contains(name, case=False, na=False)]
            print(f"  “{name}” — {len(exact)} literal matches: "
                  f"{', '.join(exact['player_name'].head(8))}")
            top = vector_top(name, M, ids, 10)
            print(top.to_string(index=False))
            found = top[top["player"].str.contains(name, case=False, na=False)]
            if len(found) and found["rank"].min() == 1:
                print("  → vector search finds them at rank 1 — NOT usable as an exact-match failure\n")
            else:
                r = found["rank"].min() if len(found) else ">10"
                print(f"  → vector search ranks the literal match at {r} — this one works for the lab\n")

    section("4 · LAB 1 PROBES", _lab1)

    def _ab():
        pool = gen_players[~gen_players["text"].str.startswith(ERROR_SENTINEL)].head(20)
        if not len(pool):
            print("No usable player profiles.")
            return
        pairs = {int(r["id"]): r["text"] for _, r in pool.iterrows()}
        task_type_ab(pairs, "diminutive Argentine playmaker with a magical left foot")
        print("\nThat compares two MATCHED pairs, which is not what ships. The pairing that ships")
        print("is RETRIEVAL_DOCUMENT documents against whatever google_ml.embedding() sends:")
        task_type_mismatch(pairs, "diminutive Argentine playmaker with a magical left foot")

    section("5 · EMBEDDING TASK-TYPE A/B", _ab)
    section("6 · HANDOFF CHECK", lambda: handoff_ready())
    section("7 · FINAL ACCEPTANCE (from the staged bytes)", final_acceptance)
    section("8 · MANIFEST REPAIR", lambda: restore_stage3_manifest(apply=True))
    section("9 · PLUMBING LEAKS — the 11 nobody had read",
            lambda: (inspect_plumbing_leaks("players"), inspect_plumbing_leaks("clubs"), None)[-1])
    section("10 · HALLUCINATION SPOT-CHECK", hallucination_spot_check)
    section("11 · LAB 1 EXACT-MATCH CANDIDATES", lambda: (
        discover_exact_match_candidates("players"),
        discover_exact_match_candidates("clubs"), None)[-1])
    section("12 · LAB 1 PROBES — rare tokens", lab1_rare_token_probes)

    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(buf.getvalue())
    print(buf.getvalue())
    print("\n" + "=" * 90)
    print(f"Written to {out}  ({out.stat().st_size/1024:.0f} KB). Attach that file rather than")
    print("copying the scrollback — it is the same content and nothing gets clipped.")
    return out


# Defining this and not calling it is how the cell came to print nothing at all: the last thing
# the notebook does should BE the handback, not the ability to produce one.

## 17 · Final acceptance — read the bytes, not the variables

Stage 3 closed with an acceptance cell that tested the staged files rather than the in-memory
frames, and that is the check that actually protected the Start Lab `COPY`. This is its Stage
4/5/6 counterpart. It downloads what is in the bucket and validates that, so it can be re-run
by the Terraform session months from now with no notebook state at all.

It also runs the Lab 1 rare-token probes, because the exact-match example needs replacing and
the evidence for its replacement belongs in the same place as the evidence that the corpus is
sound.

In [ ]:
RARE_TOKEN_PROBES = [
    "Yanmar Diesel SC",          # Cerezo Osaka's pre-1993 name, appears once
    "Newton Heath LYR",          # Manchester United's founding name
    "The Hatters",               # Luton Town's nickname
    "Without Club",              # the free-agent marker
]


def final_acceptance(sample_vectors: int = 50) -> bool:
    """
    Prove the handoff from the staged bytes alone. Returns True if everything passes.

    Deliberately does not trust `exports`, `emb_players`, or any other in-memory object — those
    are the things that were wrong the last three times something looked finished and was not.
    """
    results: list[tuple[str, bool, str]] = []

    def ok(name: str, cond: bool, detail: str = "") -> None:
        results.append((name, bool(cond), detail))

    want = {"players_profiles.csv.gz": (len(players), "player_id", set(players["player_id"].astype(int))),
            "clubs_profiles.csv.gz": (len(clubs), "club_id", set(clubs["club_id"].astype(int)))}

    tmp = Path(tempfile.mkdtemp())
    for fname, (expected, key, valid_ids) in want.items():
        uri = f"{GCS_PREFIX}/{fname}"
        local = tmp / fname
        try:
            with open(local, "wb") as fh:
                proc = subprocess.run(["gcloud", "storage", "cat", uri], stdout=fh, check=False)
            if proc.returncode != 0 or not local.stat().st_size:
                ok(f"{fname} downloaded from the bucket", False, "not present or empty")
                continue
        except FileNotFoundError:
            ok(f"{fname} downloaded from the bucket", False, "gcloud unavailable")
            continue
        ok(f"{fname} downloaded from the bucket", True, f"{local.stat().st_size/1e6:.1f} MB gz")

        ids: list[int] = []
        bad_fields = bad_width = sentinel = float_id = 0
        sampled: list[str] = []
        n = 0
        with gzip.open(local, "rt", encoding="utf-8", newline="") as fh:
            for row in csv.reader(fh):
                n += 1
                if len(row) != 3:
                    bad_fields += 1
                    continue
                rid, text, vec = row
                if not rid.isdigit():
                    float_id += 1
                else:
                    ids.append(int(rid))
                if ERROR_SENTINEL in text:
                    sentinel += 1
                # Counting separators is O(len) and avoids parsing 3,072 floats 13,439 times.
                if vec.count(",") != EMBED_DIM - 1:
                    bad_width += 1
                if len(sampled) < sample_vectors and n % 97 == 1:
                    sampled.append(vec)

        pct = 100 * n / max(expected, 1)
        ok(f"{fname} row count", n == expected, f"{n:,} of {expected:,} ({pct:.1f}%)")
        ok(f"{fname} every row has exactly 3 fields", bad_fields == 0, f"{bad_fields} bad")
        ok(f"{fname} keys are integers, never 1234.0", float_id == 0, f"{float_id} float-formatted")
        ok(f"{fname} no error sentinel in the bytes", sentinel == 0, f"{sentinel} rows")
        ok(f"{fname} every vector is {EMBED_DIM} wide", bad_width == 0, f"{bad_width} wrong width")
        ok(f"{fname} no duplicate keys", len(ids) == len(set(ids)),
           f"{len(ids) - len(set(ids))} duplicates")
        orphans = set(ids) - valid_ids
        ok(f"{fname} every key exists in the relational table", not orphans,
           f"{len(orphans)} orphans" if orphans else "the UPDATE will match every row")

        # Fully parse a sample: separator counting proves shape, not that the values are numbers.
        bad_vec = 0
        for v in sampled:
            try:
                arr = np.fromstring(v.strip("[]"), sep=",", dtype=np.float32)
                if arr.size != EMBED_DIM or not np.isfinite(arr).all() or float(np.linalg.norm(arr)) == 0.0:
                    bad_vec += 1
            except Exception:  # noqa: BLE001
                bad_vec += 1
        ok(f"{fname} sampled vectors parse and are non-degenerate", bad_vec == 0,
           f"{len(sampled)} sampled, {bad_vec} bad")

        if key == "player_id":
            ok("Lab 1 canaries present", {28003, 449151} <= set(ids),
               "Lionel Messi 28003, Junior Messias 449151")

    sql = _cat_gcs(f"{GCS_PREFIX}/load_profiles.sql")
    ok("load_profiles.sql staged", bool(sql))
    if sql:
        ok("  it names both profile files",
           "players_profiles.csv.gz" in sql and "clubs_profiles.csv.gz" in sql)
        ok("  it declares VECTOR(3072)", f"VECTOR({EMBED_DIM})" in sql or f"vector({EMBED_DIM})" in sql)
        ok("  it never uses gsutil", "gsutil" not in sql)

    print("=" * 90)
    print("FINAL ACCEPTANCE — validated against the staged bytes, not notebook state")
    print("=" * 90)
    for name, good, detail in results:
        print(f"  {'OK ' if good else 'X  '} {name:<56} {detail}")
    passed = all(g for _, g, _ in results)
    print("\n" + ("✅ PASS — the pass-2 files are sound. Carry artifacts/manifest.json into the "
                  "Terraform session."
                  if passed else
                  "🔴 FAIL — do not hand off until the X lines above are resolved."))
    return passed


def lab1_rare_token_probes(k: int = 5) -> None:
    """
    Evidence for Lab 1's replacement exact-match example.

    The name-based version is dead: vector search returns Messi, Paulinho, Danilo and Fernando at
    rank 1, because every profile opens with its subject's name. Rare tokens are the opposite —
    they appear once, in passing, with no semantic neighbourhood — which is where BM25 wins and
    embeddings cannot.
    """
    if not len(emb_players) or not len(emb_clubs):
        print("Embeddings not loaded — skipping.")
        return
    for label, emb_df, gen_df, names in (("clubs", emb_clubs, gen_clubs, _club_names),
                                         ("players", emb_players, gen_players, _names)):
        e = emb_df[(emb_df["dims"] == EMBED_DIM) & (emb_df["embedding"].str.len() > 0)]
        if not len(e):
            continue
        M = np.vstack([np.fromstring(s.strip("[]"), sep=",", dtype=np.float32) for s in e["embedding"]])
        M /= (np.linalg.norm(M, axis=1, keepdims=True) + 1e-12)
        ids = e["id"].astype(int).tolist()
        txt = gen_df.set_index("id")["text"]
        print(f"\n{'=' * 90}\n{label.upper()}\n{'=' * 90}")
        for probe in RARE_TOKEN_PROBES:
            literal = int(txt.astype(str).str.contains(re.escape(probe), case=False, regex=True).sum())
            q = embed_one((-1, probe), task_type="RETRIEVAL_QUERY")
            qv = np.fromstring(q["embedding"].strip("[]"), sep=",", dtype=np.float32)
            qv /= np.linalg.norm(qv) + 1e-12
            sims = M @ qv
            top = np.argsort(-sims)[:k]
            hit_rank = next((r for r, i in enumerate(top, 1)
                             if re.search(re.escape(probe), str(txt.get(ids[i], "")), re.I)), None)
            print(f"\n  “{probe}”")
            print(f"    profiles containing it literally (what BM25 finds): {literal}")
            print(f"    vector search rank of a profile containing it: "
                  f"{hit_rank if hit_rank else f'>{k}'}")
            for r, i in enumerate(top, 1):
                mark = "←literal" if re.search(re.escape(probe), str(txt.get(ids[i], "")), re.I) else ""
                print(f"      {r}. {sims[i]:.4f}  {names.get(ids[i], ids[i])} {mark}")
        print()
    print("A good replacement for Lab 1's exact-match example is a probe where the literal count")
    print("is small and non-zero, and vector search does NOT surface it in the top 5.")

### Finding Lab 1's exact-match example from the data, not from my guesses

My four hand-picked rare tokens mostly failed as a demo: `Newton Heath LYR` and `The Hatters`
both come back at **rank 1**, and `Without Club` never appears in profile text at all. Only
`Yanmar Diesel SC` misbehaves, and only mildly — rank 4 of 796.

The reason is instructive. Those strings are semantically rich: "Newton Heath" reads like a
place, "The Hatters" like a nickname, and the profile that contains them is *about* the thing
being searched for. Embeddings handle that well.

The class embeddings genuinely cannot handle is **exact figures** — a transfer fee, a capacity,
a contract date. They carry almost no semantic signal, so the model has nothing to place them
near, while BM25 treats them as any other token and matches them exactly.

Rather than guess again, this searches the corpus for tokens that occur in exactly one profile
and measures what vector search does with each. The data picks the example.

In [ ]:
CANDIDATE_PATTERNS = {
    # The character class must not end on a comma, or the candidate string carries a trailing
    # one — "€116,600,000," is not what a student would ever type into a search box.
    "transfer fee": r"€\d{1,3}(?:,\d{3})+",
    "capacity":     r"\b\d{2},\d{3}\b",
    "exact date":   r"\b\d{1,2} (?:January|February|March|April|May|June|July|August|September"
                    r"|October|November|December) \d{4}\b",
    "historic name": r"\b(?:[A-Z][A-Za-z]+ ){1,3}(?:SC|FC|AFC|CF)\b",
}


def discover_exact_match_candidates(kind: str = "clubs", per_pattern: int = 3,
                                    max_literal: int = 2, k: int = 5) -> pd.DataFrame:
    """
    Find strings that occur in only a handful of profiles, then measure where vector search ranks
    the profile that contains them.

    A usable Lab 1 example needs BOTH halves: BM25 must find it (literal count >= 1) and vector
    search must not (no containing profile in the top k). Reporting one without the other is how
    you end up on stage with a demo that quietly works.
    """
    gen_df = gen_players if kind == "players" else gen_clubs
    emb_df = emb_players if kind == "players" else emb_clubs
    names = _names if kind == "players" else _club_names

    e = emb_df[(emb_df["dims"] == EMBED_DIM) & (emb_df["embedding"].str.len() > 0)]
    if not len(e):
        print(f"no {kind} embeddings — skipping")
        return pd.DataFrame()
    M = np.vstack([np.fromstring(s.strip("[]"), sep=",", dtype=np.float32) for s in e["embedding"]])
    M /= (np.linalg.norm(M, axis=1, keepdims=True) + 1e-12)
    ids = e["id"].astype(int).tolist()
    txt = gen_df.set_index("id")["text"].astype(str)

    rows = []
    for label, pat in CANDIDATE_PATTERNS.items():
        counts: dict[str, int] = {}
        for t in txt:
            for m in set(re.findall(pat, t)):
                counts[m] = counts.get(m, 0) + 1
        # Rare, but not unique-to-nothing: it must exist for BM25 to win.
        rare = sorted([c for c, n in counts.items() if 1 <= n <= max_literal],
                      key=lambda c: (counts[c], -len(c)))[:per_pattern]
        for cand in rare:
            q = embed_one((-1, cand), task_type="RETRIEVAL_QUERY")
            if not q["embedding"]:
                continue
            qv = np.fromstring(q["embedding"].strip("[]"), sep=",", dtype=np.float32)
            qv /= np.linalg.norm(qv) + 1e-12
            top = np.argsort(-(M @ qv))[:k]
            hit = next((r for r, i in enumerate(top, 1)
                        if re.search(re.escape(cand), txt.get(ids[i], ""), re.I)), None)
            rows.append({"pattern": label, "candidate": cand, "literal_hits": counts[cand],
                         "vector_rank": hit if hit else f">{k}",
                         "usable": hit is None,
                         "top1": names.get(ids[top[0]], ids[top[0]])})

    df = pd.DataFrame(rows)
    if not len(df):
        print(f"no candidates found in {kind}")
        return df
    df = df.sort_values(["usable", "literal_hits"], ascending=[False, True])
    print(f"\n{'=' * 96}\n{kind.upper()} — candidate exact-match examples for Lab 1\n{'=' * 96}")
    print(df.to_string(index=False))
    good = df[df["usable"]]
    print(f"\n  {len(good)} of {len(df)} are USABLE: BM25 finds them, vector search does not.")
    if len(good):
        best = good.iloc[0]
        print(f"  Best: “{best['candidate']}” — {best['literal_hits']} literal hit(s), "
              f"vector top-1 is {best['top1']}, which is the wrong answer.")
    else:
        print("  None usable. Vector search finds every candidate — see the note below.")
    return df

## 18 · Close-out — the four things standing between here and lab planning

Three checks that were promised and never run at full scale, plus the one repair the dead runtime
made necessary. None of them regenerate anything.

In [ ]:
def restore_stage3_manifest(apply: bool = False) -> dict:
    """
    Rebuild the manifest fields the runtime failure destroyed, by MEASURING the staged files.

    The Stage 3 manifest was written locally and never uploaded (a gap between P-36 and S-31), so
    it died with the runtime and was rebuilt from `schema.sql` — which recovers column order and
    nothing else. Row counts, byte sizes and sha256 came back as "unknown", and the S-32 acceptance
    cell needs them.

    Those three are recoverable, and recovering them by measurement is better than restoring them
    from a record: a sha256 computed from the object now in the bucket describes the bytes that
    will actually be loaded, whereas a restored one only describes what Stage 3 believed it wrote.
    The snapshot fields cannot be measured — they describe the upstream download, not our files —
    so they are restored from the pinned constants and labelled as such.
    """
    man_path = ARTIFACT_DIR / "manifest.json"
    man = json.loads(man_path.read_text()) if man_path.exists() else {}
    staged = man.get("staged_files", {})
    if not staged:
        print("no staged_files block to repair — run the loader cell first")
        return man

    print(f"measuring {len(staged)} staged files from {GCS_PREFIX}/ …")
    tmp = Path(tempfile.mkdtemp())
    repaired = 0
    for name, entry in staged.items():
        uri = f"{GCS_PREFIX}/{entry.get('file', name + '.csv.gz')}"
        local = tmp / f"{name}.csv.gz"
        try:
            with open(local, "wb") as fh:
                r = subprocess.run(["gcloud", "storage", "cat", uri], stdout=fh, check=False)
            if r.returncode != 0 or not local.stat().st_size:
                print(f"  X  {name:<20} not in the bucket")
                continue
        except FileNotFoundError:
            print("  gcloud unavailable — cannot measure. Nothing changed.")
            return man
        raw = local.read_bytes()
        with gzip.open(local, "rt", encoding="utf-8", newline="") as fh:
            rows = sum(1 for _ in csv.reader(fh))
        entry.update({"rows": rows, "bytes_gz": len(raw),
                      "sha256": hashlib.sha256(raw).hexdigest(),
                      "measured_from": uri})
        repaired += 1
        print(f"  OK {name:<20} {rows:>9,} rows  {len(raw)/1e6:>7.1f} MB  "
              f"sha256 {entry['sha256'][:16]}…")

    man["staged_files"] = staged
    man["snapshot"] = {"date": SNAPSHOT_DATE, "sha256": SNAPSHOT_SHA,
                       "provenance": "restored from the pinned Stage 3 constants, not measured — "
                                     "these describe the upstream download, not our staged files"}
    man.pop("derived_from", None)
    man["repaired"] = {"by": "restore_stage3_manifest()",
                       "note": "rows/bytes/sha256 measured from the objects in the bucket; "
                               "column_order derived from schema.sql; snapshot fields restored"}

    print(f"\n{repaired} of {len(staged)} files measured.")
    if apply:
        man_path.write_text(json.dumps(man, indent=2))
        try:
            subprocess.run(["gcloud", "storage", "cp", str(man_path),
                            f"{GCS_PREFIX}/manifest.json"], capture_output=True, check=False)
            print(f"  ✅ written to {man_path} and staged to {GCS_PREFIX}/manifest.json")
        except Exception:  # noqa: BLE001
            print(f"  ✅ written to {man_path}; upload failed, stage it by hand")
    else:
        print("  (dry run — call with apply=True to write and stage it)")
    return man


def inspect_plumbing_leaks(kind: str = "players", n: int = 20) -> pd.DataFrame:
    """
    Print the profiles that still leak internal vocabulary.

    11 rows, 0.08%, comfortably under the 2% export threshold — so they never blocked anything and
    nobody has read them. That is exactly the argument for reading them: an advisory nobody looks
    at is a warning that has been switched off without anyone deciding to.
    """
    gen_df = gen_players if kind == "players" else gen_clubs
    names = _names if kind == "players" else _club_names
    hit = gen_df[gen_df["text"].astype(str).str.contains(PLUMBING_RE, regex=True)]
    print(f"\n{kind}: {len(hit):,} profiles leak data plumbing "
          f"({100*len(hit)/max(len(gen_df),1):.2f}%)")
    for i, (_, r) in enumerate(hit.head(n).iterrows(), 1):
        t = " ".join(str(r["text"]).split())
        m = re.search(PLUMBING_RE, t)
        s, e = (max(0, m.start() - 90), min(len(t), m.end() + 90)) if m else (0, 180)
        print(f"\n  [{i}] {names.get(int(r['id']), r['id'])}")
        print(f"      …{t[s:e]}…")
        if m:
            print(f"      matched: “{m.group(0)}”")
    return hit


CLAIM_PATTERNS = {
    "appearances": r"(\d[\d,]*)\s+(?:senior\s+|league\s+|competitive\s+)?appearances",
    "goals": r"(\d[\d,]*)\s+(?:league\s+|competitive\s+)?goals",
}


PLUMBING_PHRASE_FIXES = [
    (r"\s+across all covered competitions", " across all competitions"),
    (r"\s+in all covered competitions", " in all competitions"),
    (r"\bcovered major competitions\b", "major competitions"),
    (r"\bin covered major competitions\b", "in major competitions"),
    (r"\s+at the snapshot\b", ""),
    (r"\bcovered competitions\b", "competitions"),
]


def repair_plumbing_phrases(kind: str = "players", apply: bool = False) -> dict:
    """
    Fix the formulaic plumbing leaks by phrase substitution.

    Safe where the possessive-apostrophe fix is not, and for a concrete reason: these are a closed
    set of stock phrases the model reuses verbatim — "across all covered competitions" nine times
    out of eleven — so the edit is a lookup, not a judgement. Inserting an apostrophe correctly
    needs to know whether "Italy top stage" wants "Italy's" or "Italy, top", which is a judgement
    per sentence and therefore riskier than the defect.

    Every edited row is re-screened, and a repair that trips any NEW flag is discarded.
    """
    path = checkpoint_path(kind)
    if not path.exists():
        print(f"no checkpoint for {kind}")
        return {}
    names = _names if kind == "players" else _club_names
    df = pd.read_csv(path)
    df["text"] = df["text"].fillna("").astype(str)
    leaking = df["text"].str.contains(PLUMBING_RE, regex=True)
    if not leaking.any():
        print(f"{kind}: no plumbing leaks to repair")
        return {"leaking": 0, "repaired": 0}

    cand = df.copy()
    def _fix(t: str) -> str:
        for pat, rep in PLUMBING_PHRASE_FIXES:
            t = re.sub(pat, rep, t)
        return t
    cand.loc[leaking, "text"] = cand.loc[leaking, "text"].map(_fix)

    before = screen(df, names)
    after = screen(cand, names)
    cleared = leaking & ~after["leaks the data plumbing"]
    # A repair must not introduce anything. Compare every other flag, not just the one we targeted.
    other = [c for c in before.columns if c != "leaks the data plumbing"]
    worse = leaking & (after[other].sum(axis=1) > before[other].sum(axis=1))
    cand.loc[worse, "text"] = df.loc[worse, "text"]
    cleared &= ~worse

    print(f"{kind}: {int(leaking.sum())} leaking, {int(cleared.sum())} cleanly repaired, "
          f"{int((leaking & ~cleared).sum())} left alone")
    if worse.any():
        print(f"  {int(worse.sum())} repair(s) refused — they tripped another flag")
    for _, r in df[cleared].head(3).iterrows():
        b = " ".join(str(r["text"]).split())
        a = " ".join(_fix(str(r["text"])).split())
        m = re.search(PLUMBING_RE, b)
        if m:
            print(f"    {names.get(int(r['id']), r['id'])}")
            print(f"      before: …{b[max(0,m.start()-60):m.end()+60]}…")
            print(f"      after : …{a[max(0,m.start()-60):m.end()+40]}…")

    if apply and cleared.any():
        backup_checkpoint(path)
        df.loc[cleared, "text"] = cand.loc[cleared, "text"]
        df.loc[cleared, "words"] = df.loc[cleared, "text"].str.split().str.len()
        df.to_csv(path, index=False)
        print(f"  ✅ written back to {path.name}.")
        print("  ⚠️  The staged CSV and its embeddings still hold the OLD text. Re-run the export")
        print("      and re-embed these rows, or leave the corpus as it is — the leak is cosmetic")
        print("      and 0.08%, and a half-updated corpus is worse than a slightly untidy one.")
    elif cleared.any():
        print("  (dry run — call with apply=True to write these back)")
    return {"leaking": int(leaking.sum()), "repaired": int(cleared.sum())}


def hallucination_spot_check(n_per_band: int = 7, ratio_flag: float = 3.0) -> pd.DataFrame:
    """
    Compare the numbers profiles CLAIM against the numbers we computed, on a stratified sample.

    Deliberately not an automated verdict. Our totals are scope-limited (Big 5 plus UCL and UEL,
    2012 onward) while grounding legitimately knows about a player's whole career, so a claim
    LARGER than ours is often correct rather than invented. What is worth a human's eyes is a claim
    that is large relative to a player who barely features in scope — so the sample is stratified
    across the appearance bands and every row is printed with both numbers side by side.
    """
    if not len(gen_players):
        print("no profiles loaded")
        return pd.DataFrame()

    auth = career.set_index("player_id")[["appearances", "goals"]] if "player_id" in career.columns \
        else career[["appearances", "goals"]]
    txt = gen_players.set_index("id")["text"].astype(str)

    bands = [("superstar", 300, 10**9), ("regular", 60, 300),
             ("squad", 15, 60), ("tail", 0, 15)]
    rows = []
    for label, lo, hi in bands:
        pool = [p for p in txt.index if lo <= _apps.get(int(p), 0) < hi]
        for pid in sorted(pool)[:: max(1, len(pool) // max(n_per_band, 1))][:n_per_band]:
            t = " ".join(txt[pid].split())
            rec = {"band": label, "pid": int(pid), "player": _names.get(int(pid), pid),
                   "our_apps": int(_apps.get(int(pid), 0))}
            try:
                rec["our_goals"] = int(auth.loc[int(pid), "goals"])
            except Exception:  # noqa: BLE001
                rec["our_goals"] = None
            for field, pat in CLAIM_PATTERNS.items():
                # Profiles quote several figures — a competition split, then a career total. The
                # first match is frequently the narrow one, which is why v4.0 showed Kyle Walker
                # claiming 39 appearances against our 480 and called it unremarkable. Take the
                # largest, which is the closest thing to a career claim.
                found = [int(x.replace(",", "")) for x in re.findall(pat, t, re.I)]
                rec[f"claimed_{field}"] = max(found) if found else None
            ca, oa = rec.get("claimed_appearances"), rec["our_apps"]
            cg, og = rec.get("claimed_goals"), rec["our_goals"]
            flags = []
            if ca and oa and ca > oa * ratio_flag:
                flags.append(f"apps {ca/oa:.1f}x ours")
            if ca and ca > 1200:
                flags.append("impossible career apps")
            # This rule was missing entirely in v4.0, and it is the one that mattered: a profile
            # claiming 98 goals for a player we computed 0 goals for sailed through as unflagged.
            # Building a column and never writing a rule against it is worse than not collecting
            # it, because the empty column reads as evidence that it was checked.
            if cg is not None and og is not None:
                if og == 0 and cg >= 5:
                    flags.append(f"claims {cg} goals, we computed 0")
                elif og and cg > og * ratio_flag:
                    flags.append(f"goals {cg/og:.1f}x ours")
            if cg and ca and cg > ca:
                flags.append("more goals than appearances")
            rec["flag"] = "; ".join(flags)
            rows.append(rec)

    df = pd.DataFrame(rows)
    print("\nHALLUCINATION SPOT-CHECK — claimed vs computed, stratified across appearance bands")
    print("Our totals cover Big 5 + UCL/UEL from 2012 only, so a larger claim is often a whole")
    print("career and legitimate. Read the flagged rows; the rest are context.\n")
    print(df.to_string(index=False))
    flagged = df[df["flag"] != ""]
    print(f"\n  {len(flagged)} of {len(df)} rows worth a human's eyes.")
    if len(flagged):
        for _, r in flagged.iterrows():
            t = " ".join(str(txt.get(r["pid"], "")).split())
            print(f"\n  {r['player']} ({r['band']}): {r['flag']}")
            print(f"      ours: {r['our_apps']} apps, {r['our_goals']} goals | "
                  f"claimed: {r['claimed_appearances']} apps, {r['claimed_goals']} goals")
            for field in ("goals", "appearances"):
                m = re.search(CLAIM_PATTERNS[field], t, re.I)
                if m:
                    print(f"      …{t[max(0, m.start()-110):m.end()+110]}…")
                    break
    return df

In [ ]:
# The call lives at the very END of the notebook, after every function it invokes. It sat in the
# middle once, and each new section it gained silently reported SECTION FAILED instead of running.
handback_bundle()